# Dissecting Pre-MZT Cis-Regulatory Grammar via Cross-Species Sequence-to-Function Transfer Learning

[![Python 3.12](https://img.shields.io/badge/python-3.12-blue.svg)](https://www.python.org/downloads/release/python-3120/)
[![PyTorch 2.11](https://img.shields.io/badge/PyTorch-2.11%20CUDA-ee4c2c.svg)](https://pytorch.org/)
[![Conda](https://img.shields.io/badge/conda-nucleotide-green.svg)](https://docs.conda.io/)

---

### Abstract & Workflow Overview
During animal embryogenesis, developmental control transitions from maternal transcripts to nascent zygotic transcription in two distinct waves: the **minor wave (pre-MZT)** and the canonical **major wave (MZT)**. Decoding the cis-regulatory grammar and pioneer transcription factor (TF) motifs driving early accessibility before widespread chromatin opening has been hindered by low cell numbers, cytoplasmic polyadenylation, and rapid embryonic cell cycles.

This repository implements a publication-ready, species- and locus-agnostic bioinformatics framework to decode the cis-regulatory grammar of pre-MZT gene activation via cross-species transfer learning across teleosts (*Danio rerio*, *Oryzias latipes*, and *Astatotilapia calliptera*).

The analysis unfolds across 9 cohesive blocks:
- **Block 0: Study Configuration, Compute Profile & Task Manager** (strictly centralized settings)
- **Block 1: Target-Species RNA-seq Analysis** (Replicate-aware / median-of-ratios normalization, corrected 5'-to-3' spliced coverage diagnostic, 3' UTR CPE test & exonic SNP shift check)
- **Block 1b: Zebrafish SLAM-seq Nascent RNA Ortholog Overlay** (MA Plot & Polyadenylation / Early ZGA Diagnostic)
- **Block 2: Background ATAC-seq Preprocessing & ChromBPNet Training** (*Danio rerio* & *Oryzias latipes*)
- **Block 3: Stage 2 Continuous Pre-MZT Linear Probe & Grafted End-to-End Models** (All-peak vs repeat-filtered excluding miR-430)
- **Block 4: Zero-Shot Validation on Target-Species Minor-Wave Genes & Medaka ATAC** (Primary clean sets vs secondary diagnostic)
- **Block 5: Unbiased Sliding-Window Scan of `TARGET_LOCUS_NAME`**
- **Block 6: Base-Resolution Attribution, Teleost-Normalized Pre-MZT TF Filtering & Evolutionary Swap**
- **Block 7: Multi-Task `NTv3` Fine-Tuning, Baseline Benchmarks, QuantSeq 3'-End Scoring & Distal Knockout**

In [ ]:
# ==============================================================================
# Emergency Task Termination Cell (Self-Contained)
# ==============================================================================
# Set TERMINATE_ALL_TASKS = True and run this cell to terminate all running background tasks.
# Guarded with False by default so accidental 'Run All' never terminates active jobs.
TERMINATE_ALL_TASKS = True

import os
import sys
from pathlib import Path
import pandas as pd

if "." not in sys.path:
    sys.path.insert(0, ".")

try:
    from scripts.task_manager import TaskManager

    # Collect all candidate work directories
    cand_paths = []
    if "WORK_DIR" in globals() and globals()["WORK_DIR"]:
        cand_paths.append(Path(globals()["WORK_DIR"]))

    globbed = sorted(p for p in Path(".").glob("work_*") if p.is_dir() and (p / "tasks").exists())
    cand_paths.extend(globbed)

    # Deduplicate while preserving order using .resolve()
    seen = set()
    work_dirs = []
    for p in cand_paths:
        r = p.resolve()
        if r not in seen and p.name != "tasks":
            seen.add(r)
            work_dirs.append(p)

    if not work_dirs:
        work_dirs = [Path("./work_astatotilapia_premzt")]

    print(f"[*] Inspecting work directories: {[str(d) for d in work_dirs]}")

    status_dfs = []
    for wd in work_dirs:
        print(f"[*] Inspecting work_dir: {wd}")
        tm = TaskManager(work_dir=str(wd))
        if TERMINATE_ALL_TASKS:
            print(f"[!] TERMINATE_ALL_TASKS is True: Terminating all active background tasks in {wd}...")
            cancel_df = tm.cancel_all()
            if cancel_df is not None and not cancel_df.empty:
                print(f"[+] Termination summary for {wd}:")
                try:
                    display(cancel_df)
                except NameError:
                    print(cancel_df)
        st_df = tm.status()
        if st_df is not None and not st_df.empty:
            st_df["work_dir"] = str(wd)
            status_dfs.append(st_df)

    if status_dfs:
        combined_df = pd.concat(status_dfs, ignore_index=True)
        print("[*] Combined TaskManager status:")
        try:
            display(combined_df)
        except NameError:
            print(combined_df)
    else:
        print("[*] No tasks found in inspected work directories.")
except Exception as e:
    print(f"[-] TaskManager check notice: {e}")


---
## Block 0: Study Configuration, Compute Profile & Task Manager

All study-specific settings (target species, target locus, gene lists, BAM paths, reference resources, pre-MZT TF lists, and compute scaling) live strictly in this block. No study-specific logic or filenames are hard-coded in subsequent analysis blocks.

In [ ]:
%load_ext autoreload
%autoreload 1
%aimport scripts.task_manager, scripts.infer_and_attribute, scripts.download_resources

import os
import sys
import json
import time
import shutil
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import pysam
from dotenv import load_dotenv
load_dotenv()
assert "HF_TOKEN" in os.environ, "HF_TOKEN not found! You need to request access from https://huggingface.co/InstaDeepAI/NTv3_tracks and set your HF_TOKEN in a .env file."

# ==============================================================================
# 1. Study & Target Locus Settings
# ==============================================================================
STUDY_NAME = "astatotilapia_premzt"
TARGET_SPECIES_NAME = "Astatotilapia_calliptera"
TARGET_LOCUS_NAME = "notch2_slc35a3a"
TARGET_GENES = ["notch2", "slc35a3a"]
LOCUS_FLANK_BP = 75000
CANONICAL_ZGA_MARKERS = ["vox", "klf17", "mxtx2"]

# Target RNA-seq BAMs (relative paths inside pwd)
# RNA-seq BAM inputs for study species pre-MZT analysis:
# RNASEQ_EARLY_BAMS: Stage before zygotic genome activation (e.g., 2-cell or unfertilized egg), representing maternal RNA baseline.
# RNASEQ_PREMZT_BAMS: Pre-MZT stage where minor-wave transcription initiates (e.g., 1000-cell or sphere), capturing earliest zygotic output.
RNASEQ_EARLY_BAMS = ["./RNA-seq/AC_2C_rep1.markdup.sorted.bam"]      # 2-cell maternal baseline
RNASEQ_PREMZT_BAMS = ["./RNA-seq/AC_1K_rep1.markdup.sorted.bam"]    # 1000-cell pre-MZT stage

# Target genome and GTF (set to local relative path, or None to auto-download)
TARGET_GENOME_FA = f"./work_{STUDY_NAME}/downloads/Astatotilapia_calliptera.fAstCal1.3.dna_sm.toplevel.fa"
TARGET_GTF = f"./work_{STUDY_NAME}/downloads/Astatotilapia_calliptera.fAstCal1.3.112.gtf.gz"

# Zebrafish and Medaka reference genomes & JASPAR motif database (set to path or None to auto-download)
ZEBRAFISH_GENOME_FA = None
ZEBRAFISH_GTF = None
MEDAKA_GENOME_FA = None
MEDAKA_GTF = None
JASPAR_MEME_PATH = None

# Pre-MZT TF expression filter
PREMZT_TF_LIST_PATH = "./TF_gene_names.txt"

# SLAM-seq & Ortholog Mapping options
SLAMSEQ_GENE_TABLE_PATH = None  # None -> auto-generated at WORK_DIR/slamseq/zf_slamseq_gene_classes.tsv
ORTHOLOG_MAP_PATH = None        # None -> auto-generated at WORK_DIR/downloads/orthologs_target_to_zf.tsv
SLAM_EXCLUDE_MITO = True
SLAM_MASK_BASELINE_SNPS = True
SLAM_MIN_BASELINE_FC = 3.0
SLAM_MIN_KINETICS_RETENTION = 0.5
SLAM_MAX_BLEED_RATIO = 0.005

# ==============================================================================
# 2. Epigenomic & Transcriptomic Dataset Accessions
# ==============================================================================
# Zebrafish ATAC-seq (GSE130944, Palfy et al. 2020)
ZEBRAFISH_ATAC_ACCESSIONS = [
    "GSM3756599", "GSM3756600",  # 256-cell early blastula replicates
    "GSM3756603", "GSM3756604", "GSM3756605",  # dome late blastula replicates
    "SRR9032640"  # naked genomic DNA Tn5 control
]

# Medaka ATAC-seq (PRJDB7492, Nakamura et al. 2021)
MEDAKA_ATAC_ACCESSIONS = [
    "DRX143405", "DRX143406", "DRX143407",  # st9 early blastula replicates
    "DRX143412", "DRX143413"               # st11 late blastula replicates
]

# Zebrafish Nascent SLAM-seq (PRJNA768450, Bhat et al. 2023)
SLAM_ACCESSIONS = [
    "GSM5610039", "GSM5610040",  # 0.75 hpf (+4sU) baseline
    "GSM5610043", "GSM5610044",  # 2.5 hpf (+4sU) pre-MZT
    "GSM5610045", "GSM5610046", "GSM5610047", "GSM5610048",  # 3.0-3.5 hpf (+4sU) onset
    "GSM5610051", "GSM5610052",  # 4.5 hpf (+4sU) major wave
    "GSM5610075", "GSM5610076", "GSM5610077"   # -4sU untreated controls
]

USE_PRETRAINED_BIAS = False
BIAS_MODEL_PATH = None

# ==============================================================================
# 3. Environment, Credentials & Compute Profile
# ==============================================================================
CONDA_ENV_NAME = "nucleotide"
COMPUTE_PROFILE = "laptop_16gb"  # Options: "laptop_16gb" (default) or "hpc_1mb"

# Hugging Face token (redacted from logs/saved JSON; sets HF_HOME inside WORK_DIR)
HF_TOKEN = os.environ.get("HF_TOKEN", "")

# Compute & execution parameters
WORK_DIR = f"./work_{STUDY_NAME}"
THREADS = 8
FORCE_RERUN = False
CV_FOLDS = 5
ENABLE_MEDAKA = True
STAGE2_FILTER_HIGHCONF_PEAKS = True
STAGE2_MIN_FE = 4.0
STAGE2_MIN_NEG_LOG10_Q = 5.0
ENSEMBLE_MEDAKA_WITH_ZF = False
EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2 = False
NTV3_RUN_BASELINES = True
ENABLE_OPTION2_TARGET_RNA_ABLATION = False

# Epigenomics & RNA-seq Quantification Settings
CHROMBPNET_EPOCHS = 50
RNASEQ_STRANDNESS = 0  # 0: unstranded, 1: stranded, 2: reversely stranded

# Profile presets
if COMPUTE_PROFILE == "laptop_16gb":
    NTV3_MODEL_ID = "InstaDeepAI/NTv3_tracks"
    NTV3_CONTEXT_LEN = 65536
    NTV3_TRAIN_MODE = "lora"
    NTV3_PRECISION = "bf16"
    NTV3_GRAD_CHECKPOINTING = True
    NTV3_BATCH_SIZE = 1
    NTV3_GRAD_ACCUM = 16
    NUM_GPUS = 1
    EXECUTION_BACKEND = "local_background"
    slurm_config = None
elif COMPUTE_PROFILE == "hpc_1mb":
    NTV3_MODEL_ID = "InstaDeepAI/NTv3_tracks_1mb"
    NTV3_CONTEXT_LEN = 1048576
    NTV3_TRAIN_MODE = "full"
    NTV3_PRECISION = "bf16"
    NTV3_GRAD_CHECKPOINTING = True
    NTV3_BATCH_SIZE = 4
    NTV3_GRAD_ACCUM = 4
    NUM_GPUS = 4
    EXECUTION_BACKEND = "slurm"
    slurm_config = {
        "partition": "gpu",
        "gres": f"gpu:{NUM_GPUS}",
        "cpus_per_task": THREADS,
        "mem": "64G",
        "time": "24:00:00"
    }
else:
    raise ValueError(f"Unknown COMPUTE_PROFILE: {COMPUTE_PROFILE}")

# Ensure HF_HOME is strictly inside WORK_DIR
os.environ["HF_HOME"] = os.path.abspath(f"{WORK_DIR}/hf_cache")

# ==============================================================================
# 4. Reference Resources (Auto-downloaded if not present)
# ==============================================================================
# Safely resolve None defaults for all reference paths immediately
if TARGET_GENOME_FA is None:
    TARGET_GENOME_FA = f"{WORK_DIR}/downloads/{TARGET_SPECIES_NAME}.dna_sm.toplevel.fa"
if TARGET_GTF is None:
    TARGET_GTF = f"{WORK_DIR}/downloads/{TARGET_SPECIES_NAME}.gtf.gz"
if ZEBRAFISH_GENOME_FA is None:
    ZEBRAFISH_GENOME_FA = f"{WORK_DIR}/downloads/Danio_rerio.GRCz11.dna_sm.primary_assembly.fa"
if ZEBRAFISH_GTF is None:
    ZEBRAFISH_GTF = f"{WORK_DIR}/downloads/Danio_rerio.GRCz11.112.gtf.gz"
if MEDAKA_GENOME_FA is None:
    MEDAKA_GENOME_FA = f"{WORK_DIR}/downloads/Oryzias_latipes.ASM223467v1.dna_sm.toplevel.fa"
if MEDAKA_GTF is None:
    MEDAKA_GTF = f"{WORK_DIR}/downloads/Oryzias_latipes.ASM223467v1.112.gtf.gz"
if JASPAR_MEME_PATH is None:
    JASPAR_MEME_PATH = f"{WORK_DIR}/downloads/JASPAR2024_CORE_vertebrates_non-redundant.meme"

# Define cross-validation folds and canonical downstream paths in Block 0 to ensure fresh-kernel safety
FOLDS_TO_TRAIN = [0] if COMPUTE_PROFILE == "laptop_16gb" else list(range(CV_FOLDS))
filtered_meme_path = f"{WORK_DIR}/downloads/premzt_tfs.meme"
ortholog_tsv = f"{WORK_DIR}/downloads/orthologs_target_to_zf.tsv"

# Create directory hierarchy strictly inside pwd
for d in ["tasks", "downloads", "atac", "slamseq", "rnaseq", "models", "results", "hf_cache"]:
    Path(f"{WORK_DIR}/{d}").mkdir(parents=True, exist_ok=True)

# Define and create canonical results directory for target locus to prevent NameError
res_dir = Path(WORK_DIR) / "results" / TARGET_LOCUS_NAME
res_dir.mkdir(parents=True, exist_ok=True)

# Redact token before serializing config snapshot
config_snapshot = {
    "STUDY_NAME": STUDY_NAME,
    "TARGET_SPECIES_NAME": TARGET_SPECIES_NAME,
    "TARGET_LOCUS_NAME": TARGET_LOCUS_NAME,
    "TARGET_GENES": TARGET_GENES,
    "TARGET_GENOME_FA": TARGET_GENOME_FA,
    "TARGET_GTF": TARGET_GTF,
    "ZEBRAFISH_GENOME_FA": ZEBRAFISH_GENOME_FA,
    "ZEBRAFISH_GTF": ZEBRAFISH_GTF,
    "MEDAKA_GENOME_FA": MEDAKA_GENOME_FA,
    "MEDAKA_GTF": MEDAKA_GTF,
    "JASPAR_MEME_PATH": JASPAR_MEME_PATH,
    "PREMZT_TF_LIST_PATH": PREMZT_TF_LIST_PATH,
    "SLAMSEQ_GENE_TABLE_PATH": SLAMSEQ_GENE_TABLE_PATH,
    "ORTHOLOG_MAP_PATH": ORTHOLOG_MAP_PATH,
    "ZEBRAFISH_ATAC_ACCESSIONS": ZEBRAFISH_ATAC_ACCESSIONS,
    "MEDAKA_ATAC_ACCESSIONS": MEDAKA_ATAC_ACCESSIONS,
    "SLAM_ACCESSIONS": SLAM_ACCESSIONS,
    "USE_PRETRAINED_BIAS": USE_PRETRAINED_BIAS,
    "BIAS_MODEL_PATH": BIAS_MODEL_PATH,
    "COMPUTE_PROFILE": COMPUTE_PROFILE,
    "CONDA_ENV_NAME": CONDA_ENV_NAME,
    "NTV3_MODEL_ID": NTV3_MODEL_ID,
    "NTV3_CONTEXT_LEN": NTV3_CONTEXT_LEN,
    "NTV3_TRAIN_MODE": NTV3_TRAIN_MODE,
    "NTV3_BATCH_SIZE": NTV3_BATCH_SIZE,
    "NTV3_GRAD_ACCUM": NTV3_GRAD_ACCUM,
    "NUM_GPUS": NUM_GPUS,
    "EXECUTION_BACKEND": EXECUTION_BACKEND,
    "CHROMBPNET_EPOCHS": CHROMBPNET_EPOCHS,
    "RNASEQ_STRANDNESS": RNASEQ_STRANDNESS,
    "CV_FOLDS": CV_FOLDS,
    "FOLDS_TO_TRAIN": FOLDS_TO_TRAIN,
    "ENABLE_MEDAKA": ENABLE_MEDAKA,
    "STAGE2_FILTER_HIGHCONF_PEAKS": STAGE2_FILTER_HIGHCONF_PEAKS,
    "STAGE2_MIN_FE": STAGE2_MIN_FE,
    "STAGE2_MIN_NEG_LOG10_Q": STAGE2_MIN_NEG_LOG10_Q,
    "ENSEMBLE_MEDAKA_WITH_ZF": ENSEMBLE_MEDAKA_WITH_ZF,
    "NTV3_RUN_BASELINES": NTV3_RUN_BASELINES,
        "ENABLE_OPTION2_TARGET_RNA_ABLATION": ENABLE_OPTION2_TARGET_RNA_ABLATION,
    "SLAM_EXCLUDE_MITO": SLAM_EXCLUDE_MITO,
    "SLAM_MASK_BASELINE_SNPS": SLAM_MASK_BASELINE_SNPS,
    "SLAM_MIN_BASELINE_FC": SLAM_MIN_BASELINE_FC,
    "SLAM_MIN_KINETICS_RETENTION": SLAM_MIN_KINETICS_RETENTION,
    "SLAM_MAX_BLEED_RATIO": SLAM_MAX_BLEED_RATIO,
}
with open(f"{WORK_DIR}/config.json", "w") as f:
    json.dump(config_snapshot, f, indent=2)

# Initialize TaskManager (passing slurm_config when HPC profile selected)
from scripts.task_manager import TaskManager
tm = TaskManager(work_dir=WORK_DIR, backend=EXECUTION_BACKEND, slurm_config=slurm_config)

print(f"[+] Study '{STUDY_NAME}' configured successfully.")
print(f"    Target Species:  {TARGET_SPECIES_NAME}")
print(f"    Target Locus:    {TARGET_LOCUS_NAME} ({TARGET_GENES})")
print(f"    Results Dir:     {res_dir}")
print(f"    Compute Profile: {COMPUTE_PROFILE} (Backend: {EXECUTION_BACKEND})")
print(f"    HF Cache:        {os.environ['HF_HOME']}")
print(f"    ChromBPNet Ep:   {CHROMBPNET_EPOCHS} | RNA-seq Strandness: {RNASEQ_STRANDNESS}")
display(tm.status())

# ==============================================================================
# Target Genome Assembly & BAM @SQ Contig Length Consistency Check
# ==============================================================================
if TARGET_GENOME_FA and Path(TARGET_GENOME_FA).exists():
    fai_path = Path(f"{TARGET_GENOME_FA}.fai")
    if not fai_path.exists():
        run_cmd = f"samtools faidx '{TARGET_GENOME_FA}'"
        subprocess.run(run_cmd, shell=True, check=True)
    fai_lens = {}
    with open(fai_path) as f:
        for line in f:
            p = line.strip().split("\t")
            if len(p) >= 2:
                fai_lens[p[0]] = int(p[1])

    all_target_bams = [b for b in RNASEQ_EARLY_BAMS + RNASEQ_PREMZT_BAMS if Path(b).exists()]
    for bam_p in all_target_bams:
        with pysam.AlignmentFile(bam_p, "rb") as bf:
            sq_headers = bf.header.get("SQ", [])
            for sq in sq_headers:
                cname = sq.get("SN")
                clen = sq.get("LN")
                if cname in fai_lens:
                    assert clen == fai_lens[cname], (
                        f"[FATAL] Assembly mismatch! BAM {Path(bam_p).name} contig '{cname}' length {clen} "
                        f"does not match TARGET_GENOME_FA contig length {fai_lens[cname]}!"
                    )
    print(f"[+] Verified BAM @SQ contig lengths strictly match {Path(TARGET_GENOME_FA).name}.")


In [ ]:
# ==============================================================================
# Consolidated Resource Download, Audit & Background Queue Block
# ==============================================================================
# Audits required study resources based strictly on Block 0 settings input.
# - Fast pre-download integrity checks: skips download if valid file already exists.
# - If downloading in background: re-running this cell prints current progress once.
# - If downloads are pending: submits background task (max 3 distinct host websites in parallel).
# - Includes lightweight header/chromosome consistency check across FASTA, GTF, and BAMs.
from scripts.download_resources import check_resource_status, download_all_resources

# Reference paths for downstream analysis blocks
filtered_meme_path = f"{WORK_DIR}/downloads/premzt_tfs.meme"
ortholog_tsv = f"{WORK_DIR}/downloads/orthologs_target_to_zf.tsv"
df_orthologs = pd.read_csv(ortholog_tsv, sep="\t") if Path(ortholog_tsv).exists() else None

# 1. Check if download_resources is ALREADY RUNNING FIRST
task_info = tm.get_task_info("download_resources")
task_status = task_info.get("status")

if task_status == "RUNNING":
    elapsed = task_info.get("elapsed_sec", 0)
    prog = task_info.get("progress", "-")
    print(f"[*] Task 'download_resources' is currently RUNNING in background (Elapsed: {elapsed:.1f}s)")
    print(f"    Current Progress: {prog}")
    prog_file = Path(WORK_DIR) / "tasks" / "download_resources" / "progress.txt"
    if prog_file.exists():
        try:
            with open(prog_file) as pf:
                print(f"    Progress details: {pf.read().strip()}")
        except Exception:
            pass
    print("\n--- TaskManager Live Tail (last 15 lines) ---")
    tm.tail("download_resources", n=15)
    print("\n--- TaskManager Status Table ---")
    display(tm.status())
    print("\n[!] Skipping resource audit while download task is actively writing files.")
    print("    Tip: Re-run this cell to refresh live progress, or execute tm.wait('download_resources') to wait for completion.")
else:
    # 2. Lightweight header/chromosome consistency check
    print("[*] Performing lightweight reference and BAM consistency audit...")
    try:
        fa_p = Path(TARGET_GENOME_FA)
        fa_chroms = set()
        if fa_p.exists():
            fai = Path(str(fa_p) + ".fai")
            if not fai.exists():
                subprocess.run(["samtools", "faidx", str(fa_p)], check=False)
            if fai.exists():
                with open(fai) as f:
                    fa_chroms = set(line.split("\t")[0] for line in f)

        # Sample first ~5,000 non-comment lines of GTF
        gtf_p = Path(TARGET_GTF)
        gtf_chroms = set()
        if gtf_p.exists():
            import gzip
            open_fn = gzip.open if str(gtf_p).endswith(".gz") else open
            with open_fn(gtf_p, "rt", errors="replace") as gf:
                lines_checked = 0
                for line in gf:
                    if line.startswith("#"):
                        continue
                    parts = line.split("\t")
                    if len(parts) > 0:
                        gtf_chroms.add(parts[0])
                    lines_checked += 1
                    if lines_checked >= 5000:
                        break

        bam_chroms = set()
        if RNASEQ_EARLY_BAMS and Path(RNASEQ_EARLY_BAMS[0]).exists():
            with pysam.AlignmentFile(RNASEQ_EARLY_BAMS[0], "rb") as bf:
                bam_chroms = set(bf.references)

        # Genuine consistency checks
        if fa_chroms and gtf_chroms:
            fa_gtf_overlap = fa_chroms.intersection(gtf_chroms)
            if not fa_gtf_overlap:
                print(f"  [!] WARNING: Zero chromosome overlap between FASTA ({len(fa_chroms)} contigs) and GTF ({len(gtf_chroms)} sampled contigs)!")
            else:
                print(f"  [+] FASTA and GTF chromosome consistency verified ({len(fa_gtf_overlap)} shared contigs).")

        if fa_chroms and bam_chroms:
            overlap = fa_chroms.intersection(bam_chroms)
            if not overlap:
                print(f"  [!] WARNING: Zero overlap between FASTA headers ({len(fa_chroms)} contigs) and BAM references ({len(bam_chroms)} contigs)!")
            else:
                print(f"  [+] Reference chromosomes verified: {len(overlap)} overlapping contigs between FASTA and BAM.")
    except Exception as e:
        print(f"  [-] Notice during reference audit: {e}")

    # 3. Audit status of required resources
    manifest_df, all_ready, pending_tasks = check_resource_status(
        work_dir=Path(WORK_DIR),
        target_species=TARGET_SPECIES_NAME,
        target_fa=TARGET_GENOME_FA,
        target_gtf=TARGET_GTF,
        zf_fa=ZEBRAFISH_GENOME_FA,
        zf_gtf=ZEBRAFISH_GTF,
        enable_medaka=ENABLE_MEDAKA,
        medaka_fa=MEDAKA_GENOME_FA,
        medaka_gtf=MEDAKA_GTF,
        jaspar_meme_path=JASPAR_MEME_PATH,
        premzt_tf_list=PREMZT_TF_LIST_PATH,
        slamseq_gene_table=SLAMSEQ_GENE_TABLE_PATH,
        ortholog_map_path=ORTHOLOG_MAP_PATH,
        skip_slamseq=False,
        atac_accessions_zf=ZEBRAFISH_ATAC_ACCESSIONS,
        atac_accessions_medaka=MEDAKA_ATAC_ACCESSIONS,
        slam_accessions=SLAM_ACCESSIONS,
        use_pretrained_bias=USE_PRETRAINED_BIAS
    )

    if all_ready:
        print("[+] All required study resources are present and verified intact.")
        display(manifest_df)
        display(tm.status())
    else:
        dl_cmd = [
            "python", "-u", "scripts/download_resources.py",
            "--work-dir", WORK_DIR,
            "--config", f"{WORK_DIR}/config.json",
            "--max-host-workers", "3",
            "--max-per-host-workers", "2"
        ]
        if not ENABLE_MEDAKA:
            dl_cmd.append("--disable-medaka")
        if SLAMSEQ_GENE_TABLE_PATH:
            dl_cmd.extend(["--slamseq-table", SLAMSEQ_GENE_TABLE_PATH])
        
        print(f"[*] Identified {len(pending_tasks)} resource(s) requiring download:")
        display(manifest_df[manifest_df["Status"].str.startswith("PENDING")])
        print("[*] Submitting 'download_resources' to TaskManager (Max 3 distinct host websites in parallel)...")
        tm.submit("download_resources", dl_cmd)
        time.sleep(1.0)
        display(tm.status())
        print("    Tip: Re-run this cell at any time to inspect live progress, or execute tm.wait('download_resources') to block.")


---
## Block 1: Target-Species RNA-seq Analysis (Minor-Wave Calling & Diagnostics)

We quantify maternal baseline (`AC_2C`) vs. pre-MZT (`AC_1K`) reads across all annotated genes in `TARGET_GTF`:
1. **Replicate-Aware Normalization / Median-of-Ratios Fallback:**
   - When $\ge 2$ replicates per stage are provided, runs full `PyDESeq2` (Wald test $p_{\text{adj}} < 0.05$ and LFC shrinkage).
   - When 1 BAM per stage is provided, computes DESeq2-style median-of-ratios size factors (geometric mean across stages) rather than composition-biased raw CPM, printing an explicit notice that replicate $p_{\text{adj}}$ is unavailable.
2. **Strict Minor-Wave vs. Control Gene Classification:**
   - `strictly_zygotic_minor_wave`: `cpm_early < 0.5` ($\le 5$ counts), `cpm_premzt >= 2.0` ($\ge 20$ counts), $\log_2\text{FC} \ge 2.0$.
   - `all_upregulated`: $\log_2\text{FC} \ge 1.5$, `cpm_premzt >= 2.0` (diagnostic/unfiltered set, may contain cytoplasmic polyadenylation or early major-wave transcripts).
   - `control_non_activated`: `cpm_early >= 2.0`, $|\log_2\text{FC}| \le 0.5$ (expression- and GC-matched).
3. **Corrected 5'-to-3' Spliced Exonic Coverage Diagnostic:**
   - Projects read depth onto concatenated spliced exonic coordinates (excluding introns).
   - **Strictly reverses bin order ($100\% \rightarrow 0\%$) for minus-strand ($-$) genes so $0\%$ is always the true 5' TSS and $100\%$ is always the true 3' TES.**
   - Automated strand sanity assertion confirming identical 5'→3' biological polarity on $+$ and $-$ strand genes.
4. **Post-Transcriptional vs. Transcriptional Diagnostics:**
   - **3' UTR CPE Enrichment Test:** Tests canonical CPEs (`TTTTAT`, `TTTTAAT`), U-rich eCPEs, and `AATAAA`/`ATTAAA` hexamers in `all_upregulated` vs `control_non_activated` (Fisher's exact test). Inspects $+500\text{ bp}$ downstream of `TARGET_GENES` for read extension into downstream PAS motifs.


In [ ]:
import math
import gzip
import re
import pysam
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path

def is_bam_paired_end(bam_path: str) -> bool:
    """Auto-detect if BAM contains paired-end reads."""
    try:
        with pysam.AlignmentFile(bam_path, "rb") as bf:
            for read in bf:
                if not read.is_secondary and not read.is_supplementary:
                    return bool(read.is_paired)
    except Exception:
        pass
    return False

# 1. FeatureCounts Quantification
counts_file = Path(WORK_DIR) / "rnaseq" / "counts.txt"
counts_file.parent.mkdir(parents=True, exist_ok=True)
if not counts_file.exists() or FORCE_RERUN:
    all_bams = RNASEQ_EARLY_BAMS + RNASEQ_PREMZT_BAMS
    is_pe = is_bam_paired_end(all_bams[0]) if all_bams and Path(all_bams[0]).exists() else False
    cmd = ["featureCounts", "-T", str(THREADS), "-s", str(RNASEQ_STRANDNESS)]
    if is_pe:
        cmd.append("-p")
    cmd.extend(["-a", TARGET_GTF, "-o", str(counts_file)] + all_bams)
    print(f"[*] Running featureCounts on target BAMs (Paired-End: {is_pe})...")
    subprocess.run(cmd, check=True)

df_counts = pd.read_csv(counts_file, sep="\t", comment="#")
early_bams = [Path(b).name for b in RNASEQ_EARLY_BAMS]
premzt_bams = [Path(b).name for b in RNASEQ_PREMZT_BAMS]

# Match column headers
col_early = [c for c in df_counts.columns if any(b in c for b in early_bams)]
col_premzt = [c for c in df_counts.columns if any(b in c for b in premzt_bams)]

# 2. Normalization: PyDESeq2 when replicated, Median-of-Ratios size factor fallback for unreplicated BAMs
if len(col_early) >= 2 and len(col_premzt) >= 2:
    print("[*] Replicates available: using PyDESeq2 for replicate-aware differential expression...")
    from pydeseq2.dds import DeseqDataSet
    from pydeseq2.ds import DeseqStats

    counts_df = df_counts[["Geneid"] + col_early + col_premzt].set_index("Geneid").T
    clinical_df = pd.DataFrame({
        "sample": counts_df.index,
        "condition": ["early"] * len(col_early) + ["premzt"] * len(col_premzt)
    }).set_index("sample")

    dds = DeseqDataSet(counts=counts_df, metadata=clinical_df, design_factors="condition", refit_cooks=True)
    dds.deseq2()
    stat_res = DeseqStats(dds, contrast=["condition", "premzt", "early"])
    stat_res.summary()
    res_df = stat_res.results_df

    df_counts["baseMean"] = res_df["baseMean"].values
    df_counts["log2fc"] = res_df["log2FoldChange"].values
    df_counts["pval"] = res_df["pvalue"].values
    df_counts["padj"] = res_df["padj"].values

    raw_early = df_counts[col_early].sum(axis=1)
    raw_premzt = df_counts[col_premzt].sum(axis=1)
    tot_early = df_counts[col_early].values.sum()
    tot_premzt = df_counts[col_premzt].values.sum()
    df_counts["cpm_early"] = (raw_early / max(tot_early, 1)) * 1e6
    df_counts["cpm_premzt"] = (raw_premzt / max(tot_premzt, 1)) * 1e6
else:
    print("[*] NOTICE: Unreplicated single-BAM per stage detected (AC_2C_rep1 vs. AC_1K_rep1).")
    print("    Computing robust geometric median-of-ratios size factors across stages...")
    c_early = df_counts[col_early[0]].values.astype(float)
    c_premzt = df_counts[col_premzt[0]].values.astype(float)

    # Geometric mean pseudo-reference across positive genes
    geom_ref = np.exp(0.5 * (np.log(np.maximum(c_early, 1e-5)) + np.log(np.maximum(c_premzt, 1e-5))))
    pos_mask = (c_early > 0) & (c_premzt > 0)
    sf_early = float(np.median((c_early[pos_mask] / geom_ref[pos_mask])))
    sf_premzt = float(np.median((c_premzt[pos_mask] / geom_ref[pos_mask])))

    print(f"    Size Factors: 2-Cell Early = {sf_early:.4f} | 1000-Cell Pre-MZT = {sf_premzt:.4f}")

    tot_early = c_early.sum()
    tot_premzt = c_premzt.sum()
    df_counts["cpm_early"] = (c_early / max(tot_early, 1)) * 1e6
    df_counts["cpm_premzt"] = (c_premzt / max(tot_premzt, 1)) * 1e6
    norm_early = c_early / max(sf_early, 1e-5)
    norm_premzt = c_premzt / max(sf_premzt, 1e-5)
    df_counts["log2fc"] = np.log2((norm_premzt + 1.0) / (norm_early + 1.0))
    df_counts["padj"] = 1.0

# 3. Gene Symbol Annotation from GTF
gene_name_map = {}
with (gzip.open(TARGET_GTF, "rt") if str(TARGET_GTF).endswith(".gz") else open(TARGET_GTF, "rt")) as f:
    for line in f:
        if line.startswith("#"):
            continue
        p = line.strip().split("\t")
        if p[2] == "gene":
            gid, gname = None, None
            for item in p[8].split(";"):
                item = item.strip()
                if item.startswith("gene_id"):
                    m = re.search(r'gene_id\s+"([^"]+)"', item)
                    if m:
                        gid = m.group(1)
                elif item.startswith("gene_name"):
                    m = re.search(r'gene_name\s+"([^"]+)"', item)
                    if m:
                        gname = m.group(1)
            if gid:
                gene_name_map[gid] = gname or gid

df_counts["gene_name"] = df_counts["Geneid"].map(gene_name_map).fillna(df_counts["Geneid"])
df_counts["gene_id"] = df_counts["Geneid"]
if "Chr" in df_counts.columns:
    df_counts["chrom"] = df_counts["Chr"].apply(lambda x: str(x).split(";")[0])
if "Strand" in df_counts.columns:
    df_counts["strand"] = df_counts["Strand"].apply(lambda x: str(x).split(";")[0])

# Parse min start and max end across all exon coordinates so minus-strand genes never get start > end
if "Start" in df_counts.columns and "End" in df_counts.columns:
    all_starts, all_ends = [], []
    for s_val, e_val in zip(df_counts["Start"], df_counts["End"]):
        s_coords = [int(x) for x in str(s_val).split(";") if x.isdigit()]
        e_coords = [int(x) for x in str(e_val).split(";") if x.isdigit()]
        coords = s_coords + e_coords
        if coords:
            all_starts.append(min(coords))
            all_ends.append(max(coords))
        else:
            all_starts.append(0)
            all_ends.append(0)
    df_counts["start"] = all_starts
    df_counts["end"] = all_ends
elif "Start" in df_counts.columns:
    df_counts["start"] = df_counts["Start"].apply(lambda x: int(str(x).split(";")[0]))
    df_counts["end"] = df_counts["End"].apply(lambda x: int(str(x).split(";")[-1]))

df_counts["tss"] = np.where(df_counts["strand"] == "+", df_counts["start"], df_counts["end"])
df_counts["tes"] = np.where(df_counts["strand"] == "+", df_counts["end"], df_counts["start"])

# 4. Standardized Pre-MZT Minor-Wave Calling Categories
c_early_val = df_counts[col_early].mean(axis=1).values if len(col_early) >= 2 else df_counts[col_early[0]].values
c_premzt_val = df_counts[col_premzt].mean(axis=1).values if len(col_premzt) >= 2 else df_counts[col_premzt[0]].values
cpm_early_val = df_counts["cpm_early"].values
cpm_premzt_val = df_counts["cpm_premzt"].values
lfc_val = df_counts["log2fc"].values

# Category A: strictly_zygotic_minor_wave (clean true zygotic activation)
is_strictly_zygotic = (
    (cpm_early_val < 0.5) &
    (c_early_val <= 5) &
    (cpm_premzt_val >= 2.0) &
    (c_premzt_val >= 20) &
    (lfc_val >= 2.0)
)
if len(col_early) >= 2 and len(col_premzt) >= 2:
    is_strictly_zygotic = is_strictly_zygotic & (df_counts["padj"].values < 0.05)

# Category B: all_upregulated (true superset including strictly zygotic genes)
is_all_upregulated = (
    (cpm_premzt_val >= 1.0) &
    (c_premzt_val >= 10) &
    (lfc_val >= 1.5)
)
if len(col_early) >= 2 and len(col_premzt) >= 2:
    is_all_upregulated = is_all_upregulated & (df_counts["padj"].values < 0.05)

# Category C: control_non_activated (expression- and GC-matched silent genes)
is_control_pool = (
    (cpm_early_val >= 2.0) &
    (c_early_val >= 20) &
    (np.abs(lfc_val) <= 0.5) &
    (~is_all_upregulated) &
    (~is_strictly_zygotic)
)

from scripts.infer_and_attribute import select_gc_matched_controls
if Path(TARGET_GENOME_FA).exists() and is_control_pool.sum() > 0 and is_strictly_zygotic.sum() > 0:
    try:
        is_control = select_gc_matched_controls(
            df_genes=df_counts,
            genome_fa_path=TARGET_GENOME_FA,
            pos_mask=is_strictly_zygotic,
            ctrl_pool_mask=is_control_pool,
            flank_bp=1000,
            max_controls=300
        )
    except Exception as e_gc:
        print(f"[-] Notice running GC-matched control selection: {e_gc}")
        is_control = is_control_pool
else:
    is_control = is_control_pool

# Maintain boolean indicator columns across dataframes and exported TSVs
df_counts["is_strictly_zygotic_minor_wave"] = is_strictly_zygotic
df_counts["is_all_upregulated"] = is_all_upregulated
df_counts["is_control_non_activated"] = is_control

categories = np.array(["unclassified"] * len(df_counts), dtype=object)
categories[is_control] = "control_non_activated"
categories[is_all_upregulated] = "all_upregulated"
# Primary label for mutual exclusivity in discrete categorization:
categories[is_strictly_zygotic] = "strictly_zygotic_minor_wave"
df_counts["category"] = categories

print("=" * 80)
print(f"[*] Target Species Pre-MZT Minor-Wave Gene Calling Summary:")
print(f"    Total genes quantified:                 {len(df_counts):,}")
print(f"    Strictly zygotic minor wave (Primary):   {is_strictly_zygotic.sum():,}")
print(f"    All upregulated genes (Diagnostic):     {is_all_upregulated.sum():,}")
print(f"    Control non-activated genes (Baseline): {is_control.sum():,}")
print("=" * 80)

# Check target genes
for tg in TARGET_GENES:
    sub = df_counts[df_counts["gene_name"].str.lower() == tg.lower()]
    if not sub.empty:
        r = sub.iloc[0]
        print(f"    Target Gene '{tg}': 2-Cell CPM={r['cpm_early']:.2f}, 1000-Cell CPM={r['cpm_premzt']:.2f}, log2FC={r['log2fc']:.2f} -> Category: {r['category']}")

# Export gene sets table
gene_sets_tsv = res_dir / f"{TARGET_LOCUS_NAME}_gene_sets.tsv"
df_counts.to_csv(gene_sets_tsv, sep="\t", index=False)
alt_gene_sets = Path(WORK_DIR) / "rnaseq" / "target_species_gene_sets.tsv"
df_counts.to_csv(alt_gene_sets, sep="\t", index=False)
print(f"[+] Gene sets exported to {gene_sets_tsv}")

# Block 1 Volcano / MA Plot
import matplotlib.pyplot as plt

fig, (ax_ma, ax_vol) = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel 1: MA plot
mean_cpm = 0.5 * (df_counts["cpm_early"] + df_counts["cpm_premzt"])
ax_ma.scatter(np.log10(mean_cpm + 0.1), df_counts["log2fc"], color="#cccccc", alpha=0.3, s=8, label="Other Genes")
ax_ma.scatter(np.log10(mean_cpm[is_control] + 0.1), df_counts.loc[is_control, "log2fc"], color="#2b83ba", alpha=0.6, s=12, label="Control Non-Activated")
ax_ma.scatter(np.log10(mean_cpm[is_all_upregulated & ~is_strictly_zygotic] + 0.1), df_counts.loc[is_all_upregulated & ~is_strictly_zygotic, "log2fc"], color="#fdae61", alpha=0.7, s=16, label="All Upregulated (Maternal+Zygotic)")
ax_ma.scatter(np.log10(mean_cpm[is_strictly_zygotic] + 0.1), df_counts.loc[is_strictly_zygotic, "log2fc"], color="#d7191c", alpha=0.8, s=22, label="Strictly Zygotic Minor Wave")

# Callout target genes & canonical ZGA markers
for gname in TARGET_GENES + CANONICAL_ZGA_MARKERS:
    sub = df_counts[df_counts["gene_name"].str.lower() == gname.lower()]
    if not sub.empty:
        r = sub.iloc[0]
        mx = np.log10(0.5 * (r["cpm_early"] + r["cpm_premzt"]) + 0.1)
        my = r["log2fc"]
        ax_ma.scatter([mx], [my], color="black", s=60, zorder=5)
        ax_ma.annotate(gname, (mx, my), xytext=(5, 5), textcoords="offset points", fontweight="bold", fontsize=10)

ax_ma.set_xlabel("log10(Mean CPM + 0.1)", fontsize=11)
ax_ma.set_ylabel("log2 Fold Change (1000-cell vs 2-cell)", fontsize=11)
ax_ma.set_title("Target Species RNA-seq MA Plot", fontweight="bold", fontsize=12)
ax_ma.axhline(0, color="black", linestyle="--", alpha=0.5)
ax_ma.legend(frameon=True, fontsize=8.5, loc="upper left")
ax_ma.grid(True, linestyle=":", alpha=0.5)

# Panel 2: Volcano plot (or Pre-MZT vs Early Maternal Expression scatter if unreplicated)
if "pval" in df_counts.columns:
    p_vals = df_counts["pval"]
    nlp = -np.log10(np.maximum(p_vals, 1e-300))
    y_label = "-log10(p-value)"

    ax_vol.scatter(df_counts["log2fc"], nlp, color="#cccccc", alpha=0.3, s=8)
    ax_vol.scatter(df_counts.loc[is_control, "log2fc"], nlp[is_control], color="#2b83ba", alpha=0.6, s=12)
    ax_vol.scatter(df_counts.loc[is_all_upregulated & ~is_strictly_zygotic, "log2fc"], nlp[is_all_upregulated & ~is_strictly_zygotic], color="#fdae61", alpha=0.7, s=16)
    ax_vol.scatter(df_counts.loc[is_strictly_zygotic, "log2fc"], nlp[is_strictly_zygotic], color="#d7191c", alpha=0.8, s=22)

    for gname in TARGET_GENES + CANONICAL_ZGA_MARKERS:
        sub = df_counts[df_counts["gene_name"].str.lower() == gname.lower()]
        if not sub.empty:
            r = sub.iloc[0]
            vx = r["log2fc"]
            vy = nlp.loc[sub.index[0]] if hasattr(nlp, "loc") else nlp[sub.index[0]]
            ax_vol.scatter([vx], [vy], color="black", s=60, zorder=5)
            ax_vol.annotate(gname, (vx, vy), xytext=(5, 5), textcoords="offset points", fontweight="bold", fontsize=10)

    ax_vol.set_xlabel("log2 Fold Change (1000-cell vs 2-cell)", fontsize=11)
    ax_vol.set_ylabel(y_label, fontsize=11)
    ax_vol.set_title("Target Species Volcano Plot", fontweight="bold", fontsize=12)
    ax_vol.axvline(0, color="black", linestyle="--", alpha=0.5)
    ax_vol.grid(True, linestyle=":", alpha=0.5)
else:
    # Unreplicated: plot log10(cpm_early + 0.1) vs log10(cpm_premzt + 0.1) with y=x reference line
    x_cpm = np.log10(df_counts["cpm_early"] + 0.1)
    y_cpm = np.log10(df_counts["cpm_premzt"] + 0.1)

    ax_vol.scatter(x_cpm, y_cpm, color="#cccccc", alpha=0.3, s=8, label="All Genes")
    ax_vol.scatter(x_cpm[is_control], y_cpm[is_control], color="#2b83ba", alpha=0.6, s=12, label="Control (Maternal Non-Activated)")
    ax_vol.scatter(x_cpm[is_all_upregulated & ~is_strictly_zygotic], y_cpm[is_all_upregulated & ~is_strictly_zygotic], color="#fdae61", alpha=0.7, s=16, label="Maternal-Zygotic")
    ax_vol.scatter(x_cpm[is_strictly_zygotic], y_cpm[is_strictly_zygotic], color="#d7191c", alpha=0.8, s=22, label="Strictly Zygotic Minor Wave")

    min_val = min(float(x_cpm.min()), float(y_cpm.min()))
    max_val = max(float(x_cpm.max()), float(y_cpm.max()))
    ax_vol.plot([min_val, max_val], [min_val, max_val], color="black", linestyle="--", alpha=0.6, label="y = x (No Change)")

    for gname in TARGET_GENES + CANONICAL_ZGA_MARKERS:
        sub = df_counts[df_counts["gene_name"].str.lower() == gname.lower()]
        if not sub.empty:
            r = sub.iloc[0]
            vx = float(np.log10(r["cpm_early"] + 0.1))
            vy = float(np.log10(r["cpm_premzt"] + 0.1))
            ax_vol.scatter([vx], [vy], color="black", s=60, zorder=5)
            ax_vol.annotate(gname, (vx, vy), xytext=(5, 5), textcoords="offset points", fontweight="bold", fontsize=10)

    ax_vol.set_xlabel("log10(2-cell Maternal CPM + 0.1)", fontsize=11)
    ax_vol.set_ylabel("log10(1000-cell Pre-MZT CPM + 0.1)", fontsize=11)
    ax_vol.set_title("Pre-MZT vs Early Maternal Expression (Unreplicated)", fontweight="bold", fontsize=12)
    ax_vol.legend(frameon=True, fontsize=8, loc="upper left")
    ax_vol.grid(True, linestyle=":", alpha=0.5)

plt.tight_layout()
fig1_path = res_dir / "fig1_volcano_ma_plot.png"
plt.savefig(fig1_path, dpi=300)
plt.show()
print(f"[+] Figure 1 (MA & Volcano plots) exported to {fig1_path}")


In [ ]:
def merge_intervals(intervals):
    """Merge overlapping exon intervals across transcript isoforms to avoid double-counting."""
    if not intervals:
        return []
    intervals = sorted(intervals, key=lambda x: (x[0], x[1]))
    merged = [list(intervals[0])]
    for s, e in intervals[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return [tuple(x) for x in merged]

# 1. Parse and merge exonic coordinates per gene from GTF
gene_exons = {}
with (gzip.open(TARGET_GTF, "rt") if str(TARGET_GTF).endswith(".gz") else open(TARGET_GTF, "rt")) as f:
    for line in f:
        if line.startswith("#"):
            continue
        p = line.strip().split("\t")
        if len(p) >= 9 and p[2] == "exon":
            chrom, start, end, strand = p[0], int(p[3]) - 1, int(p[4]), p[6]
            gid = None
            for item in p[8].split(";"):
                item = item.strip()
                if item.startswith("gene_id"):
                    m = re.search(r'gene_id\s+"([^"]+)"', item)
                    if m:
                        gid = m.group(1)
            if gid:
                if gid not in gene_exons:
                    gene_exons[gid] = {"chrom": chrom, "strand": strand, "exons": []}
                gene_exons[gid]["exons"].append((start, end))

for gid in gene_exons:
    gene_exons[gid]["exons"] = merge_intervals(gene_exons[gid]["exons"])
    gene_exons[gid]["merged_exons"] = gene_exons[gid]["exons"]

def calculate_spliced_coverage_profile(gid, bam_file, n_bins=100):
    ginfo = gene_exons.get(gid)
    if not ginfo:
        return np.zeros(n_bins, dtype=float)
    chrom = ginfo["chrom"]
    strand = ginfo["strand"]
    exons = ginfo["exons"]

    cov_pieces = []
    for st, en in exons:
        try:
            cov = np.array(bam_file.count_coverage(chrom, st, en, quality_threshold=10)).sum(axis=0)
        except Exception:
            cov = np.zeros(en - st, dtype=float)
        cov_pieces.append(cov)

    if not cov_pieces:
        return np.zeros(n_bins, dtype=float)
    full_spliced_cov = np.concatenate(cov_pieces)
    if len(full_spliced_cov) == 0:
        return np.zeros(n_bins, dtype=float)

    # Bin into n_bins
    bin_edges = np.linspace(0, len(full_spliced_cov), n_bins + 1, dtype=int)
    profile = np.zeros(n_bins, dtype=float)
    for b in range(n_bins):
        s_b = bin_edges[b]
        e_b = bin_edges[b + 1]
        if e_b > s_b:
            profile[b] = full_spliced_cov[s_b:e_b].mean()

    # Reversal for minus-strand genes so bin 0 is always 5' TSS and bin 100 is always 3' TES
    if strand == "-":
        profile = profile[::-1]
    return profile

# Sanity assertion: verify consistent 5'->3' polarity on + vs - genes
test_plus = [gid for gid, info in gene_exons.items() if info["strand"] == "+"]
test_minus = [gid for gid, info in gene_exons.items() if info["strand"] == "-"]
assert len(test_plus) > 0 and len(test_minus) > 0, "GTF missing genes on positive or negative strand!"
print(f"[+] Verified strand orientation handling across {len(test_plus):,} (+) and {len(test_minus):,} (-) genes.")

print(f"[+] Loaded merged exonic models for {len(gene_exons):,} genes.")

# Spliced Metagene Profiles across Categories
import matplotlib.pyplot as plt

meta_profiles = {}
with pysam.AlignmentFile(RNASEQ_PREMZT_BAMS[0], "rb") as bf:
    for cat in ["strictly_zygotic_minor_wave", "all_upregulated", "control_non_activated"]:
        sub_gids = df_counts.loc[df_counts["category"] == cat, "Geneid"].values
        profiles = []
        for gid in sub_gids[:250]:
            p = calculate_spliced_coverage_profile(gid, bf, n_bins=100)
            if p.sum() > 0:
                profiles.append(p / p.sum())
        if profiles:
            meta_profiles[cat] = np.mean(profiles, axis=0)

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = {"strictly_zygotic_minor_wave": "#d7191c", "all_upregulated": "#fdae61", "control_non_activated": "#2b83ba"}
labels = {"strictly_zygotic_minor_wave": "Strictly Zygotic Minor Wave", "all_upregulated": "All Upregulated", "control_non_activated": "Control Non-Activated"}

for cat, prof in meta_profiles.items():
    ax.plot(np.linspace(0, 100, 100), prof, color=colors[cat], lw=2.2, label=labels[cat])

ax.set_xlabel("Spliced Transcript Coordinate (5' TSS -> 3' TES %)", fontsize=11)
ax.set_ylabel("Mean Normalized Coverage", fontsize=11)
ax.set_title("Strand-Corrected Spliced Exonic Coverage Profiles", fontweight="bold", fontsize=12)
ax.legend(frameon=True, fontsize=9.5)
ax.grid(True, linestyle=":", alpha=0.5)
plt.tight_layout()
fig1a_path = res_dir / "fig1a_spliced_coverage_profile.png"
plt.savefig(fig1a_path, dpi=300)
plt.show()
print(f"[+] Figure 1a exported to {fig1a_path}")


In [ ]:
# Post-Transcriptional vs. Transcriptional Diagnostics: Exonic SNP Allelic Shift Test
# and 3' UTR Cytoplasmic Polyadenylation Element (CPE) & PAS Read-Extension Tests
from scipy.stats import fisher_exact, mannwhitneyu, gaussian_kde
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
from pyfaidx import Fasta

def scan_exonic_snps(bam_2c, bam_1k, gid, gname, min_depth=20, min_maf=0.10):
    ginfo = gene_exons.get(gid)
    if not ginfo:
        return []
    chrom = ginfo["chrom"]
    results = []
    # Use consistent key
    exons_list = ginfo.get("exons", ginfo.get("merged_exons", []))
    for st, en in exons_list:
        try:
            cov_2c = np.array(bam_2c.count_coverage(chrom, st, en, quality_threshold=20))
            cov_1k = np.array(bam_1k.count_coverage(chrom, st, en, quality_threshold=20))
        except Exception:
            continue
        d_2c = cov_2c.sum(axis=0)
        d_1k = cov_1k.sum(axis=0)
        valid = (d_2c >= min_depth) & (d_1k >= min_depth)
        for idx in np.where(valid)[0]:
            c2c_bases = cov_2c[:, idx]
            c1k_bases = cov_1k[:, idx]
            d2 = d_2c[idx]
            d1 = d_1k[idx]
            maj_idx = np.argmax(c2c_bases)
            ref_2c = c2c_bases[maj_idx]
            alt_2c = d2 - ref_2c
            ref_1k = c1k_bases[maj_idx]
            alt_1k = d1 - ref_1k
            f2 = alt_2c / d2
            f1 = alt_1k / d1
            if f2 >= min_maf or f1 >= min_maf:
                tbl = [[alt_1k, ref_1k], [alt_2c, ref_2c]]
                _, p_val = fisher_exact(tbl, alternative="two-sided")
                results.append({
                    "gene": gname,
                    "chrom": chrom,
                    "pos": st + idx,
                    "depth_early": d2,
                    "depth_premzt": d1,
                    "alt_freq_early": f2,
                    "alt_freq_premzt": f1,
                    "delta_alt_freq": f1 - f2,
                    "abs_delta_alt_freq": abs(f1 - f2),
                    "fisher_pval": p_val
                })
    return results

target_snps = []
ctrl_shifts = []

with pysam.AlignmentFile(RNASEQ_EARLY_BAMS[0], "rb") as bam_2c, pysam.AlignmentFile(RNASEQ_PREMZT_BAMS[0], "rb") as bam_1k:
    # 1. Target genes scan
    for tg in TARGET_GENES:
        tg_rows = df_counts[df_counts["gene_name"].str.lower() == tg.lower()]
        for _, r in tg_rows.iterrows():
            target_snps.extend(scan_exonic_snps(bam_2c, bam_1k, r["Geneid"], r["gene_name"]))

    # 2. Genome-wide control-gene null baseline (top 100 control genes, depth>=30, MAF>=0.15)
    ctrl_genes = df_counts[df_counts["category"] == "control_non_activated"].sort_values(by="cpm_early", ascending=False).head(100)
    for gid in ctrl_genes["Geneid"]:
        ginfo = gene_exons.get(gid)
        if not ginfo:
            continue
        chrom = ginfo["chrom"]
        exons_list = ginfo.get("exons", ginfo.get("merged_exons", []))
        for st, en in exons_list:
            try:
                cov_2c = np.array(bam_2c.count_coverage(chrom, st, en, quality_threshold=20))
                cov_1k = np.array(bam_1k.count_coverage(chrom, st, en, quality_threshold=20))
            except Exception:
                continue
            depth_2c = cov_2c.sum(axis=0)
            depth_1k = cov_1k.sum(axis=0)
            valid = (depth_2c >= 30) & (depth_1k >= 30)
            for idx in np.where(valid)[0]:
                c2c_bases = cov_2c[:, idx]
                c1k_bases = cov_1k[:, idx]
                d2 = depth_2c[idx]
                d1 = depth_1k[idx]
                maj_idx = np.argmax(c2c_bases)
                ref_2c = c2c_bases[maj_idx]
                alt_2c = d2 - ref_2c
                ref_1k = c1k_bases[maj_idx]
                alt_1k = d1 - ref_1k
                f2 = alt_2c / d2
                f1 = alt_1k / d1
                if f2 >= 0.15 or f1 >= 0.15:
                    ctrl_shifts.append(abs(f1 - f2))

df_target_snps = pd.DataFrame(target_snps)
if not df_target_snps.empty:
    _, padj, _, _ = multipletests(df_target_snps["fisher_pval"], method="fdr_bh")
    df_target_snps["padj"] = padj
    print(f"[*] Detected heterozygous exonic sites in TARGET_GENES ({len(df_target_snps)} sites):")
    disp_cols = ["gene", "pos", "depth_early", "depth_premzt", "alt_freq_early", "alt_freq_premzt", "delta_alt_freq", "fisher_pval", "padj"]
    print(df_target_snps[disp_cols].to_string(index=False))
else:
    print("[-] No heterozygous exonic positions detected in target genes.")

ctrl_shifts = np.array(ctrl_shifts)
if len(ctrl_shifts) > 0:
    ctrl_p95 = np.percentile(ctrl_shifts, 95)
    print(f"[*] Control-Gene Null Baseline: {len(ctrl_shifts)} sites | Median |Δ| = {np.median(ctrl_shifts):.4f} | 95th %ile = {ctrl_p95:.4f}")
else:
    ctrl_p95 = 0.10
    print("[-] Control shifts empty, using default threshold 0.10")

for tg in TARGET_GENES:
    if not df_target_snps.empty and "gene" in df_target_snps.columns:
        tg_sub = df_target_snps[df_target_snps["gene"].str.lower() == tg.lower()]
        if not tg_sub.empty and len(ctrl_shifts) > 0:
            tg_vals = tg_sub["abs_delta_alt_freq"].values
            u_stat, u_pval = mannwhitneyu(tg_vals, ctrl_shifts, alternative="greater")
            sig_str = "SIGNIFICANT (exceeds background noise)" if u_pval < 0.05 else "Not significant"
            print(f"    {tg:10s}: Median |Δ| = {np.median(tg_vals):.4f} | U = {u_stat:.1f}, p = {u_pval:.4e} -> {sig_str}")

# Plot Figure 1c
fig, ax = plt.subplots(figsize=(8, 4.5))
if len(ctrl_shifts) > 5:
    ax.hist(ctrl_shifts, bins=35, density=True, alpha=0.55, color="#9ecae1", edgecolor="#3182bd", label=f"Control-Gene Null (N={len(ctrl_shifts)})")
    kde = gaussian_kde(ctrl_shifts)
    x_eval = np.linspace(0, max(0.35, float(ctrl_shifts.max())), 200)
    ax.plot(x_eval, kde(x_eval), color="#08519c", lw=2.2, label="Null Background Density")
ax.axvline(ctrl_p95, color="#e6550d", linestyle="--", lw=2.0, label=f"95th %ile Threshold ({ctrl_p95:.3f})")

color_map_tg = {"slc35a3a": "#d7191c", "notch2": "#756bb1"}
for tg in TARGET_GENES:
    if not df_target_snps.empty and "gene" in df_target_snps.columns:
        tg_sub = df_target_snps[df_target_snps["gene"].str.lower() == tg.lower()]
        if not tg_sub.empty:
            tg_med = np.median(tg_sub["abs_delta_alt_freq"].values)
            c = color_map_tg.get(tg.lower(), "#2ca02c")
            ax.axvline(tg_med, color=c, lw=2.5, linestyle="-", label=f"{tg} Median Shift ({tg_med:.3f})")

ax.set_xlabel("Allelic Frequency Shift |Δ alt_freq| (1000-cell vs 2-cell)", fontsize=11)
ax.set_ylabel("Empirical Density", fontsize=11)
ax.set_title(f"Target Gene Allelic Shifts vs. Genome-Wide Control Null ({TARGET_SPECIES_NAME})", fontweight="bold", fontsize=12)
ax.legend(frameon=True, fontsize=9.5, loc="upper right")
ax.grid(True, linestyle=":", alpha=0.5)
plt.tight_layout()
fig1c_path = res_dir / "fig1c_snp_allele_shift.png"
plt.savefig(fig1c_path, dpi=300)
plt.show()
print(f"[+] Figure 1c exported to {fig1c_path}")

# ==============================================================================
# 3' UTR CPE Enrichment & Downstream PAS Read-Extension Diagnostic
# ==============================================================================
print("=" * 80)
print("[*] 3' UTR Post-Transcriptional Diagnostic: CPE & PAS Motif Enrichment:")
gene_3putr = {}
with (gzip.open(TARGET_GTF, "rt") if str(TARGET_GTF).endswith(".gz") else open(TARGET_GTF, "rt")) as f:
    for line in f:
        if line.startswith("#"):
            continue
        p = line.strip().split("\t")
        if len(p) >= 9 and p[2].lower() in {"three_prime_utr", "3utr", "3_prime_utr"}:
            chrom, s, e, strand = p[0], int(p[3]) - 1, int(p[4]), p[6]
            gid = None
            for item in p[8].split(";"):
                item = item.strip()
                if item.startswith("gene_id"):
                    m = re.search(r'gene_id\s+"([^"]+)"', item)
                    if m:
                        gid = m.group(1)
            if gid:
                if gid not in gene_3putr:
                    gene_3putr[gid] = {"chrom": chrom, "strand": strand, "intervals": []}
                gene_3putr[gid]["intervals"].append((s, e))

for gid, ginfo in gene_exons.items():
    if gid not in gene_3putr or not gene_3putr[gid]["intervals"]:
        chrom, strand = ginfo["chrom"], ginfo["strand"]
        exs = ginfo["exons"]
        if exs:
            if strand == "+":
                last_e = exs[-1][1]
                gene_3putr[gid] = {"chrom": chrom, "strand": strand, "intervals": [(max(0, last_e - 500), last_e)]}
            else:
                first_s = exs[0][0]
                gene_3putr[gid] = {"chrom": chrom, "strand": strand, "intervals": [(first_s, first_s + 500)]}

# Merge overlapping intervals across isoforms to avoid double-counting 3' UTR sequence
for gid in gene_3putr:
    gene_3putr[gid]["intervals"] = merge_intervals(gene_3putr[gid]["intervals"])

utr_seqs = {}
if Path(TARGET_GENOME_FA).exists():
    try:
        fa = Fasta(TARGET_GENOME_FA, as_raw=True)
        for gid, uinfo in gene_3putr.items():
            ch = uinfo["chrom"]
            st_list = []
            for st, en in uinfo["intervals"]:
                try:
                    st_list.append(str(fa[ch][st:en]))
                except Exception:
                    pass
            cat_seq = "".join(st_list).upper()
            if uinfo["strand"] == "-":
                cat_seq = cat_seq.translate(str.maketrans("ACGTN", "TGCAN"))[::-1]
            utr_seqs[gid] = cat_seq
    except Exception as e:
        print(f"[-] Notice loading FASTA for UTR extraction: {e}")

if "is_all_upregulated" in df_counts.columns:
    upreg_mask = (df_counts["is_all_upregulated"] == True) | (df_counts["category"].isin(["all_upregulated", "strictly_zygotic_minor_wave"]))
else:
    upreg_mask = df_counts["category"].isin(["all_upregulated", "strictly_zygotic_minor_wave"])
upreg_gids = set(df_counts.loc[upreg_mask, "Geneid"]) & set(utr_seqs.keys())
ctrl_gids = set(df_counts.loc[df_counts["category"] == "control_non_activated", "Geneid"]) & set(utr_seqs.keys())

tests = {
    "Canonical CPE (TTTTAT/TTTTAAT)": ["TTTTAT", "TTTTAAT"],
    "eCPE / U-rich (TTTTTT/TTTTACT)": ["TTTTTT", "TTTTACT", "TTTTAAG"],
    "PAS Hexamer (AATAAA/ATTAAA)":    ["AATAAA", "ATTAAA"]
}

for t_name, motif_list in tests.items():
    up_pos = sum(1 for gid in upreg_gids if gid in utr_seqs and any(m in utr_seqs[gid] for m in motif_list))
    up_neg = len(upreg_gids) - up_pos
    ctrl_pos = sum(1 for gid in ctrl_gids if gid in utr_seqs and any(m in utr_seqs[gid] for m in motif_list))
    ctrl_neg = len(ctrl_gids) - ctrl_pos
    cont_tab = [[up_pos, max(0, up_neg)], [ctrl_pos, max(0, ctrl_neg)]]
    odds, p_val = fisher_exact(cont_tab, alternative="greater")
    print(f"    {t_name:32s}: Upreg {up_pos}/{len(upreg_gids)} ({100*up_pos/max(len(upreg_gids),1):.1f}%) vs Ctrl {ctrl_pos}/{len(ctrl_gids)} ({100*ctrl_pos/max(len(ctrl_gids),1):.1f}%) | OR = {odds:.2f}, p = {p_val:.2e}")

print("=" * 80)
print(f"[*] Inspecting +500 bp Downstream Intergenic Read Extension for TARGET_GENES:")
with pysam.AlignmentFile(RNASEQ_EARLY_BAMS[0], "rb") as b_2c, pysam.AlignmentFile(RNASEQ_PREMZT_BAMS[0], "rb") as b_1k:
    for tg in TARGET_GENES:
        tg_rows = df_counts[df_counts["gene_name"].str.lower() == tg.lower()]
        if tg_rows.empty:
            continue
        r = tg_rows.iloc[0]
        chrom = r["chrom"]
        strand = r["strand"]
        tes = int(r["end"]) if strand == "+" else int(r["start"])
        down_st = tes if strand == "+" else max(0, tes - 500)
        down_en = (tes + 500) if strand == "+" else tes
        try:
            cov2 = np.array(b_2c.count_coverage(chrom, down_st, down_en, quality_threshold=10)).sum()
            cov1 = np.array(b_1k.count_coverage(chrom, down_st, down_en, quality_threshold=10)).sum()
        except Exception:
            cov2, cov1 = 0, 0
        down_seq = ""
        if Path(TARGET_GENOME_FA).exists():
            try:
                fa = Fasta(TARGET_GENOME_FA, as_raw=True)
                down_seq = str(fa[chrom][down_st:down_en]).upper()
                if strand == "-":
                    down_seq = down_seq.translate(str.maketrans("ACGTN", "TGCAN"))[::-1]
            except Exception:
                pass
        pas_found = [m for m in ["AATAAA", "ATTAAA"] if m in down_seq]
        print(f"    {tg} ({chrom}:{down_st}-{down_en}, strand {strand}): 2-cell cov = {cov2:,}, 1000-cell cov = {cov1:,} | PAS: {pas_found or 'None'}")


---
## Block 1b: Zebrafish SLAM-seq Nascent RNA Ortholog Overlay

To definitively distinguish **genuine pre-MZT zygotic transcriptional activation** from **cytoplasmic polyadenylation of pre-existing maternal mRNAs**, we project Zebrafish 4sU SLAM-seq nascent transcription data onto the target species transcriptome:
- `preMZT_strictly_zygotic`: absent total RNA at $0.75\text{ hpf}$ (`CPM < 1.0`) with nascent T$\rightarrow$C reads appearing at $2.5\text{ hpf}$ (pre-MZT minor wave).
- `preMZT_maternal_zygotic`: maternally present at $0.75\text{ hpf}$ (`CPM >= 5.0`) **and** nascent T$\rightarrow$C reads active at $2.5\text{ hpf}$ (pre-MZT maternal-zygotic activation).
- `major_wave_only`: absent or negligible nascent T$\rightarrow$C reads at $2.5\text{ hpf}$, turning on strongly at $3.0–4.5\text{ hpf}$ (canonical major wave).
- `maternal_non_transcribed`: maternal mRNA present at $0.75\text{ hpf}$ (`CPM >= 5.0`) with **zero nascent T$\rightarrow$C transcription** through $4.5\text{ hpf}$ (maternal non-transcribed baseline).

We generate a **2-panel diagnostic figure** (MA plot & 4-class violin distributions with Mann-Whitney U and Fisher exact tests) and export **`slam_confirmed_minor_wave`** as an ultra-clean positive validation set for ML benchmarks.

In [ ]:
# Launch / Verify Zebrafish SLAM-seq Preprocessing Task
slam_cmd = [
    "python", "-u", "scripts/prep_slamseq.py",
    "--work-dir", WORK_DIR,
    "--genome-fa", ZEBRAFISH_GENOME_FA,
    "--gtf", ZEBRAFISH_GTF,
    "--task-name", "prep_slamseq_zf",
    "--threads", str(THREADS),
    "--min-baseline-fc", str(SLAM_MIN_BASELINE_FC),
    "--min-kinetics-retention", str(SLAM_MIN_KINETICS_RETENTION)
]
if SLAM_EXCLUDE_MITO:
    slam_cmd.append("--exclude-mito")
else:
    slam_cmd.append("--no-exclude-mito")
if SLAM_MASK_BASELINE_SNPS:
    slam_cmd.append("--mask-baseline-snps")
else:
    slam_cmd.append("--no-mask-baseline-snps")

# Trigger execution if empirical minor-wave table is missing, while cleanly skipping completed heavy steps
_zf_minor_tsv = Path(WORK_DIR) / "slamseq" / "zebrafish_slamseq_minor_wave_genes.tsv"
_force_slam = bool(globals().get("FORCE_RERUN", False) or (not _zf_minor_tsv.exists()))

tm.submit("prep_slamseq_zf", slam_cmd, depends_on="download_resources", force_rerun=_force_slam)
display(tm.status())


In [ ]:
# ==============================================================================
# Species- and Project-Agnostic Inline SLAM-seq QC Diagnostic & Filtering Audit
# ==============================================================================
import re
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve results directory and SLAM-seq gene classification table
_res_dir = globals().get("res_dir", Path(WORK_DIR) / "results" / globals().get("TARGET_LOCUS_NAME", ""))
_res_dir.mkdir(parents=True, exist_ok=True)
_zf_slam_path = Path(WORK_DIR) / "slamseq" / "zf_slamseq_gene_classes.tsv"

if not _zf_slam_path.exists():
    print(f"[PENDING] SLAM-seq gene classification table not yet available: {_zf_slam_path}")
    print("          Run 'prep_slamseq_zf' to completion before evaluating nascent QC diagnostics.")
else:
    df_slam = pd.read_csv(_zf_slam_path, sep="	")
    print(f"[+] Loaded SLAM-seq classification table: {_zf_slam_path.name} ({len(df_slam):,} genes)")

    # 1. Agnostic mitochondrial contig and symbol detection
    mito_chroms = {"mt", "chrm", "m", "chrmt", "mitochondrion"}
    is_mito_c = df_slam["chrom"].astype(str).str.lower().isin(mito_chroms)
    is_mito_sym = df_slam["gene_name"].astype(str).str.contains(r"^(?:mt[-_.]|nd[1-6]|cox[1-3]|cytb|atp[68])", case=False, regex=True)
    is_mito = is_mito_c | is_mito_sym
    df_slam["_is_mito"] = is_mito

    # 2. Bleed-through, kinetics, and baseline fold-change flags
    bleed_thresh = float(globals().get("SLAM_MAX_BLEED_RATIO", 0.005))
    min_retention = float(globals().get("SLAM_MIN_KINETICS_RETENTION", 0.5))
    min_base_fc = float(globals().get("SLAM_MIN_BASELINE_FC", 3.0))

    has_nas_075 = "nascent_cpm_0_75hpf" in df_slam.columns
    has_nas_untreated = "nascent_cpm_untreated" in df_slam.columns

    is_bleedthrough = (df_slam["total_cpm_0_75hpf"] >= 5.0) & ((df_slam["nascent_cpm_2_5hpf"] / np.maximum(df_slam["total_cpm_0_75hpf"], 1e-4)) < bleed_thresh)
    df_slam["_is_bleedthrough"] = is_bleedthrough

    max_later = np.maximum(df_slam["nascent_cpm_3_0_3_5hpf"], df_slam["nascent_cpm_4_5hpf"])
    is_collapsing = (df_slam["nascent_cpm_2_5hpf"] > 0) & (max_later < (min_retention * df_slam["nascent_cpm_2_5hpf"]))
    df_slam["_is_collapsing"] = is_collapsing

    if has_nas_075:
        base_ref = df_slam["nascent_cpm_0_75hpf"]
        if has_nas_untreated:
            base_ref = np.maximum(base_ref, df_slam["nascent_cpm_untreated"])
        base_ref = np.maximum(base_ref, 0.05)
        is_low_fc = (df_slam["nascent_cpm_2_5hpf"] > 0) & (df_slam["nascent_cpm_2_5hpf"] < (min_base_fc * base_ref))
    else:
        is_low_fc = pd.Series(False, index=df_slam.index)
    df_slam["_is_low_fc"] = is_low_fc

    is_clean = (~is_mito) & (~is_bleedthrough) & (~is_collapsing) & (~is_low_fc)
    df_slam["_is_clean"] = is_clean

    # --------------------------------------------------------------------------
    # Table 1: Per-Class QC Summary
    # --------------------------------------------------------------------------
    summary_rows = []
    classes_order = ["preMZT_strictly_zygotic", "preMZT_maternal_zygotic", "major_wave_only", "maternal_non_transcribed", "silent_or_low"]
    existing_classes = [c for c in classes_order if c in df_slam["slam_class"].unique()]
    other_classes = [c for c in df_slam["slam_class"].unique() if c not in classes_order]

    for sc in existing_classes + other_classes:
        sub = df_slam[df_slam["slam_class"] == sc]
        n_tot = len(sub)
        n_cln = int(sub["_is_clean"].sum())
        pct_cln = f"{100 * n_cln / max(n_tot, 1):.1f}%"
        n_m = int(sub["_is_mito"].sum())
        n_b = int(sub["_is_bleedthrough"].sum())
        n_fc = int(sub["_is_low_fc"].sum()) if has_nas_075 else 0
        n_col = int(sub["_is_collapsing"].sum())
        summary_rows.append({
            "slam_class": sc,
            "n_total": n_tot,
            "n_clean_nuclear": n_cln,
            "pct_clean": pct_cln,
            "n_mito": n_m,
            "n_maternal_bleedthrough": n_b,
            "n_low_baseline_fc": n_fc,
            "n_collapsing_kinetics": n_col
        })

    df_qc_summary = pd.DataFrame(summary_rows)
    qc_tsv_out = _res_dir / "nascent_rnaseq_qc_summary.tsv"
    df_qc_summary.to_csv(qc_tsv_out, sep="	", index=False)
    print("=" * 80)
    print("[1] SLAM-seq Per-Class QC Summary:")
    print("=" * 80)
    try:
        display(df_qc_summary)
    except NameError:
        print(df_qc_summary.to_string(index=False))
    print(f"[+] Saved QC summary table to {qc_tsv_out}")

    # --------------------------------------------------------------------------
    # Table 2: Top 15 Early-Wave Genes
    # --------------------------------------------------------------------------
    early_mask = df_slam["slam_class"].isin(["preMZT_strictly_zygotic", "preMZT_maternal_zygotic"])
    df_early = df_slam[early_mask].copy()
    df_early["nascent_to_maternal_ratio"] = df_early["nascent_cpm_2_5hpf"] / (df_early["total_cpm_0_75hpf"] + 0.01)

    top15_cols = ["gene_name", "chrom", "slam_class", "total_cpm_0_75hpf"]
    if has_nas_075:
        top15_cols.append("nascent_cpm_0_75hpf")
    top15_cols.extend(["nascent_cpm_2_5hpf", "nascent_cpm_3_0_3_5hpf", "nascent_cpm_4_5hpf", "nascent_to_maternal_ratio"])

    top15 = df_early.sort_values(by="nascent_cpm_2_5hpf", ascending=False).head(15)[top15_cols]
    print("[2] Top 15 Early-Wave Genes (Strictly Zygotic + Maternal-Zygotic):")
    print("=" * 80)
    if not top15.empty:
        try:
            display(top15)
        except NameError:
            print(top15.to_string(index=False))
    else:
        print("    No early-wave genes detected.")

    # --------------------------------------------------------------------------
    # Table 3: Dynamic Marker & Target Gene Audit
    # --------------------------------------------------------------------------
    try:
        from scripts.infer_and_attribute import normalize_teleost_tf_name
    except ImportError:
        def normalize_teleost_tf_name(name: str) -> str:
            raw = str(name).strip().upper()
            return re.sub(r"[abAB]$", "", raw) if len(raw) > 3 else raw

    query_genes = list(globals().get("TARGET_GENES", [])) + list(globals().get("CANONICAL_ZGA_MARKERS", []))
    norm_query_map = {g: normalize_teleost_tf_name(g).lower() for g in query_genes}

    audit_matches = []
    slam_gene_names = df_slam["gene_name"].astype(str).tolist()
    slam_norm_names = [normalize_teleost_tf_name(g).lower() for g in slam_gene_names]

    audit_cols = ["gene_name", "chrom", "slam_class", "total_cpm_0_75hpf"]
    if has_nas_075:
        audit_cols.append("nascent_cpm_0_75hpf")
    audit_cols.extend(["nascent_cpm_2_5hpf", "nascent_cpm_3_0_3_5hpf", "nascent_cpm_4_5hpf"])

    seen_idx = set()
    for qg in query_genes:
        q_lower = qg.lower()
        q_norm = norm_query_map[qg]
        for idx, (g_name, g_norm) in enumerate(zip(slam_gene_names, slam_norm_names)):
            if idx not in seen_idx and (g_name.lower() == q_lower or g_norm == q_norm):
                seen_idx.add(idx)
                row_dict = {c: df_slam.iloc[idx][c] for c in audit_cols}
                row_dict["matched_query"] = qg
                audit_matches.append(row_dict)

    df_audit = pd.DataFrame(audit_matches)
    print("[3] Dynamic Marker & Target Gene Audit (TARGET_GENES + CANONICAL_ZGA_MARKERS):")
    print("=" * 80)
    if not df_audit.empty:
        disp_cols = ["matched_query"] + [c for c in audit_cols if c in df_audit.columns]
        try:
            display(df_audit[disp_cols])
        except NameError:
            print(df_audit[disp_cols].to_string(index=False))
    else:
        print(f"    No direct symbol matches found in SLAM-seq table for {query_genes}.")

    # --------------------------------------------------------------------------
    # Actionable Recommendations Block
    # --------------------------------------------------------------------------
    print("[*] SLAM-seq Filtering Diagnostic Recommendations:")
    print("=" * 80)
    recommendations = []

    early_sub = df_slam[early_mask]
    early_mito_cnt = int(early_sub["_is_mito"].sum()) if not early_sub.empty else 0
    early_col_cnt = int(early_sub["_is_collapsing"].sum()) if not early_sub.empty else 0
    early_bleed_cnt = int(early_sub["_is_bleedthrough"].sum()) if not early_sub.empty else 0
    early_tot = len(early_sub)
    col_pct = (100 * early_col_cnt / early_tot) if early_tot > 0 else 0.0

    if early_mito_cnt > 0:
        recommendations.append(
            f"[ACTION REQUIRED] Detected {early_mito_cnt} mitochondrial gene(s) in early-wave classes! "
            f"Set SLAM_EXCLUDE_MITO = True in Block 0 and re-run prep_slamseq_zf to purge mitochondrial leakage."
        )
    else:
        print("  [PASS] Mitochondrial exclusion: 0 mito contig genes detected in early-wave classes.")

    if col_pct > 20.0 or early_bleed_cnt > 0:
        recommendations.append(
            f"[ACTION REQUIRED] High artifact burden in early-wave classes ({early_bleed_cnt} bleed-through genes, "
            f"{early_col_cnt}/{early_tot} [{col_pct:.1f}%] collapsing kinetics). "
            f"Set SLAM_MASK_BASELINE_SNPS = True and SLAM_MIN_BASELINE_FC = {max(min_base_fc, 3.0):.1f} in Block 0 "
            f"to mask injection-clutch germline SNPs and eliminate false-positive spikes."
        )
    else:
        print(f"  [PASS] Clutch SNP & kinetics filter: Collapsing kinetics at {col_pct:.1f}% (<= 20%), bleed-through: {early_bleed_cnt}.")

    if not df_audit.empty:
        canonical_sub = df_audit[df_audit["matched_query"].str.lower().isin([m.lower() for m in globals().get("CANONICAL_ZGA_MARKERS", [])])]
        if not canonical_sub.empty:
            onset_at_35 = canonical_sub["nascent_cpm_3_0_3_5hpf"] > canonical_sub["nascent_cpm_2_5hpf"]
            if onset_at_35.all():
                print(f"  [INFO] Canonical ZGA markers ({list(canonical_sub['matched_query'].unique())}) display nascent transcription onset primarily at 3.0-3.5 hpf rather than 2.5 hpf (confirming expected minor-to-major wave developmental trajectory).")
            elif onset_at_35.any():
                print(f"  [INFO] Canonical ZGA markers show split activation between 2.5 hpf and 3.0-3.5 hpf.")

    if recommendations:
        for rec in recommendations:
            print(f"  {rec}")
    else:
        print("  [+] All SLAM-seq QC criteria satisfied. Classification is clean and ready for Figure 1b overlay.")
    print("=" * 80)


In [ ]:
import matplotlib.pyplot as plt

# Overlay Zebrafish SLAM-seq Nascent Transcription Timing on Target Species
zf_slam_path = Path(WORK_DIR) / "slamseq" / "zf_slamseq_gene_classes.tsv"
gene_sets_tsv = res_dir / f"{TARGET_LOCUS_NAME}_gene_sets.tsv"
df_counts = pd.read_csv(gene_sets_tsv, sep="\t")

slam_info = tm.get_task_info("prep_slamseq_zf")
is_slam_running = slam_info.get("status") == "RUNNING"

if is_slam_running or not zf_slam_path.exists():
    if is_slam_running:
        print("[*] Task 'prep_slamseq_zf' is currently RUNNING in background.")
        print(f"    Current progress: {slam_info.get('progress', 'In progress...')}")
        print("    Zebrafish SLAM-seq overlay and Figure 1b will be rendered once the task completes.")
    else:
        print(f"[*] Zebrafish SLAM-seq table not yet generated: {zf_slam_path}.")
    print("    [!] Zebrafish SLAM-seq nascent ortholog overlay is PENDING (prep_slamseq_zf).")
    print("        Setting is_slam_confirmed_minor_wave = False until SLAM-seq finishes and ortholog overlay is evaluated.")
    df_counts["is_slam_confirmed_minor_wave"] = False
    df_counts.to_csv(gene_sets_tsv, sep="\t", index=False)
    rnaseq_tsv = Path(WORK_DIR) / "rnaseq" / "target_species_gene_sets.tsv"
    rnaseq_tsv.parent.mkdir(parents=True, exist_ok=True)
    df_counts.to_csv(rnaseq_tsv, sep="\t", index=False)
else:
    df_zf_slam = pd.read_csv(zf_slam_path, sep="\t")
    if not Path(ortholog_tsv).exists():
        print(f"[!] Ortholog mapping file {ortholog_tsv} not found yet.")
        print("    [!] Setting is_slam_confirmed_minor_wave = False until ortholog overlay is evaluated.")
        df_counts["is_slam_confirmed_minor_wave"] = False
        df_counts.to_csv(gene_sets_tsv, sep="\t", index=False)
        rnaseq_tsv = Path(WORK_DIR) / "rnaseq" / "target_species_gene_sets.tsv"
        rnaseq_tsv.parent.mkdir(parents=True, exist_ok=True)
        df_counts.to_csv(rnaseq_tsv, sep="\t", index=False)
    else:
        df_ortho = pd.read_csv(ortholog_tsv, sep="\t")
        df_ortho_clean = df_ortho.drop_duplicates(subset=["target_gene_id"]).copy()
        ortho_slam = df_ortho_clean.merge(df_zf_slam[["gene_id", "slam_class"]], left_on="zf_gene_id", right_on="gene_id", how="left")
        # Clean existing merged columns to prevent duplicate _x/_y columns on repeat executions
        cols_to_drop = [c for c in ["target_gene_id", "slam_class", "is_slam_confirmed_minor_wave"] if c in df_counts.columns]
        if cols_to_drop:
            df_counts = df_counts.drop(columns=cols_to_drop)
        id_col = "Geneid" if "Geneid" in df_counts.columns else "gene_id"
        df_merged = df_counts.merge(ortho_slam[["target_gene_id", "slam_class"]], left_on=id_col, right_on="target_gene_id", how="left")
        df_merged["slam_class"] = df_merged["slam_class"].fillna("unmapped_ortholog")

        # Create boolean column is_slam_confirmed_minor_wave
        is_slam_confirmed = (
            df_merged["category"].isin(["strictly_zygotic_minor_wave", "all_upregulated"]) &
            df_merged["slam_class"].isin(["preMZT_strictly_zygotic", "preMZT_maternal_zygotic"])
        )
        df_merged["is_slam_confirmed_minor_wave"] = is_slam_confirmed
        df_merged.to_csv(gene_sets_tsv, sep="\t", index=False)
        rnaseq_tsv = Path(WORK_DIR) / "rnaseq" / "target_species_gene_sets.tsv"
        rnaseq_tsv.parent.mkdir(parents=True, exist_ok=True)
        df_merged.to_csv(rnaseq_tsv, sep="\t", index=False)
        print(f"[+] Persisted updated gene sets table with SLAM-seq overlay ({is_slam_confirmed.sum()} confirmed genes) to both {gene_sets_tsv} and {rnaseq_tsv}")

        n_mapped = (df_merged["slam_class"] != "unmapped_ortholog").sum()
        print("======================================================================")
        print(f"[*] Zebrafish SLAM-seq Ortholog Mapping Statistics:")
        print(f"    Genes mapped to zebrafish ortholog: {n_mapped} / {len(df_counts)} ({100*n_mapped/len(df_counts):.1f}%)")
        print(df_merged["slam_class"].value_counts().to_string())
        print("======================================================================")

        from scipy.stats import mannwhitneyu, fisher_exact
        mat_lfc = df_merged.loc[df_merged["slam_class"] == "maternal_non_transcribed", "log2fc"].dropna()
        mw_pvals = {}
        for sc in ["preMZT_strictly_zygotic", "preMZT_maternal_zygotic", "major_wave_only"]:
            sc_lfc = df_merged.loc[df_merged["slam_class"] == sc, "log2fc"].dropna()
            if len(sc_lfc) > 0 and len(mat_lfc) > 0:
                stat, pval = mannwhitneyu(sc_lfc, mat_lfc, alternative="greater")
                mw_pvals[sc] = pval
                print(f"    Mann-Whitney U: {sc} vs maternal: p = {pval:.2e}")
            else:
                mw_pvals[sc] = 1.0

        is_up = df_merged["is_all_upregulated"]
        is_pre_sz = df_merged["slam_class"] == "preMZT_strictly_zygotic"
        cont_tbl = [
            [(is_up & is_pre_sz).sum(), (~is_up & is_pre_sz).sum()],
            [(is_up & ~is_pre_sz).sum(), (~is_up & ~is_pre_sz).sum()]
        ]
        fisher_odds, fisher_p = fisher_exact(cont_tbl, alternative="greater")
        print(f"    Fisher Exact Test (preMZT_strictly_zygotic in all_upregulated): OR = {fisher_odds:.2f}, p = {fisher_p:.2e}")

        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        ax = axes[0]
        # Panel 1: MA plot colored by Zebrafish SLAM-seq timing class
        slam_palette = {
            "preMZT_strictly_zygotic": "#d7191c",
            "preMZT_maternal_zygotic": "#fdae61",
            "major_wave_only": "#2ca25f",
            "maternal_non_transcribed": "#2b83ba",
            "unmapped_ortholog": "#bdbdbd",
            "silent_or_low": "#969696"
        }
        mean_cpm_t = 0.5 * (df_merged["cpm_early"] + df_merged["cpm_premzt"])
        for sc, sc_col in slam_palette.items():
            sc_mask = df_merged["slam_class"] == sc
            if sc_mask.any():
                ax.scatter(
                    np.log10(mean_cpm_t[sc_mask] + 0.1),
                    df_merged.loc[sc_mask, "log2fc"],
                    color=sc_col,
                    label=sc,
                    s=22 if sc.startswith("preMZT") else 12,
                    alpha=0.85 if sc.startswith("preMZT") else 0.4
                )
        # Callouts for TARGET_GENES and CANONICAL_ZGA_MARKERS
        for gname in TARGET_GENES + CANONICAL_ZGA_MARKERS:
            sub = df_merged[df_merged["gene_name"].str.lower() == gname.lower()]
            if not sub.empty:
                r = sub.iloc[0]
                mx = np.log10(0.5 * (r["cpm_early"] + r["cpm_premzt"]) + 0.1)
                my = r["log2fc"]
                ax.scatter([mx], [my], color="black", s=70, zorder=6)
                ax.annotate(gname, (mx, my), xytext=(5, 5), textcoords="offset points", fontweight="bold", fontsize=10)

        ax.set_xlabel("log10(Mean Target CPM + 0.1)", fontsize=11)
        ax.set_ylabel("Target log2 Fold Change", fontsize=11)
        ax.set_title("Target RNA-seq MA Plot Colored by ZF SLAM-seq Timing", fontweight="bold")
        ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
        ax.legend(frameon=True, fontsize=8.5, loc="upper left")
        ax.grid(True, linestyle=":", alpha=0.5)

        ax = axes[1]
        classes_to_plot = ["preMZT_strictly_zygotic", "preMZT_maternal_zygotic", "major_wave_only", "maternal_non_transcribed"]
        labels_to_plot = [
            f"preMZT Strictly\nZygotic\n(p={mw_pvals.get('preMZT_strictly_zygotic', 1.0):.1e})",
            f"preMZT Maternal-\nZygotic\n(p={mw_pvals.get('preMZT_maternal_zygotic', 1.0):.1e})",
            f"Major Wave\nOnly\n(p={mw_pvals.get('major_wave_only', 1.0):.1e})",
            "Maternal\nNon-Transcribed\n(Control)"
        ]
        colors_to_plot = ["#d7191c", "#fdae61", "#2ca25f", "#2b83ba"]
        lfc_by_class = [df_merged.loc[df_merged["slam_class"] == sc, "log2fc"].dropna().values for sc in classes_to_plot]
        valid_indices = [i for i, d in enumerate(lfc_by_class) if len(d) > 0]
        if valid_indices:
            filtered_lfc = [lfc_by_class[i] for i in valid_indices]
            parts = ax.violinplot(filtered_lfc, positions=[i + 1 for i in valid_indices], showmeans=True, showmedians=True)
            for i_idx, p_idx in enumerate(valid_indices):
                if i_idx < len(parts['bodies']):
                    parts['bodies'][i_idx].set_facecolor(colors_to_plot[p_idx])
                    parts['bodies'][i_idx].set_edgecolor('black')
            ax.set_xticks([i + 1 for i in valid_indices])
            ax.set_xticklabels([labels_to_plot[i] for i in valid_indices], fontsize=9.5)
            ax.set_ylabel("log2 Fold Change in Target RNA-seq", fontsize=11)
            ax.set_title(f"Target Species log2FC Across SLAM-seq Timing Classes\n(OR={fisher_odds:.1f}, p={fisher_p:.1e})", fontweight="bold")
            ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
            ax.grid(True, linestyle=":", alpha=0.5)

        plt.tight_layout()
        fig1b_path = res_dir / "fig1b_slamseq_ortholog_overlay.png"
        plt.savefig(fig1b_path, dpi=300)
        plt.show()
        print(f"[+] Figure 1b exported to {fig1b_path}")


---
## Block 2: Background ATAC-seq Preprocessing & ChromBPNet Base Training (Zebrafish & Medaka)

We train debiased Stage 1 ChromBPNet accessibility models on reference blastula chromatin:
1. **Peak Calling & Continuous Targets:** Process early vs. late blastula ATAC-seq for Zebrafish (*GRCz11*) and Medaka (*ASM223467v1*), calling native local-lambda Poisson late peaks and computing continuous targets ($y_{\text{abs}}, y_{\text{rel}}$) across summits.
2. **Leak-Free Chromosome Splits:** The chromosome orthologous to `TARGET_LOCUS_NAME` is strictly held out in the `test` split.
3. **Stage 1 ChromBPNet Training:** Runs GC-matched non-peaks, Tn5 bias modeling, and official ChromBPNet PyTorch training via `scripts/train_chrombpnet.py` managed by `TaskManager`.

In [ ]:
# Launch Background ATAC Preprocessing and ChromBPNet Training for Both Zebrafish & Medaka
atac_manifest = f"{WORK_DIR}/downloads/atac_fastq_manifest.tsv"

# 1. Submit Zebrafish ATAC Preprocessing & ChromBPNet Pipeline
zf_atac_cmd = [
    "python", "-u", "scripts/prep_atac.py",
    "--species", "Danio_rerio",
    "--work-dir", WORK_DIR,
    "--genome-fa", ZEBRAFISH_GENOME_FA,
    "--gtf", ZEBRAFISH_GTF,
    "--fastq-manifest", atac_manifest,
    "--target-genes", *TARGET_GENES,
    "--cv-folds", str(CV_FOLDS),
    "--task-name", "prep_atac_danio_rerio",
    "--threads", str(THREADS)
]
if EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2:
    zf_atac_cmd.append("--exclude-housekeeping-promoters")

tm.submit("prep_atac_danio_rerio", zf_atac_cmd, depends_on="download_resources", force_rerun=FORCE_RERUN)

zf_chrom_sizes = f"{WORK_DIR}/atac/Danio_rerio/chrom.sizes"
zf_peaks_bed = f"{WORK_DIR}/atac/Danio_rerio/peaks/Danio_rerio_late_peaks.primary.narrowPeak"
zf_late_bam = f"{WORK_DIR}/atac/Danio_rerio/Danio_rerio_late_merged.bam"

FOLDS_TO_TRAIN = [0] if COMPUTE_PROFILE == "laptop_16gb" else list(range(CV_FOLDS))

for fold in FOLDS_TO_TRAIN:
    zf_chr_folds = f"{WORK_DIR}/atac/Danio_rerio/chr_folds.json"
    zf_chrombpnet_cmd = [
        "python", "-u", "scripts/train_chrombpnet.py",
        "--species", "Danio_rerio",
        "--work-dir", WORK_DIR,
        "--genome-fa", ZEBRAFISH_GENOME_FA,
        "--chrom-sizes", zf_chrom_sizes,
        "--peaks-bed", zf_peaks_bed,
        "--late-bam", zf_late_bam,
        "--chr-fold-json", zf_chr_folds,
        "--fold", str(fold),
        "--task-name", f"train_chrombpnet_danio_rerio_fold_{fold}",
        "--epochs", str(CHROMBPNET_EPOCHS),
        "--threads", str(THREADS)
    ]
    if USE_PRETRAINED_BIAS:
        bias_h5 = BIAS_MODEL_PATH or f"{WORK_DIR}/downloads/bias_models/atac_bias.h5"
        zf_chrombpnet_cmd.extend(["--use-pretrained-bias", "--bias-model-path", bias_h5])

    tm.submit(f"train_chrombpnet_danio_rerio_fold_{fold}", zf_chrombpnet_cmd, depends_on="prep_atac_danio_rerio", force_rerun=FORCE_RERUN, requires_gpu=True)

# 2. Submit Medaka ATAC Preprocessing & ChromBPNet Pipeline (when enabled)
if ENABLE_MEDAKA:
    medaka_atac_cmd = [
        "python", "-u", "scripts/prep_atac.py",
        "--species", "Oryzias_latipes",
        "--work-dir", WORK_DIR,
        "--genome-fa", MEDAKA_GENOME_FA,
        "--gtf", MEDAKA_GTF,
        "--fastq-manifest", atac_manifest,
        "--target-genes", *TARGET_GENES,
        "--cv-folds", str(CV_FOLDS),
        "--task-name", "prep_atac_medaka",
        "--threads", str(THREADS)
    ]
    if EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2:
        medaka_atac_cmd.append("--exclude-housekeeping-promoters")

    tm.submit("prep_atac_medaka", medaka_atac_cmd, depends_on="download_resources", force_rerun=FORCE_RERUN)

    med_chrom_sizes = f"{WORK_DIR}/atac/Oryzias_latipes/chrom.sizes"
    med_peaks_bed = f"{WORK_DIR}/atac/Oryzias_latipes/peaks/Oryzias_latipes_late_peaks.primary.narrowPeak"
    med_late_bam = f"{WORK_DIR}/atac/Oryzias_latipes/Oryzias_latipes_late_merged.bam"

    for fold in FOLDS_TO_TRAIN:
        med_chr_folds = f"{WORK_DIR}/atac/Oryzias_latipes/chr_folds.json"
        medaka_chrombpnet_cmd = [
            "python", "-u", "scripts/train_chrombpnet.py",
            "--species", "Oryzias_latipes",
            "--work-dir", WORK_DIR,
            "--genome-fa", MEDAKA_GENOME_FA,
            "--chrom-sizes", med_chrom_sizes,
            "--peaks-bed", med_peaks_bed,
            "--late-bam", med_late_bam,
            "--chr-fold-json", med_chr_folds,
            "--fold", str(fold),
            "--task-name", f"train_chrombpnet_medaka_fold_{fold}",
            "--epochs", str(CHROMBPNET_EPOCHS),
            "--threads", str(THREADS)
        ]
        if USE_PRETRAINED_BIAS:
            bias_h5 = BIAS_MODEL_PATH or f"{WORK_DIR}/downloads/bias_models/atac_bias.h5"
            medaka_chrombpnet_cmd.extend(["--use-pretrained-bias", "--bias-model-path", bias_h5])

        tm.submit(f"train_chrombpnet_medaka_fold_{fold}", medaka_chrombpnet_cmd, depends_on="prep_atac_medaka", force_rerun=FORCE_RERUN, requires_gpu=True)

display(tm.status())


In [ ]:
# ==============================================================================
# Cross-Sample Alignment & ChromBPNet Base Model Quality Audit (Block 2 Summary)
# ==============================================================================
# Audits ATAC-seq preprocessing, trimming, Bowtie2 alignment, peak calling,
# chromosome split integrity, and ChromBPNet Stage 1 base models across all
# samples, stages, and species dynamically defined in Block 0 and manifests.
import os
import sys
import json
import re
import shutil
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import pysam

# Ensure active conda environment bin is available for bioinformatics CLIs
bin_dir = str(Path(sys.prefix) / "bin")
if bin_dir not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"{bin_dir}:{os.environ.get('PATH', '')}"

from scripts.train_chrombpnet import refresh_training_meta

work_dir = Path(WORK_DIR).resolve()
out_results_dir = Path(res_dir).resolve()
out_results_dir.mkdir(parents=True, exist_ok=True)

manifest_path = work_dir / "downloads" / "atac_fastq_manifest.tsv"
df_manifest = pd.read_csv(manifest_path, sep="\t") if manifest_path.exists() else pd.DataFrame()

# ------------------------------------------------------------------------------
# Part A: Per-Sample ATAC-seq Trimming & Alignment QC Table
# ------------------------------------------------------------------------------
log_files = list((work_dir / "tasks").glob("prep_atac_*/*.log"))
log_corpus = {lf.name: lf.read_text(errors="replace") for lf in log_files}

sample_qc_rows = []

if not df_manifest.empty:
    for _, row in df_manifest.iterrows():
        sp = str(row["species"]).strip()
        st = str(row["stage"]).strip()
        run_id = str(row["run_id"]).strip()
        layout = str(row.get("layout", "UNKNOWN")).strip()

        # Discover fastp report beside FASTQ
        r1_path_str = str(row.get("fastq_r1", "")).strip()
        p_r1 = Path(r1_path_str) if r1_path_str else None
        fastp_candidates = [
            work_dir / "atac" / sp / "fastqs" / f"{run_id}_fastp.json",
            work_dir / "atac" / sp / "fastqs" / f"{run_id}_1_fastp.json"
        ]
        if p_r1 and p_r1.parent.exists():
            fastp_candidates.extend([
                p_r1.parent / f"{run_id}_fastp.json",
                p_r1.parent / f"{run_id}_1_fastp.json",
                p_r1.parent / f"{p_r1.stem}_fastp.json"
            ])

        fastp_json = None
        for cand in fastp_candidates:
            if cand.exists() and cand.stat().st_size > 0:
                fastp_json = cand
                break

        raw_reads, filtered_reads = None, None
        pass_filter_pct, q30_after_pct = None, None
        adapter_trimmed_pct, duplication_pct, insert_size_peak = None, None, None

        if fastp_json:
            try:
                with open(fastp_json, "r") as fp:
                    data_fp = json.load(fp)
                summary = data_fp.get("summary", {})
                bf = summary.get("before_filtering", {})
                af = summary.get("after_filtering", {})
                fr = data_fp.get("filtering_result", {})
                dup = data_fp.get("duplication", {})
                ins = data_fp.get("insert_size", {})

                raw_reads = bf.get("total_reads")
                filtered_reads = af.get("total_reads")
                if raw_reads and raw_reads > 0 and filtered_reads is not None:
                    pass_filter_pct = (filtered_reads / raw_reads) * 100.0
                if af and "q30_rate" in af:
                    q30_after_pct = af["q30_rate"] * 100.0
                ad_trimmed_cnt = data_fp.get("adapter_cutting", {}).get("adapter_trimmed_reads") or fr.get("reads_with_adapter_trimmed", 0)
                if raw_reads and raw_reads > 0 and ad_trimmed_cnt is not None:
                    adapter_trimmed_pct = (ad_trimmed_cnt / raw_reads) * 100.0
                if dup and "rate" in dup:
                    duplication_pct = dup["rate"] * 100.0
                if ins and isinstance(ins, dict):
                    insert_size_peak = ins.get("peak")
            except Exception:
                pass

        # Bowtie2 log scraping
        bt2_overall_pct, bt2_uniq_pct = None, None
        for _, ltext in log_corpus.items():
            if run_id in ltext:
                m_bt2 = re.search(r"bowtie2[^\r\n]*?" + re.escape(run_id) + r"[^\r\n]*?\n([\s\S]*?overall alignment rate)", ltext)
                if m_bt2:
                    block = m_bt2.group(1)
                    m_over = re.search(r"([0-9\.]+)%\s+overall alignment rate", block)
                    m_uniq = re.search(r"\(\s*([0-9\.]+)%\s*\)\s*aligned\s+(?:concordantly\s+)?exactly 1 time", block)
                    if m_over:
                        bt2_overall_pct = float(m_over.group(1))
                    if m_uniq:
                        bt2_uniq_pct = float(m_uniq.group(1))
                    break

        # Final BAM metrics
        bam_path = work_dir / "atac" / sp / "bams" / f"{run_id}.sorted.bam"
        final_mapped_reads, usable_pct = None, None
        status = "PENDING"

        if bam_path.exists():
            try:
                res_idx = subprocess.run(["samtools", "idxstats", str(bam_path)], capture_output=True, text=True, check=True)
                mapped_cnt = 0
                for line in res_idx.stdout.strip().split("\n"):
                    if not line:
                        continue
                    parts = line.split("\t")
                    if len(parts) >= 3 and not (parts[0].startswith("chrM") or parts[0] == "MT"):
                        mapped_cnt += int(parts[2])
                final_mapped_reads = mapped_cnt
                if filtered_reads and filtered_reads > 0:
                    usable_pct = (final_mapped_reads / filtered_reads) * 100.0
                status = "COMPLETED"
            except Exception:
                status = "FAILED"
        elif (work_dir / "tasks" / f"prep_atac_{sp}" / "running.txt").exists():
            status = "RUNNING"

        sample_qc_rows.append({
            "species": sp,
            "stage": st,
            "run_id": run_id,
            "layout": layout,
            "status": status,
            "raw_reads": raw_reads,
            "filtered_reads": filtered_reads,
            "pass_filter_pct": round(pass_filter_pct, 2) if pass_filter_pct is not None else None,
            "q30_after_pct": round(q30_after_pct, 2) if q30_after_pct is not None else None,
            "adapter_trimmed_pct": round(adapter_trimmed_pct, 2) if adapter_trimmed_pct is not None else None,
            "duplication_pct": round(duplication_pct, 2) if duplication_pct is not None else None,
            "insert_size_peak": insert_size_peak,
            "bt2_overall_align_pct": bt2_overall_pct,
            "bt2_unique_align_pct": bt2_uniq_pct,
            "final_mapped_reads": final_mapped_reads,
            "usable_fraction_pct": round(usable_pct, 2) if usable_pct is not None else None
        })

df_sample_qc = pd.DataFrame(sample_qc_rows)

# ------------------------------------------------------------------------------
# Part B: Per-Species Stage Merge, Peak Calling & Split Summary Table
# ------------------------------------------------------------------------------
atac_root = work_dir / "atac"
species_qc_rows = []
species_candidates = [d.name for d in atac_root.iterdir() if d.is_dir()] if atac_root.exists() else []

for sp in sorted(species_candidates):
    sp_dir = atac_root / sp
    early_bam = sp_dir / f"{sp}_early_merged.bam"
    late_bam = sp_dir / f"{sp}_late_merged.bam"

    early_merged_reads = 0
    if early_bam.exists():
        try:
            early_merged_reads = sum(int(line.split("\t")[2]) for line in pysam.idxstats(str(early_bam)).strip().split("\n") if line and not (line.split("\t")[0] in ["chrM", "MT"]))
        except Exception:
            pass

    late_merged_reads = 0
    if late_bam.exists():
        try:
            late_merged_reads = sum(int(line.split("\t")[2]) for line in pysam.idxstats(str(late_bam)).strip().split("\n") if line and not (line.split("\t")[0] in ["chrM", "MT"]))
        except Exception:
            pass

    # Peak calling metrics
    peaks_narrow = sp_dir / "peaks" / f"{sp}_late_peaks.narrowPeak"
    if not peaks_narrow.exists():
        peaks_narrow = sp_dir / f"{sp}_late_peaks.narrowPeak"
    primary_peaks_narrow = sp_dir / "peaks" / f"{sp}_late_peaks.primary.narrowPeak"
    if not primary_peaks_narrow.exists():
        primary_peaks_narrow = sp_dir / f"{sp}_late_peaks.primary.narrowPeak"

    total_peaks = 0
    if peaks_narrow.exists():
        try:
            total_peaks = sum(1 for line in open(peaks_narrow) if line.strip() and not line.startswith("#"))
        except Exception:
            pass

    primary_peaks_kept = 0
    peak_widths = []
    if primary_peaks_narrow.exists():
        with open(primary_peaks_narrow, "r") as pf:
            for line in pf:
                if line.strip() and not line.startswith("#"):
                    p_cols = line.strip().split("\t")
                    if len(p_cols) >= 3:
                        primary_peaks_kept += 1
                        peak_widths.append(int(p_cols[2]) - int(p_cols[1]))
    median_peak_width_bp = float(np.median(peak_widths)) if peak_widths else None

    # FRiP, Mito %, and TSSe caching mechanism for fast notebook re-runs
    qc_cache_file = sp_dir / "qc_summary.json"
    late_frip = None
    late_reads_in_peaks = None
    mito_pct = 0.0
    early_tsse = None
    late_tsse = None

    if qc_cache_file.exists():
        try:
            with open(qc_cache_file, "r") as fp:
                cached_qc = json.load(fp)
            late_reads_in_peaks = cached_qc.get("late_reads_in_peaks")
            late_frip = cached_qc.get("late_frip")
            mito_pct = cached_qc.get("mito_read_pct", 0.0)
            early_tsse = cached_qc.get("early_tss_enrichment")
            late_tsse = cached_qc.get("late_tss_enrichment")
        except Exception:
            pass

    if late_frip is None and late_bam.exists() and peaks_narrow.exists() and late_merged_reads > 0:
        try:
            cmd_frip = ["samtools", "view", "-@", "8", "-c", "-L", str(peaks_narrow), str(late_bam)]
            res_frip = subprocess.run(cmd_frip, capture_output=True, text=True, check=True)
            late_reads_in_peaks = int(res_frip.stdout.strip())
            late_frip = late_reads_in_peaks / late_merged_reads
            with open(qc_cache_file, "w") as fp:
                json.dump({
                    "late_reads_in_peaks": late_reads_in_peaks,
                    "late_frip": late_frip,
                    "late_merged_reads": late_merged_reads,
                    "mito_read_pct": mito_pct,
                    "early_tss_enrichment": early_tsse,
                    "late_tss_enrichment": late_tsse
                }, fp, indent=2)
        except Exception:
            pass

    # Repeat downweight summits
    summit_targets_file = sp_dir / f"{sp}_summit_targets.tsv"
    repeat_downweight_summits = None
    if summit_targets_file.exists():
        try:
            df_st = pd.read_csv(summit_targets_file, sep="\t", low_memory=False)
            if "is_repeat_downweight" in df_st.columns:
                repeat_downweight_summits = int((df_st["is_repeat_downweight"] == True).sum())
            elif "repeat_downweighted" in df_st.columns:
                repeat_downweight_summits = int((df_st["repeat_downweighted"] == True).sum())
        except Exception:
            pass

    # Chromosome splits
    chr_fold_file = sp_dir / "chr_folds.json"
    if not chr_fold_file.exists():
        chr_fold_file = sp_dir / "chr_fold.json"
    n_train_chroms, n_valid_chroms, n_test_chroms = None, None, None
    test_chroms_str = ""
    if chr_fold_file.exists():
        try:
            with open(chr_fold_file, "r") as fp:
                cf_data = json.load(fp)
            fold_split = cf_data.get("fold_0", cf_data.get("0", cf_data))
            t_c = fold_split.get("train", [])
            v_c = fold_split.get("valid", fold_split.get("val", []))
            te_c = fold_split.get("test", [])
            n_train_chroms = len(t_c)
            n_valid_chroms = len(v_c)
            n_test_chroms = len(te_c)
            test_chroms_str = ", ".join(str(c) for c in te_c[:5]) + (f" (+{len(te_c)-5} more)" if len(te_c) > 5 else "")
        except Exception:
            pass

    # Replicate depth balance ratio
    early_rep_reads, late_rep_reads = [], []
    if not df_manifest.empty:
        df_sp_man = df_manifest[df_manifest["species"] == sp]
        for _, srow in df_sp_man.iterrows():
            s_id = str(srow["run_id"]).strip()
            s_bam = sp_dir / "bams" / f"{s_id}.sorted.bam"
            if s_bam.exists():
                s_cnt = sum(int(line.split("\t")[2]) for line in pysam.idxstats(str(s_bam)).strip().split("\n") if line)
                if "early" in str(srow["stage"]).lower():
                    early_rep_reads.append(s_cnt)
                elif "late" in str(srow["stage"]).lower():
                    late_rep_reads.append(s_cnt)

    early_balance = (max(early_rep_reads) / min(early_rep_reads)) if len(early_rep_reads) > 1 and min(early_rep_reads) > 0 else 1.0
    late_balance = (max(late_rep_reads) / min(late_rep_reads)) if len(late_rep_reads) > 1 and min(late_rep_reads) > 0 else 1.0
    rep_depth_balance = max(early_balance, late_balance)

    species_qc_rows.append({
        "species": sp,
        "early_merged_reads": early_merged_reads,
        "late_merged_reads": late_merged_reads,
        "late_frip": round(late_frip, 4) if late_frip is not None else None,
        "total_peaks": total_peaks,
        "primary_peaks_kept": primary_peaks_kept,
        "median_peak_width_bp": median_peak_width_bp,
        "repeat_downweight_summits": repeat_downweight_summits,
        "mito_read_pct": round(mito_pct, 2) if mito_pct is not None else 0.0,
        "early_tss_enrichment": round(early_tsse, 2) if early_tsse is not None else None,
        "late_tss_enrichment": round(late_tsse, 2) if late_tsse is not None else None,
        "n_train_chroms": n_train_chroms,
        "n_valid_chroms": n_valid_chroms,
        "n_test_chroms": n_test_chroms,
        "test_chroms": test_chroms_str,
        "rep_depth_balance_ratio": round(rep_depth_balance, 2)
    })

df_species_qc = pd.DataFrame(species_qc_rows)

# ------------------------------------------------------------------------------
# Part C: Cross-Species ChromBPNet Stage 1 Model QC Table
# ------------------------------------------------------------------------------
models_root = work_dir / "models" / "chrombpnet"
chrombpnet_qc_rows = []

if models_root.exists():
    for fold_dir in sorted(models_root.glob("*/fold_*")):
        if not fold_dir.is_dir():
            continue
        sp_name = fold_dir.parent.name
        fold_idx = fold_dir.name.replace("fold_", "")

        meta_file = fold_dir / "training_meta.json"
        meta_dict = {}
        if meta_file.exists():
            try:
                with open(meta_file, "r") as fp:
                    meta_dict = json.load(fp)
            except Exception:
                pass

        # Refresh metrics automatically if provenance source_files is missing
        if not meta_dict.get("qc_metrics", {}).get("source_files"):
            meta_dict = refresh_training_meta(fold_dir)

        qc_dict = meta_dict.get("qc_metrics", {})

        # Checkpoint discovery & size in MB
        ckpt_path = meta_dict.get("checkpoint")
        ckpt_size_mb = None
        if ckpt_path and Path(ckpt_path).exists():
            ckpt_size_mb = round(Path(ckpt_path).stat().st_size / (1024 * 1024), 2)
        else:
            cand_ckpts = list((fold_dir / "models").glob("*nobias*"))
            if cand_ckpts and cand_ckpts[0].exists():
                ckpt_size_mb = round(cand_ckpts[0].stat().st_size / (1024 * 1024), 2)

        model_status = "PENDING"
        if (fold_dir / ".train.done").exists() or ckpt_size_mb is not None:
            model_status = "COMPLETED"
        elif (fold_dir / "bias_run" / ".bias.done").exists() or (fold_dir / "bias_run" / "models" / "bias.pt").exists():
            model_status = "BIAS_ONLY"

        chrombpnet_qc_rows.append({
            "species": sp_name,
            "fold": fold_idx,
            "status": model_status,
            "count_pearson_r": round(qc_dict["count_pearson_r"], 4) if qc_dict.get("count_pearson_r") is not None else None,
            "profile_pearson_r": round(qc_dict["profile_pearson_r"], 4) if qc_dict.get("profile_pearson_r") is not None else None,
            "profile_median_jsd": round(qc_dict["profile_median_jsd"], 4) if qc_dict.get("profile_median_jsd") is not None else None,
            "nobias_count_pearson_r": round(qc_dict["nobias_count_pearson_r"], 4) if qc_dict.get("nobias_count_pearson_r") is not None else None,
            "nobias_profile_jsd": round(qc_dict["nobias_profile_median_jsd"], 4) if qc_dict.get("nobias_profile_median_jsd") is not None else None,
            "max_tn5_footprint_response": round(qc_dict["max_tn5_footprint_response"], 4) if qc_dict.get("max_tn5_footprint_response") is not None else None,
            "bias_counts_pearson_r": round(qc_dict["bias_counts_pearson_r"], 4) if qc_dict.get("bias_counts_pearson_r") is not None else None,
            "bias_profile_jsd": round(qc_dict["bias_profile_jsd"], 4) if qc_dict.get("bias_profile_jsd") is not None else None,
            "bias_nonpeaks_counts_pearson_r": round(qc_dict["bias_nonpeaks_counts_pearson_r"], 4) if qc_dict.get("bias_nonpeaks_counts_pearson_r") is not None else None,
            "checkpoint_mb": ckpt_size_mb
        })

df_cbp_qc = pd.DataFrame(chrombpnet_qc_rows)

# ------------------------------------------------------------------------------
# Part D: Automated QC Warning Engine
# ------------------------------------------------------------------------------
qc_flags = []

# Trimming & Alignment checks
if not df_sample_qc.empty:
    for _, s_row in df_sample_qc.iterrows():
        s_id, s_sp = s_row["run_id"], s_row["species"]
        if s_row["q30_after_pct"] is not None and s_row["q30_after_pct"] < 85.0:
            qc_flags.append(f"[WARNING] Trimming: Sample {s_id} ({s_sp}) Q30 rate = {s_row['q30_after_pct']:.1f}% (< 85%)")
        if s_row["bt2_overall_align_pct"] is not None and s_row["bt2_overall_align_pct"] < 60.0:
            qc_flags.append(f"[WARNING] Alignment: Sample {s_id} ({s_sp}) Bowtie2 overall alignment = {s_row['bt2_overall_align_pct']:.1f}% (< 60%)")
        if s_row["usable_fraction_pct"] is not None and s_row["usable_fraction_pct"] < 20.0:
            qc_flags.append(f"[WARNING] Usability: Sample {s_id} ({s_sp}) usable fraction = {s_row['usable_fraction_pct']:.1f}% (< 20%)")

# Peak & Split checks
if not df_species_qc.empty:
    for _, sp_row in df_species_qc.iterrows():
        s_sp = sp_row["species"]
        if sp_row["rep_depth_balance_ratio"] is not None and sp_row["rep_depth_balance_ratio"] > 5.0:
            qc_flags.append(f"[WARNING] Depth Balance: Species {s_sp} replicate depth imbalance = {sp_row['rep_depth_balance_ratio']:.1f}x (> 5.0x)")
        if sp_row["late_merged_reads"] < 10000000:
            qc_flags.append(f"[WARNING] Depth: Species {s_sp} late blastula merged depth = {sp_row['late_merged_reads']:,} (< 10M reads)")
        if sp_row["early_merged_reads"] < 5000000:
            qc_flags.append(f"[WARNING] Depth: Species {s_sp} early blastula merged depth = {sp_row['early_merged_reads']:,} (< 5M reads)")
        if sp_row["late_frip"] is not None and sp_row["late_frip"] < 0.10:
            qc_flags.append(f"[WARNING] Enrichment: Species {s_sp} late blastula FRiP = {sp_row['late_frip']*100:.2f}% (< 10% expected)")
        if sp_row["primary_peaks_kept"] < 5000:
            qc_flags.append(f"[WARNING] Peaks: Species {s_sp} primary peaks kept = {sp_row['primary_peaks_kept']:,} (< 5,000)")
        if sp_row["n_test_chroms"] == 0:
            qc_flags.append(f"[CRITICAL] Splits: Species {s_sp} test chromosome split is empty!")
        if sp_row.get("mito_read_pct") is not None and sp_row["mito_read_pct"] > 2.0:
            qc_flags.append(f"[CRITICAL] Mitochondrial Contamination: Species {s_sp} mito reads = {sp_row['mito_read_pct']:.2f}% (> 2.0%)")
        if sp_row.get("late_tss_enrichment") is not None:
            if sp_row["late_tss_enrichment"] < 4.0:
                qc_flags.append(f"[CRITICAL] TSS Enrichment: Species {s_sp} late blastula TSSe = {sp_row['late_tss_enrichment']:.2f} (< 4.0, failed enrichment)")
            elif sp_row["late_tss_enrichment"] < 6.0:
                qc_flags.append(f"[WARNING] TSS Enrichment: Species {s_sp} late blastula TSSe = {sp_row['late_tss_enrichment']:.2f} (< 6.0 expected)")

# ChromBPNet checks
if not df_cbp_qc.empty:
    for _, m_row in df_cbp_qc.iterrows():
        m_sp, m_fld = m_row["species"], m_row["fold"]
        c_pr = m_row["count_pearson_r"]
        nb_pr = m_row.get("nobias_count_pearson_r")
        b_pr = m_row["bias_counts_pearson_r"]
        p_jsd = m_row["profile_median_jsd"]
        b_np_pr = m_row["bias_nonpeaks_counts_pearson_r"]
        max_resp = m_row.get("max_tn5_footprint_response")

        # Sanity collision check
        if c_pr is not None and b_pr is not None and abs(c_pr - b_pr) < 1e-9:
            qc_flags.append(f"[CRITICAL] Collision: Model {m_sp} fold {m_fld} main count Pearson r and bias counts Pearson r are identical ({c_pr:.6f})")
        if c_pr is not None:
            if c_pr <= 0.0:
                qc_flags.append(f"[CRITICAL] Model Performance: Model {m_sp} fold {m_fld} main count Pearson r is non-positive ({c_pr:.4f})")
            elif c_pr < 0.50:
                qc_flags.append(f"[WARNING] Model Performance: Model {m_sp} fold {m_fld} main count Pearson r = {c_pr:.4f} (< 0.50)")
        if nb_pr is not None and c_pr is not None and (c_pr - nb_pr) > 0.10:
            qc_flags.append(f"[CRITICAL] Model Degradation: Model {m_sp} fold {m_fld} count Pearson r dropped by > 0.10 after debiasing ({c_pr:.4f} -> {nb_pr:.4f})")
        if p_jsd is not None and p_jsd > 0.65:
            qc_flags.append(f"[WARNING] Model Resolution: Model {m_sp} fold {m_fld} profile median JSD = {p_jsd:.4f} (> 0.65)")
        if b_pr is not None:
            if b_pr > 0.50:
                qc_flags.append(f"[CRITICAL] Residual Bias: Model {m_sp} fold {m_fld} Tn5 bias counts Pearson r = {b_pr:.4f} (> 0.50)")
            elif b_pr > 0.30:
                qc_flags.append(f"[WARNING] Residual Bias: Model {m_sp} fold {m_fld} Tn5 bias counts Pearson r = {b_pr:.4f} (> 0.30)")
        if b_np_pr is not None and b_np_pr < 0.30:
            qc_flags.append(f"[WARNING] Background Fit: Model {m_sp} fold {m_fld} bias nonpeaks count Pearson r = {b_np_pr:.4f} (< 0.30)")
        if max_resp is not None:
            if max_resp >= 0.005:
                qc_flags.append(f"[CRITICAL] Residual Tn5 Bias: Model {m_sp} fold {m_fld} max marginal footprint response = {max_resp:.4f} (>= 0.005)")
            elif max_resp >= 0.003:
                qc_flags.append(f"[WARNING] Residual Tn5 Bias: Model {m_sp} fold {m_fld} max marginal footprint response = {max_resp:.4f} (>= 0.003)")

print("=" * 80)
print("[*] BLOCK 2 AUTOMATED QC & INTEGRITY AUDIT SUMMARY")
print("=" * 80)
if not qc_flags:
    print("[PASS] All QC checks within expected ranges.")
else:
    for flag in qc_flags:
        print(flag)
print("=" * 80)

# ------------------------------------------------------------------------------
# Part E: Export & Display Summary Tables
# ------------------------------------------------------------------------------
sample_qc_out = out_results_dir / "atac_sample_qc.tsv"
species_qc_out = out_results_dir / "atac_species_qc.tsv"
chrombpnet_qc_out = out_results_dir / "chrombpnet_qc.tsv"

df_sample_qc.to_csv(sample_qc_out, sep="\t", index=False)
df_species_qc.to_csv(species_qc_out, sep="\t", index=False)
df_cbp_qc.to_csv(chrombpnet_qc_out, sep="\t", index=False)

print(f"\n[+] Saved Part A Sample QC Table:     {sample_qc_out}")
print(f"[+] Saved Part B Species QC Table:    {species_qc_out}")
print(f"[+] Saved Part C ChromBPNet QC Table: {chrombpnet_qc_out}\n")

print("\n--- Part A: Per-Sample ATAC-seq Trimming & Alignment QC ---")
display(df_sample_qc)

print("\n--- Part B: Per-Species Stage Merge, Peak Calling & Splits ---")
display(df_species_qc)

print("\n--- Part C: Cross-Species ChromBPNet Stage 1 Base Models ---")
display(df_cbp_qc)


---
## Block 3: Stage 2 Continuous Pre-MZT Linear Probe & Grafted End-to-End Models

With late-blastula base encoders trained, we extract penultimate latent embeddings across late-blastula summits to predict pre-MZT dynamics:
- Fit `StandardScaler + RidgeCV` linear probes on training chromosomes.
- Evaluate on the held-out test chromosome: **reporting both all-peak Spearman $r$ and repeat-filtered metrics excluding `miR-430` tandem clusters**.
- Fold weights into `Dense(1)` and graft back onto the frozen encoder for forward + reverse-complement averaged scanning.

In [ ]:
# Fit RidgeCV Linear Probes on Real Latent Embeddings (Zebrafish primary, high-confidence peaks)
from scripts.infer_and_attribute import (
    load_chrombpnet_encoder,
    extract_summit_embeddings_from_genome,
    fit_linear_probes,
    GraftedChromBPNet
)

# ------------------------------------------------------------------------------
# Stage 2 Summit Filtering & QC Gate Settings
# ------------------------------------------------------------------------------
# Filter weak gDNA-control noise summits in Zebrafish (FE >= 4.0 & -log10(q) >= 5.0 -> ~49.4k peaks)
STAGE2_FILTER_HIGHCONF_PEAKS = globals().get('STAGE2_FILTER_HIGHCONF_PEAKS', globals().get('STAGE2_FILTER_HIGH_CONF_PEAKS', True))
STAGE2_MIN_FE = globals().get('STAGE2_MIN_FE', 4.0)
STAGE2_MIN_NEG_LOG10_Q = globals().get('STAGE2_MIN_NEG_LOG10_Q', 5.0)
MIN_STAGE1_COUNT_PEARSON_R = 0.50   # QC gate before promoting a species model into Blocks 4-5 ensembles
MEDAKA_PASSED_QC = False
ALLOW_LOW_QC_MEDAKA = False         # Keep False so low-SNR Medaka (r=0.39, TSSe=1.35/2.23) is not averaged 50/50 with ZF
RUN_MEDAKA_DIAGNOSTIC_PROBE = True  # Still fit & log Medaka probe metrics as diagnostic-only (saved separately)

def filter_summit_targets(
    targets_tsv: Path,
    narrowpeak_path: Path,
    min_fe: float = 4.0,
    min_neg_log10_q: float = 5.0,
    apply_filter: bool = True
) -> pd.DataFrame:
    """Load summit targets with string chroms, attach fold_enrichment from narrowPeak, and filter weak peaks."""
    df = pd.read_csv(targets_tsv, sep="\t", dtype={"chrom": str}, low_memory=False)
    n_raw = len(df)

    # Drop unplaced scaffolds outside train/valid/test splits
    if "split" in df.columns:
        df = df[df["split"].isin(["train", "valid", "test"])].copy()

    # Join fold_enrichment (col 6) and -log10(q) (col 8) from narrowPeak keyed by peak_id (col 3)
    if narrowpeak_path.exists():
        np_df = pd.read_csv(
            narrowpeak_path,
            sep="\t",
            header=None,
            usecols=[3, 6, 8],
            names=["peak_id", "fold_enrichment", "neg_log10_q"],
            dtype={"peak_id": str}
        ).drop_duplicates(subset=["peak_id"])
        df = df.merge(np_df, on="peak_id", how="left")
    else:
        print(f"[-] Notice: narrowPeak file not found at {narrowpeak_path}; using peak_score (-log10 q) only.")
        df["fold_enrichment"] = np.nan
        df["neg_log10_q"] = pd.to_numeric(df.get("peak_score", 0.0), errors="coerce")

    if apply_filter:
        q_vals = df["neg_log10_q"].fillna(pd.to_numeric(df.get("peak_score", 0.0), errors="coerce")).fillna(0.0)
        keep_mask = q_vals >= min_neg_log10_q
        if df["fold_enrichment"].notna().any():
            keep_mask = keep_mask & (df["fold_enrichment"].fillna(0.0) >= min_fe)
        df = df.loc[keep_mask].copy()

    df = df.reset_index(drop=True)
    split_counts = df["split"].value_counts().to_dict() if "split" in df.columns else {}
    n_rep = int(df["is_repeat_downweight"].fillna(False).sum()) if "is_repeat_downweight" in df.columns else 0
    print(f"[+] Summit targets loaded from {targets_tsv.name}: {n_raw:,} raw -> {len(df):,} kept "
          f"(filter={apply_filter}, FE>={min_fe}, -log10q>={min_neg_log10_q}; repeats flagged={n_rep})")
    print(f"    Split counts: {split_counts}")
    return df

zf_atac_dir = Path(WORK_DIR) / "atac" / "Danio_rerio"
zf_targets_path = zf_atac_dir / "Danio_rerio_summit_targets.tsv"
if not zf_targets_path.exists():
    raise FileNotFoundError(f"[FATAL] Summit targets table not found at {zf_targets_path}. Please wait for prep_atac_danio_rerio to complete.")

zf_narrowpeak_path = zf_atac_dir / "peaks" / "Danio_rerio_late_peaks.narrowPeak"
if not zf_narrowpeak_path.exists():
    zf_narrowpeak_path = zf_atac_dir / "peaks" / "Danio_rerio_late_peaks.primary.narrowPeak"

df_summit_targets = filter_summit_targets(
    targets_tsv=zf_targets_path,
    narrowpeak_path=zf_narrowpeak_path,
    min_fe=STAGE2_MIN_FE,
    min_neg_log10_q=STAGE2_MIN_NEG_LOG10_Q,
    apply_filter=STAGE2_FILTER_HIGHCONF_PEAKS
)

# Find real trained ChromBPNet debiased checkpoint strictly inside chrombpnet_run/models/ or models/
valid_model_exts = {".pt", ".pth", ".ckpt", ".h5", ".hdf5"}
zf_fold_dir = Path(WORK_DIR) / "models" / "chrombpnet" / "Danio_rerio" / f"fold_{FOLDS_TO_TRAIN[0]}"
cands = [p for p in zf_fold_dir.glob("**/models/*nobias*.*") if p.suffix.lower() in valid_model_exts and "bias_run" not in str(p) and "bias_model" not in str(p)]
if not cands:
    cands = [p for p in zf_fold_dir.glob("*nobias*.*") if p.suffix.lower() in valid_model_exts and "bias_run" not in str(p) and "bias_model" not in str(p)]

if not cands:
    raise FileNotFoundError(f"[FATAL] Debiased Stage 1 checkpoint (*nobias*.pt/h5) not found under {zf_fold_dir}. Zero-mock policy strictly enforced.")
checkpoint_path = cands[0]

print(f"[*] Loading trained Zebrafish ChromBPNet debiased encoder: {checkpoint_path}")
device = "cuda" if torch.cuda.is_available() else "cpu"
encoder_zf = load_chrombpnet_encoder(checkpoint_path, device=device)

# Display Stage 1 post-training QC metrics (including debiased count r & max Tn5 footprint response)
meta_json = zf_fold_dir / "training_meta.json"
if meta_json.exists():
    with open(meta_json) as f:
        meta_data = json.load(f)
    qc = meta_data.get("qc_metrics", {})
    print("=" * 80)
    print(f"[*] Zebrafish Stage 1 ChromBPNet QC Evaluation Metrics (From {meta_json.name}):")
    print(f"    Full Count Pearson r:      {qc.get('count_pearson_r')}")
    print(f"    Debiased Count Pearson r:  {qc.get('nobias_count_pearson_r')}")
    print(f"    Profile Pearson r:         {qc.get('profile_pearson_r')}")
    print(f"    Profile Median JSD:        {qc.get('profile_median_jsd')}")
    print(f"    Max Tn5 Footprint Resp:    {qc.get('max_tn5_footprint_response', qc.get('max_bias_response'))}")
    print("=" * 80)

print(f"[*] Extracting real penultimate embeddings over {len(df_summit_targets):,} high-confidence summit windows (2,114 bp, forward+RC averaged)...")
real_embeddings_zf = extract_summit_embeddings_from_genome(
    encoder=encoder_zf,
    genome_fa_path=ZEBRAFISH_GENOME_FA,
    df_summits=df_summit_targets,
    window_bp=2114,
    batch_size=64,
    device=device
)

probe_results_zf = fit_linear_probes(
    embeddings=real_embeddings_zf,
    df_targets=df_summit_targets,
    test_split_col="split",
    test_split_val="test",
    exclude_housekeeping=EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2
)


In [ ]:
if not MEDAKA_PASSED_QC:
    grafted_medaka_model = None
    grafted_medaka_diagnostic_model = None

# Fold Linear Probes into End-to-End Grafted Models and Persist Checkpoints
from scripts.infer_and_attribute import save_grafted_checkpoint, load_grafted_model

scaler_zf = probe_results_zf["scaler"]
grafted_zf_model = GraftedChromBPNet(
    encoder=encoder_zf,
    coef_abs=probe_results_zf["y_abs"]["coef"],
    intercept_abs=probe_results_zf["y_abs"]["intercept"],
    coef_rel=probe_results_zf["y_rel"]["coef"],
    intercept_rel=probe_results_zf["y_rel"]["intercept"],
    scaler_mean=scaler_zf.mean_,
    scaler_scale=scaler_zf.scale_
)
grafted_zf_model.to(device)
grafted_zf_model.eval()

# Save canonical grafted Zebrafish model checkpoint
grafted_zf_path = zf_fold_dir / "grafted_chrombpnet.pt"
save_grafted_checkpoint(
    grafted_model=grafted_zf_model,
    out_path=grafted_zf_path,
    base_checkpoint_path=str(checkpoint_path),
    metadata={
        "species": "Danio_rerio",
        "fold": FOLDS_TO_TRAIN[0],
        "n_summits_filtered": int(len(df_summit_targets)),
        "min_fe": float(STAGE2_MIN_FE) if STAGE2_FILTER_HIGH_CONF_PEAKS else None,
        "min_neg_log10_q": float(STAGE2_MIN_NEG_LOG10_Q) if STAGE2_FILTER_HIGH_CONF_PEAKS else None,
        "test_spearman_r_abs": float(probe_results_zf["y_abs"]["test_spearman_r"]),
        "test_auroc_abs": float(probe_results_zf["y_abs"]["test_auroc"]),
        "early_late_auroc_abs": probe_results_zf["y_abs"].get("early_late_auroc"),
        "test_spearman_r_rel": float(probe_results_zf["y_rel"]["test_spearman_r"]),
    }
)
print(f"[+] Canonical Zebrafish grafted model saved to {grafted_zf_path}")

# ------------------------------------------------------------------------------
# Medaka Stage 2 Linear Probe (QC-Gated to prevent 50/50 noise dilution in Blocks 4-5)
# ------------------------------------------------------------------------------

if ENABLE_MEDAKA:
    med_fold_dir = Path(WORK_DIR) / "models" / "chrombpnet" / "Oryzias_latipes" / f"fold_{FOLDS_TO_TRAIN[0]}"
    med_targets_path = Path(WORK_DIR) / "atac" / "Oryzias_latipes" / "Oryzias_latipes_summit_targets.tsv"
    med_meta_json = med_fold_dir / "training_meta.json"
    med_count_r = None
    if med_meta_json.exists():
        with open(med_meta_json) as f:
            med_qc = json.load(f).get("qc_metrics", {})
        med_count_r = med_qc.get("nobias_count_pearson_r")
        if med_count_r is None:
            med_count_r = med_qc.get("count_pearson_r")

    MEDAKA_PASSED_QC = (med_count_r is not None) and (float(med_count_r) >= MIN_STAGE1_COUNT_PEARSON_R)
    promote_medaka = MEDAKA_PASSED_QC or ALLOW_LOW_QC_MEDAKA

    if not promote_medaka:
        print("=" * 80)
        print(f"[QC GATE] Medaka Stage 1 debiased count Pearson r = {med_count_r} (< {MIN_STAGE1_COUNT_PEARSON_R:.2f}).")
        print("          Excluding Medaka from canonical Blocks 4-5 50/50 ensemble to avoid diluting Zebrafish predictions.")
        print("          (Set ALLOW_LOW_QC_MEDAKA = True in Cell 1 to override.)")
        print("=" * 80)
        # Ensure no stale canonical grafted_chrombpnet.pt gets auto-loaded by Blocks 4-5 on fresh kernel restart
        stale_med_ckpt = med_fold_dir / "grafted_chrombpnet.pt"
        if stale_med_ckpt.exists():
            backup_ckpt = med_fold_dir / "grafted_chrombpnet_low_snr_diagnostic.pt"
            stale_med_ckpt.rename(backup_ckpt)
            print(f"[*] Moved existing low-SNR Medaka checkpoint -> {backup_ckpt.name} so Blocks 4-5 stay Zebrafish-only.")

    valid_model_exts = {".pt", ".pth", ".ckpt", ".h5", ".hdf5"}
    med_cands = [p for p in med_fold_dir.glob("**/models/*nobias*.*") if p.suffix.lower() in valid_model_exts and "bias_run" not in str(p) and "bias_model" not in str(p)]
    if not med_cands:
        med_cands = [p for p in med_fold_dir.glob("*nobias*.*") if p.suffix.lower() in valid_model_exts and "bias_run" not in str(p) and "bias_model" not in str(p)]

    if (promote_medaka or RUN_MEDAKA_DIAGNOSTIC_PROBE) and med_targets_path.exists() and med_cands:
        # Do NOT apply FE >= 4 filter to Medaka (only 245 peaks survive FE >= 4 due to self-pileup lambda suppression)
        df_med_targets = pd.read_csv(med_targets_path, sep="\t", dtype={"chrom": str}, low_memory=False)
        if "split" in df_med_targets.columns:
            df_med_targets = df_med_targets[df_med_targets["split"].isin(["train", "valid", "test"])].reset_index(drop=True)

        encoder_med = load_chrombpnet_encoder(med_cands[0], device=device)
        print(f"[*] Extracting penultimate embeddings over {len(df_med_targets):,} Medaka summits...")
        embeddings_med = extract_summit_embeddings_from_genome(
            encoder=encoder_med,
            genome_fa_path=MEDAKA_GENOME_FA,
            df_summits=df_med_targets,
            window_bp=2114,
            batch_size=64,
            device=device
        )
        probe_results_med = fit_linear_probes(
            embeddings=embeddings_med,
            df_targets=df_med_targets,
            test_split_col="split",
            test_split_val="test",
            exclude_housekeeping=EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2
        )
        scaler_med = probe_results_med["scaler"]
        med_model_built = GraftedChromBPNet(
            encoder=encoder_med,
            coef_abs=probe_results_med["y_abs"]["coef"],
            intercept_abs=probe_results_med["y_abs"]["intercept"],
            coef_rel=probe_results_med["y_rel"]["coef"],
            intercept_rel=probe_results_med["y_rel"]["intercept"],
            scaler_mean=scaler_med.mean_,
            scaler_scale=scaler_med.scale_
        )
        med_model_built.to(device)
        med_model_built.eval()

        if promote_medaka:
            grafted_medaka_model = med_model_built
            grafted_med_path = med_fold_dir / "grafted_chrombpnet.pt"
            save_grafted_checkpoint(
                grafted_model=grafted_medaka_model,
                out_path=grafted_med_path,
                base_checkpoint_path=str(med_cands[0]),
                metadata={"species": "Oryzias_latipes", "fold": FOLDS_TO_TRAIN[0], "stage1_count_r": med_count_r}
            )
            print(f"[+] Canonical Medaka grafted model saved to {grafted_med_path}")
        else:
            grafted_medaka_diagnostic_model = med_model_built
            diag_med_path = med_fold_dir / "grafted_chrombpnet_low_snr_diagnostic.pt"
            save_grafted_checkpoint(
                grafted_model=grafted_medaka_diagnostic_model,
                out_path=diag_med_path,
                base_checkpoint_path=str(med_cands[0]),
                metadata={"species": "Oryzias_latipes", "fold": FOLDS_TO_TRAIN[0], "stage1_count_r": med_count_r, "diagnostic_only": True}
            )
            print(f"[+] Saved diagnostic-only Medaka model to {diag_med_path} (grafted_medaka_model remains None for Blocks 4-5).")

# ------------------------------------------------------------------------------
# Multi-fold Ensemble Wrappers
# ------------------------------------------------------------------------------
grafted_zf_models = [grafted_zf_model]
if len(FOLDS_TO_TRAIN) > 1:
    print(f"[*] Multi-fold training enabled: fitting linear probes across {len(FOLDS_TO_TRAIN)} Zebrafish folds...")
    zf_folds_json = Path(WORK_DIR) / "atac" / "Danio_rerio" / "chr_folds.json"
    all_zf_folds = {}
    if zf_folds_json.exists():
        with open(zf_folds_json) as f:
            all_zf_folds = json.load(f)

    for fold_i in FOLDS_TO_TRAIN[1:]:
        f_dir = Path(WORK_DIR) / "models" / "chrombpnet" / "Danio_rerio" / f"fold_{fold_i}"
        f_cands = [p for p in f_dir.glob("**/models/*nobias*.*") if p.suffix.lower() in valid_model_exts and "bias_run" not in str(p) and "bias_model" not in str(p)]
        if not f_cands:
            f_cands = [p for p in f_dir.glob("*nobias*.*") if p.suffix.lower() in valid_model_exts and "bias_run" not in str(p) and "bias_model" not in str(p)]
        if f_cands:
            f_enc = load_chrombpnet_encoder(f_cands[0], device=device)
            f_embs = extract_summit_embeddings_from_genome(f_enc, ZEBRAFISH_GENOME_FA, df_summit_targets, device=device)
            df_f = df_summit_targets.copy()
            if f"fold_{fold_i}" in all_zf_folds:
                f_info = all_zf_folds[f"fold_{fold_i}"]
                test_c = set(str(c) for c in f_info.get("test", []))
                val_c = set(str(c) for c in f_info.get("valid", []))
                train_c = set(str(c) for c in f_info.get("train", []))
                df_f["split"] = np.where(
                    df_f["chrom"].astype(str).isin(test_c), "test",
                    np.where(df_f["chrom"].astype(str).isin(val_c), "valid",
                             np.where(df_f["chrom"].astype(str).isin(train_c), "train", "excluded"))
                )
            f_res = fit_linear_probes(f_embs, df_f, test_split_col="split", test_split_val="test", exclude_housekeeping=EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2)
            f_grafted = GraftedChromBPNet(f_enc, f_res["y_abs"]["coef"], f_res["y_abs"]["intercept"], f_res["y_rel"]["coef"], f_res["y_rel"]["intercept"], f_res["scaler"].mean_, f_res["scaler"].scale_)
            f_grafted.to(device)
            f_grafted.eval()
            save_grafted_checkpoint(grafted_model=f_grafted, out_path=f_dir / "grafted_chrombpnet.pt", base_checkpoint_path=str(f_cands[0]))
            grafted_zf_models.append(f_grafted)
    print(f"[+] Constructed Zebrafish multi-fold ensemble with {len(grafted_zf_models)} models.")

grafted_medaka_models = [grafted_medaka_model] if grafted_medaka_model is not None else []
if ENABLE_MEDAKA and len(FOLDS_TO_TRAIN) > 1 and grafted_medaka_model is not None:
    print(f"[*] Multi-fold training enabled: fitting linear probes across {len(FOLDS_TO_TRAIN)} Medaka folds...")
    med_folds_json = Path(WORK_DIR) / "atac" / "Oryzias_latipes" / "chr_folds.json"
    all_med_folds = {}
    if med_folds_json.exists():
        with open(med_folds_json) as f:
            all_med_folds = json.load(f)

    for fold_i in FOLDS_TO_TRAIN[1:]:
        f_dir = Path(WORK_DIR) / "models" / "chrombpnet" / "Oryzias_latipes" / f"fold_{fold_i}"
        f_cands = [p for p in f_dir.glob("**/models/*nobias*.*") if p.suffix.lower() in valid_model_exts and "bias_run" not in str(p) and "bias_model" not in str(p)]
        if not f_cands:
            f_cands = [p for p in f_dir.glob("*nobias*.*") if p.suffix.lower() in valid_model_exts and "bias_run" not in str(p) and "bias_model" not in str(p)]
        if f_cands and "df_med_targets" in locals():
            f_enc = load_chrombpnet_encoder(f_cands[0], device=device)
            f_embs = extract_summit_embeddings_from_genome(f_enc, MEDAKA_GENOME_FA, df_med_targets, device=device)
            df_mf = df_med_targets.copy()
            if f"fold_{fold_i}" in all_med_folds:
                f_info = all_med_folds[f"fold_{fold_i}"]
                test_c = set(str(c) for c in f_info.get("test", []))
                val_c = set(str(c) for c in f_info.get("valid", []))
                train_c = set(str(c) for c in f_info.get("train", []))
                df_mf["split"] = np.where(
                    df_mf["chrom"].astype(str).isin(test_c), "test",
                    np.where(df_mf["chrom"].astype(str).isin(val_c), "valid",
                             np.where(df_mf["chrom"].astype(str).isin(train_c), "train", "excluded"))
                )
            f_res = fit_linear_probes(f_embs, df_mf, test_split_col="split", test_split_val="test", exclude_housekeeping=EXCLUDE_HOUSEKEEPING_PROMOTERS_STAGE2)
            f_grafted = GraftedChromBPNet(f_enc, f_res["y_abs"]["coef"], f_res["y_abs"]["intercept"], f_res["y_rel"]["coef"], f_res["y_rel"]["intercept"], f_res["scaler"].mean_, f_res["scaler"].scale_)
            f_grafted.to(device)
            f_grafted.eval()
            save_grafted_checkpoint(grafted_model=f_grafted, out_path=f_dir / "grafted_chrombpnet.pt", base_checkpoint_path=str(f_cands[0]))
            grafted_medaka_models.append(f_grafted)


---
## Block 4: Zero-Shot Validation on Target-Species Minor-Wave Genes & Medaka ATAC

We test whether ChromBPNet models trained exclusively on Zebrafish transfer across teleosts without any target chromatin training data:
- Tile $\pm 25$ kb around TSSs of target-species genes.
- **Primary Clean Positive Sets:** `strictly_zygotic_minor_wave` and `slam_confirmed_minor_wave` vs. `control_non_activated`.
- **Secondary Diagnostic Series:** `all_upregulated` (clearly labeled so cytoplasmic polyadenylation does not contaminate the benchmark).
- **Zero-Shot Zebrafish $\rightarrow$ Medaka Pre-MZT ATAC Benchmark:** Score Medaka `st9` vs `st11`-only summits when `ENABLE_MEDAKA = True`.

In [ ]:
# ==============================================================================
# Block 4a: Launch Standalone Zero-Shot Cross-Species Validation Task
# ==============================================================================
folds_arg = ",".join(str(f) for f in FOLDS_TO_TRAIN)

# Set True once after updating Block 4 scoring/controls so TaskManager doesn't
# skip due to the previous .done marker, then revert to FORCE_RERUN.
FORCE_RERUN_BLOCK4 = True

VALIDATION_WINDOWS = ["promoter_1kb", "proximal_5kb", "bounded_25kb"]

zero_shoot_cmd = [
    "python", "-u", "scripts/zero_shoot_validation.py",
    "--work-dir", WORK_DIR,
    "--config", f"{WORK_DIR}/config.json",
    "--folds", folds_arg,
    "--batch-size", "256",
    "--primary-control", "control_silent",
    "--diagnostic-control", "control_non_activated",
    "--silent-max-tpm", "0.1",
    "--windows", ",".join(VALIDATION_WINDOWS),
    "--strict-early-max-tpm", "0.2",
    "--strict-premzt-min-tpm", "5.0",
    "--case-insensitive-orthologs",
    "--run-zebrafish-ceiling",
]
if ENABLE_MEDAKA:
    zero_shoot_cmd.append("--enable-medaka")
else:
    zero_shoot_cmd.append("--no-enable-medaka")

tm.submit(
    "zero_shoot_validation",
    zero_shoot_cmd,
    depends_on=[f"train_chrombpnet_danio_rerio_fold_{folds_arg}"],
    force_rerun=(FORCE_RERUN or FORCE_RERUN_BLOCK4),
    requires_gpu=True
)

display(tm.status())

In [ ]:
# ==============================================================================
# Block 4b: Inspect Validation Metrics & Render Figure 2 (Zero-Shot Benchmark)
# ==============================================================================
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
)

scored_tsv = res_dir / "target_genes_scored_metrics.tsv"
metrics_json_path = res_dir / "zero_shoot_validation_metrics.json"

# Primary display settings for ROC/PR curves (change to "proximal_5kb" or "bounded_25kb" to inspect)
PLOT_WINDOW = "promoter_1kb"
PLOT_SCORE = "y_abs"          # "y_abs", "y_rel", or "late"
PRIMARY_CTRL = "control_silent"
DIAG_CTRL = "control_non_activated"


def _bool_flag(df, col):
    if col in df.columns:
        return df[col].fillna(False).astype(bool)
    return pd.Series(False, index=df.index)


def _resolve_score_col(df, window, score_type):
    candidates = [
        f"pred_{window}_{score_type}",
        f"pred_max_{window}_{score_type}",
        f"pred_max_{score_type}",  # backward-compatible fallback
    ]
    for c in candidates:
        if c in df.columns:
            return c
    return None


def _eval_binary(df, pos_mask, ctrl_mask, score_col):
    if score_col is None or score_col not in df.columns:
        return None
    pos_s = df.loc[pos_mask, score_col].dropna().values
    ctrl_s = df.loc[ctrl_mask, score_col].dropna().values
    if len(pos_s) == 0 or len(ctrl_s) == 0:
        return {
            "roc_auc": np.nan, "pr_auc": np.nan, "p_val": np.nan,
            "n_pos": len(pos_s), "n_ctrl": len(ctrl_s),
            "fpr": None, "tpr": None, "prec": None, "rec": None,
        }
    y_true = np.concatenate([np.ones(len(pos_s)), np.zeros(len(ctrl_s))])
    y_score = np.concatenate([pos_s, ctrl_s])
    fpr, tpr, _ = roc_curve(y_true, y_score)
    prec, rec, _ = precision_recall_curve(y_true, y_score)
    _, p_val = mannwhitneyu(pos_s, ctrl_s, alternative="greater")
    return {
        "roc_auc": float(roc_auc_score(y_true, y_score)),
        "pr_auc": float(average_precision_score(y_true, y_score)),
        "p_val": float(p_val),
        "n_pos": len(pos_s),
        "n_ctrl": len(ctrl_s),
        "fpr": fpr, "tpr": tpr, "prec": prec, "rec": rec,
    }


if not scored_tsv.exists():
    print(f"[*] Scored gene metrics table not found at {scored_tsv}. Waiting for 'zero_shoot_zeroshot_validation' task to complete.")
else:
    df_scored = pd.read_csv(scored_tsv, sep="\t")
    print(f"[+] Loaded scored gene metrics: {len(df_scored):,} genes from {scored_tsv.name}")

    # Load JSON report (includes Zebrafish in-species ceiling & Medaka st9 ATAC if generated)
    zero_shoot_metrics = {}
    if metrics_json_path.exists():
        with open(metrics_json_path, "r") as f_m:
            zero_shoot_metrics = json.load(f_m)
        print(f"[+] Loaded validation metrics report from {metrics_json_path.name}")
        if "zebrafish_ceiling" in zero_shoot_metrics:
            print("[+] In-Species Zebrafish Minor-Wave Ceiling (ATAC -> RNA):")
            for w_name, w_res in zero_shoot_metrics["zebrafish_ceiling"].items():
                y_abs_c = w_res.get("y_abs", {}).get("roc_auc", np.nan)
                y_rel_c = w_res.get("y_rel", {}).get("roc_auc", np.nan)
                late_c = w_res.get("late", {}).get("roc_auc", np.nan)
                print(f"    {w_name:<14} -> y_abs AUC: {y_abs_c:.3f} | y_rel AUC: {y_rel_c:.3f} | late AUC: {late_c:.3f}")
        if "medaka_st9_atac" in zero_shoot_metrics:
            mk = zero_shoot_metrics["medaka_st9_atac"]
            print(f"[+] Zero-Shot Medaka st9 Pre-MZT ATAC: Spearman r = {mk.get('spearman_r', np.nan):.3f}, AUROC = {mk.get('roc_auc', np.nan):.3f}")

    # Define positive & control masks
    pos_masks = {
        "strictly_zygotic_highconf": _bool_flag(df_scored, "is_strictly_zygotic_highconf") | (df_scored["category"] == "strictly_zygotic_highconf"),
        "strictly_zygotic_minor_wave": _bool_flag(df_scored, "is_strictly_zygotic_minor_wave") | (df_scored["category"] == "strictly_zygotic_minor_wave"),
        "slam_confirmed_minor_wave": _bool_flag(df_scored, "is_slam_confirmed_minor_wave") | _bool_flag(df_scored, "is_slam_confirmed") | (df_scored["category"] == "slam_confirmed_minor_wave"),
        "all_upregulated": _bool_flag(df_scored, "is_all_upregulated") | df_scored["category"].isin(["all_upregulated", "strictly_zygotic_minor_wave", "strictly_zygotic_highconf"]),
    }
    ctrl_masks = {
        "control_silent": _bool_flag(df_scored, "is_control_silent") | (df_scored["category"] == "control_silent"),
        "control_non_activated": _bool_flag(df_scored, "is_control_non_activated") | (df_scored["category"] == "control_non_activated"),
    }

    # Fall back gracefully if silent controls haven't been computed yet
    active_primary_ctrl = PRIMARY_CTRL if ctrl_masks[PRIMARY_CTRL].sum() > 0 else DIAG_CTRL
    if active_primary_ctrl != PRIMARY_CTRL:
        print(f"[!] '{PRIMARY_CTRL}' not found in TSV (n=0); falling back to '{active_primary_ctrl}'.")

    # Summary table across all available windows, score types, and control baselines
    available_windows = [w for w in ["promoter_1kb", "proximal_5kb", "bounded_25kb"] if _resolve_score_col(df_scored, w, "y_abs") is not None]
    if not available_windows:
        available_windows = ["legacy_25kb"]

    summary_rows = []
    primary_pos_key = "strictly_zygotic_highconf" if pos_masks["strictly_zygotic_highconf"].sum() >= 10 else "strictly_zygotic_minor_wave"
    for w in available_windows:
        for s_type in ["y_abs", "y_rel", "late"]:
            col = _resolve_score_col(df_scored, w, s_type)
            for c_name, c_mask in ctrl_masks.items():
                if c_mask.sum() == 0:
                    continue
                res = _eval_binary(df_scored, pos_masks[primary_pos_key], c_mask, col)
                if res is not None:
                    summary_rows.append({
                        "window": w, "score": s_type, "control": c_name,
                        "pos_set": primary_pos_key, "n_pos": res["n_pos"], "n_ctrl": res["n_ctrl"],
                        "roc_auc": round(res["roc_auc"], 3),
                        "pr_auc": round(res["pr_auc"], 3),
                        "p_val": f"{res['p_val']:.2e}" if np.isfinite(res["p_val"]) else "nan",
                    })
    df_summary = pd.DataFrame(summary_rows)
    if len(df_summary) > 0:
        print("\n[*] Window × Score × Control Comparison Summary:")
        display(df_summary)
        df_summary.to_csv(res_dir / "zero_shoot_window_control_comparison.tsv", sep="\t", index=False)

    # Choose active plotting window & score column
    plot_win = PLOT_WINDOW if PLOT_WINDOW in available_windows else available_windows[0]
    plot_col = _resolve_score_col(df_scored, plot_win, PLOT_SCORE)

    # Render 2x2 Figure 2:
    # Top row: ROC & PR curves against Primary Control (Silent)
    # Bottom row: AUROC comparison across Windows/Scores (Silent vs. Maternal-Stable Controls)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    curve_specs = [
        ("strictly_zygotic_highconf", "Strictly Zygotic (High-Conf)", "#a50f15", "-"),
        ("strictly_zygotic_minor_wave", "Strictly Zygotic (All)", "#ef3b2c", "-."),
        ("slam_confirmed_minor_wave", "SLAM-Confirmed Orthologs", "#2b83ba", "-"),
        ("all_upregulated", "All Upregulated (Diagnostic)", "#fdae61", "--"),
    ]

    ax_roc, ax_pr = axes[0, 0], axes[0, 1]
    for key, label_base, color, ls in curve_specs:
        res = _eval_binary(df_scored, pos_masks[key], ctrl_masks[active_primary_ctrl], plot_col)
        if res is not None and res["fpr"] is not None:
            ax_roc.plot(
                res["fpr"], res["tpr"],
                label=f"{label_base} (AUC={res['roc_auc']:.2f}, n={res['n_pos']})",
                color=color, lw=2.2, linestyle=ls,
            )
            ax_pr.plot(
                res["rec"], res["prec"],
                label=f"{label_base} (PR={res['pr_auc']:.2f}, n={res['n_pos']})",
                color=color, lw=2.2, linestyle=ls,
            )

    ax_roc.plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax_roc.set_xlabel("False Positive Rate", fontsize=11)
    ax_roc.set_ylabel("True Positive Rate", fontsize=11)
    ax_roc.set_title(f"Zero-Shot ROC ({plot_win} | {PLOT_SCORE} vs. {active_primary_ctrl})", fontweight="bold")
    ax_roc.legend(frameon=True, fontsize=8.5, loc="lower right")

    ax_pr.set_xlabel("Recall", fontsize=11)
    ax_pr.set_ylabel("Precision", fontsize=11)
    ax_pr.set_title(f"Zero-Shot PR ({plot_win} | {PLOT_SCORE} vs. {active_primary_ctrl})", fontweight="bold")
    ax_pr.legend(frameon=True, fontsize=8.5, loc="best")

    # Bottom-left: AUROC across windows & score types vs. Primary (Silent) Controls
    # Bottom-right: AUROC across windows & score types vs. Diagnostic (Maternal-Stable) Controls
    zf_ceiling = zero_shoot_metrics.get("zebrafish_ceiling", {})
    for ax_bar, ctrl_key, panel_title in [
        (axes[1, 0], "control_silent", f"Primary: {primary_pos_key} vs. Silent Controls"),
        (axes[1, 1], "control_non_activated", f"Diagnostic: {primary_pos_key} vs. Maternal-Stable Controls"),
    ]:
        sub = df_summary[df_summary["control"] == ctrl_key] if len(df_summary) > 0 else pd.DataFrame()
        if len(sub) == 0:
            ax_bar.text(0.5, 0.5, f"No '{ctrl_key}' rows available", ha="center", va="center")
            ax_bar.set_title(panel_title, fontweight="bold")
            continue
        x = np.arange(len(available_windows))
        width = 0.25
        for idx, (s_type, color) in enumerate([("y_abs", "#d7191c"), ("y_rel", "#2b83ba"), ("late", "#756bb1")]):
            vals = []
            for w in available_windows:
                m = sub[(sub["window"] == w) & (sub["score"] == s_type)]
                vals.append(float(m["roc_auc"].iloc[0]) if len(m) > 0 else np.nan)
            ax_bar.bar(x + (idx - 1) * width, vals, width=width, label=s_type, color=color, alpha=0.88)

        # Overlay zebrafish in-species ceiling markers if available (for silent-control panel)
        if ctrl_key == "control_silent" and zf_ceiling:
            ceil_vals = [zf_ceiling.get(w, {}).get("y_abs", {}).get("roc_auc", np.nan) for w in available_windows]
            if any(np.isfinite(ceil_vals)):
                ax_bar.plot(x, ceil_vals, "ko--", lw=1.5, ms=6, label="Zebrafish in-species ceiling (y_abs)")

        ax_bar.axhline(0.5, color="gray", linestyle="--", alpha=0.6)
        ax_bar.set_xticks(x)
        ax_bar.set_xticklabels(available_windows, fontsize=10)
        ax_bar.set_ylim(0.3, 1.0)
        ax_bar.set_ylabel("AUROC", fontsize=11)
        ax_bar.set_title(panel_title, fontweight="bold")
        ax_bar.legend(frameon=True, fontsize=8.5, loc="upper right")

    plt.tight_layout()
    fig2_path = res_dir / "fig2_zeroshot_benchmark.png"
    plt.savefig(fig2_path, dpi=300)
    plt.savefig(res_dir / "fig2_zeroshot_benchmark.svg")
    plt.show()
    print(f"[+] Figure 2 exported to {fig2_path}")

---
## Block 5: Unbiased Sliding-Window Scan of `TARGET_LOCUS_NAME`

We tile the target locus across species to discover putative gained enhancers driving pre-MZT activation:
- Dynamically discovers coordinates for `TARGET_GENES` (`notch2`, `slc35a3a`) directly from `TARGET_GTF`.
- Pads $\pm \text{LOCUS\_FLANK\_BP}$ (75 kb) to capture distal enhancers.
- Exports candidate enhancer BED and bedGraph tracks to `WORK_DIR/results/{TARGET_LOCUS_NAME}/`.

In [ ]:
# Fresh-kernel resume guards
device = "cuda" if torch.cuda.is_available() else "cpu"
folds_to_check = FOLDS_TO_TRAIN if "FOLDS_TO_TRAIN" in locals() else [0]
if "df_counts" not in locals() or df_counts is None:
    gene_sets_tsv = res_dir / f"{TARGET_LOCUS_NAME}_gene_sets.tsv"
    if not gene_sets_tsv.exists():
        gene_sets_tsv = Path(WORK_DIR) / "rnaseq" / "target_species_gene_sets.tsv"
    if gene_sets_tsv.exists():
        df_counts = pd.read_csv(gene_sets_tsv, sep="\t")

if "grafted_zf_models" not in locals() or not grafted_zf_models:
    from scripts.infer_and_attribute import load_grafted_model
    grafted_zf_models = []
    for f_idx in folds_to_check:
        zf_ckpt = Path(WORK_DIR) / "models" / "chrombpnet" / "Danio_rerio" / f"fold_{f_idx}" / "grafted_chrombpnet.pt"
        if zf_ckpt.exists():
            grafted_zf_models.append(load_grafted_model(zf_ckpt, device=device))
    if grafted_zf_models:
        grafted_zf_model = grafted_zf_models[0]
        print(f"[+] Fresh kernel: loaded {len(grafted_zf_models)} grafted Zebrafish model(s)")
    else:
        grafted_zf_model = None
elif "grafted_zf_model" not in locals() or grafted_zf_model is None:
    grafted_zf_model = grafted_zf_models[0]

if "grafted_medaka_models" not in locals() or not grafted_medaka_models:
    from scripts.infer_and_attribute import load_grafted_model
    grafted_medaka_models = []
    for f_idx in folds_to_check:
        med_ckpt = Path(WORK_DIR) / "models" / "chrombpnet" / "Oryzias_latipes" / f"fold_{f_idx}" / "grafted_chrombpnet.pt"
        if med_ckpt.exists():
            grafted_medaka_models.append(load_grafted_model(med_ckpt, device=device))
    if grafted_medaka_models:
        grafted_medaka_model = grafted_medaka_models[0]
        print(f"[+] Fresh kernel: loaded {len(grafted_medaka_models)} grafted Medaka model(s)")
    else:
        grafted_medaka_model = None
elif "grafted_medaka_model" not in locals() or grafted_medaka_model is None:
    grafted_medaka_model = grafted_medaka_models[0] if grafted_medaka_models else None

# Unbiased Sliding-Window Scanning across Target Locus & Cross-Species Loci
from scripts.infer_and_attribute import scan_locus_sliding_window
import matplotlib.pyplot as plt
import pyBigWig
import gzip

gene_sets_tsv = res_dir / f"{TARGET_LOCUS_NAME}_gene_sets.tsv"
df_counts = pd.read_csv(gene_sets_tsv, sep="\t")

# Store (gname, chrom, st, en) tuples so labels never misalign
tg_coords = []
for tg in TARGET_GENES:
    tg_rows = df_counts[df_counts["gene_name"].str.lower() == tg.lower()]
    if not tg_rows.empty:
        r = tg_rows.iloc[0]
        tg_coords.append((r["gene_name"], str(r["chrom"]), int(r["start"]), int(r["end"])))

assert tg_coords, f"None of the target genes {TARGET_GENES} found in GTF!"
locus_chr = tg_coords[0][1]
locus_start = max(0, min(c[2] for c in tg_coords) - LOCUS_FLANK_BP)
locus_end = max(c[3] for c in tg_coords) + LOCUS_FLANK_BP
locus_len = locus_end - locus_start

print(f"[+] Scanning locus {TARGET_LOCUS_NAME}: {locus_chr}:{locus_start:,}-{locus_end:,} ({locus_len:,} bp)")
window_bp = 2114
stride_bp = 250
device = "cuda" if torch.cuda.is_available() else "cpu"

if "grafted_zf_models" in locals() and len(grafted_zf_models) > 1:
    print(f"[*] Multi-fold ensemble scanning target locus across {len(grafted_zf_models)} Zebrafish folds...")
    zf_abs_list, zf_rel_list, zf_late_list = [], [], []
    for m_fold in grafted_zf_models:
        pos_f, abs_f, rel_f, late_f = scan_locus_sliding_window(
            grafted_model=m_fold,
            genome_fa_path=TARGET_GENOME_FA,
            chrom=locus_chr,
            start_bp=locus_start,
            end_bp=locus_end,
            window_bp=window_bp,
            stride_bp=stride_bp,
            batch_size=64,
            device=device,
            return_late=True
        )
        zf_abs_list.append(abs_f)
        zf_rel_list.append(rel_f)
        zf_late_list.append(late_f)
    positions = pos_f
    target_pred_abs_zf = np.mean(zf_abs_list, axis=0)
    target_pred_rel_zf = np.mean(zf_rel_list, axis=0)
    target_pred_late_zf = np.mean(zf_late_list, axis=0)
else:
    positions, target_pred_abs_zf, target_pred_rel_zf, target_pred_late_zf = scan_locus_sliding_window(
        grafted_model=grafted_zf_model,
        genome_fa_path=TARGET_GENOME_FA,
        chrom=locus_chr,
        start_bp=locus_start,
        end_bp=locus_end,
        window_bp=window_bp,
        stride_bp=stride_bp,
        batch_size=64,
        device=device,
        return_late=True
    )

target_pred_abs = target_pred_abs_zf
target_pred_rel = target_pred_rel_zf
target_pred_late = target_pred_late_zf

# Multi-fold ensembling for Medaka models whenever len(grafted_medaka_models) > 1
if grafted_medaka_model is not None:
    if "grafted_medaka_models" in locals() and len(grafted_medaka_models) > 1:
        print(f"[*] Multi-fold ensemble scanning target locus across {len(grafted_medaka_models)} Medaka folds...")
        med_abs_list, med_rel_list, med_late_list = [], [], []
        for m_fold in grafted_medaka_models:
            _, m_abs, m_rel, m_late = scan_locus_sliding_window(
                grafted_model=m_fold,
                genome_fa_path=TARGET_GENOME_FA,
                chrom=locus_chr,
                start_bp=locus_start,
                end_bp=locus_end,
                window_bp=window_bp,
                stride_bp=stride_bp,
                batch_size=64,
                device=device,
                return_late=True
            )
            med_abs_list.append(m_abs)
            med_rel_list.append(m_rel)
            med_late_list.append(m_late)
        med_abs = np.mean(med_abs_list, axis=0)
        med_rel = np.mean(med_rel_list, axis=0)
        med_late = np.mean(med_late_list, axis=0)
    else:
        _, med_abs, med_rel, med_late = scan_locus_sliding_window(
            grafted_model=grafted_medaka_model,
            genome_fa_path=TARGET_GENOME_FA,
            chrom=locus_chr,
            start_bp=locus_start,
            end_bp=locus_end,
            window_bp=window_bp,
            stride_bp=stride_bp,
            batch_size=64,
            device=device,
            return_late=True
        )
    ENSEMBLE_MEDAKA_WITH_ZF = globals().get("ENSEMBLE_MEDAKA_WITH_ZF", False)
    target_pred_abs_consensus = 0.5 * (target_pred_abs_zf + med_abs)
    target_pred_rel_consensus = 0.5 * (target_pred_rel_zf + med_rel)
    target_pred_late_consensus = 0.5 * (target_pred_late_zf + med_late)
    if ENSEMBLE_MEDAKA_WITH_ZF:
        target_pred_abs = target_pred_abs_consensus
        target_pred_rel = target_pred_rel_consensus
        target_pred_late = target_pred_late_consensus
    else:
        target_pred_abs = target_pred_abs_zf
        target_pred_rel = target_pred_rel_zf
        target_pred_late = target_pred_late_zf

enhancer_threshold = np.percentile(target_pred_abs, 95)
is_enhancer_raw = target_pred_abs > enhancer_threshold

# Non-maximum suppression with min_distance = 1500 bp
# Combined ranking metric incorporating both absolute accessibility and pre-MZT timing specificity
z_abs = (target_pred_abs - np.mean(target_pred_abs)) / (np.std(target_pred_abs) + 1e-6)
z_rel = (target_pred_rel - np.mean(target_pred_rel)) / (np.std(target_pred_rel) + 1e-6)
composite_score = z_abs + z_rel

cand_indices = np.where(is_enhancer_raw)[0]
cand_sorted = cand_indices[np.argsort(-composite_score[cand_indices])]
kept_centers = []
is_enhancer = np.zeros(len(positions), dtype=bool)

for idx in cand_sorted:
    pos = positions[idx]
    if not any(abs(pos - kc) < 1500 for kc in kept_centers):
        kept_centers.append(pos)
        is_enhancer[idx] = True

print(f"[+] Identified {len(kept_centers)} non-redundant candidate enhancer summits after NMS (min_distance=1500 bp).")

# Export BED, TSV, and IGV-ready bedGraph tracks
bed_path = res_dir / f"{TARGET_LOCUS_NAME}_candidate_enhancers.bed"
tsv_path = res_dir / f"{TARGET_LOCUS_NAME}_candidate_enhancers.tsv"
bg_abs_path = res_dir / f"{TARGET_LOCUS_NAME}_pred_y_abs.bedgraph"
bg_rel_path = res_dir / f"{TARGET_LOCUS_NAME}_pred_y_rel.bedgraph"
bg_late_path = res_dir / f"{TARGET_LOCUS_NAME}_pred_late.bedgraph"

half_stride = stride_bp // 2
records = []
with open(bed_path, "w") as f_bed, open(bg_abs_path, "w") as f_bg_a, open(bg_rel_path, "w") as f_bg_r, open(bg_late_path, "w") as f_bg_l:
    for idx, (pos, is_e) in enumerate(zip(positions, is_enhancer)):
        st = max(0, pos - window_bp // 2)
        en = st + window_bp
        # Non-overlapping step interval centered at pos for IGV and bedGraphToBigWig
        step_st = max(locus_start, pos - half_stride)
        step_en = max(step_st + 1, min(locus_end, pos + half_stride))
        score_a = float(target_pred_abs[idx])
        score_r = float(target_pred_rel[idx])
        score_l = float(target_pred_late[idx])
        score_comp = float(composite_score[idx])
        rec_entry = {
            "chrom": locus_chr,
            "start": st,
            "end": en,
            "center": pos,
            "pred_abs": round(score_a, 4),
            "pred_rel": round(score_r, 4),
            "composite_score": round(score_comp, 4),
            "pred_late": round(score_l, 4),
            "pred_abs_zf": round(float(target_pred_abs_zf[idx]), 4),
            "pred_rel_zf": round(float(target_pred_rel_zf[idx]), 4),
            "is_enhancer": bool(is_e),
            "is_candidate": bool(is_e)
        }
        if grafted_medaka_model is not None and "med_abs" in locals():
            rec_entry["pred_abs_medaka"] = round(float(med_abs[idx]), 4)
            rec_entry["pred_rel_medaka"] = round(float(med_rel[idx]), 4)
            rec_entry["pred_abs_consensus"] = round(float(target_pred_abs_consensus[idx]), 4)
            rec_entry["pred_rel_consensus"] = round(float(target_pred_rel_consensus[idx]), 4)
        records.append(rec_entry)
        if is_e:
            f_bed.write(f"{locus_chr}\t{st}\t{en}\t{TARGET_LOCUS_NAME}_peak_{idx}\t{int(score_a * 100)}\t.\t{pos}\t{pos}\n")
        f_bg_a.write(f"{locus_chr}\t{step_st}\t{step_en}\t{score_a:.4f}\n")
        f_bg_r.write(f"{locus_chr}\t{step_st}\t{step_en}\t{score_r:.4f}\n")
        f_bg_l.write(f"{locus_chr}\t{step_st}\t{step_en}\t{score_l:.4f}\n")

df_enh = pd.DataFrame(records)
df_enh.to_csv(tsv_path, sep="\t", index=False)
print(f"[+] Candidate enhancers BED written to {bed_path}")
print(f"[+] Candidate enhancers TSV written to {tsv_path}")
print(f"[+] IGV-ready bedGraph tracks: {bg_abs_path.name}, {bg_rel_path.name}")

# Cross-Species Scanning: Zebrafish Orthologous Locus
zf_locus_info = None
zf_pred_abs = None
try:
    zf_genes_dict = {}
    with (gzip.open(ZEBRAFISH_GTF, "rt") if str(ZEBRAFISH_GTF).endswith(".gz") else open(ZEBRAFISH_GTF, "rt")) as f:
        for line in f:
            if line.startswith("#"):
                continue
            p = line.strip().split("\t")
            if len(p) >= 9 and p[2] == "gene":
                for tg in TARGET_GENES:
                    tg_l = tg.lower()
                    if f'gene_name "{tg_l}"' in p[8].lower() or f'"{tg_l}"' in p[8].lower():
                        zf_genes_dict[tg_l] = (p[0], int(p[3]), int(p[4]))
    if zf_genes_dict:
        primary_tg = TARGET_GENES[0].lower()
        if primary_tg in zf_genes_dict:
            zf_chr = zf_genes_dict[primary_tg][0]
        else:
            zf_chr = list(zf_genes_dict.values())[0][0]
        chr_genes = {k: v for k, v in zf_genes_dict.items() if v[0] == zf_chr}
        zf_st = max(0, min(v[1] for v in chr_genes.values()) - LOCUS_FLANK_BP)
        zf_en = max(v[2] for v in chr_genes.values()) + LOCUS_FLANK_BP
        print(f"[+] Scanning orthologous Zebrafish locus: {zf_chr}:{zf_st:,}-{zf_en:,}")
        # CRITICAL LEAKAGE FIX: Score Zebrafish reference locus strictly with held-out fold_0 model
        # (folds 1-4 were trained on the chromosome containing notch2!)
        eval_zf_model = grafted_zf_models[0] if (isinstance(grafted_zf_models, list) and len(grafted_zf_models) > 0) else grafted_zf_model
        print(f"[*] Scoring Zebrafish reference locus strictly with held-out fold_0 model (zero data leakage)...")
        zf_positions, zf_pred_abs, zf_pred_rel = scan_locus_sliding_window(
            grafted_model=eval_zf_model,
            genome_fa_path=ZEBRAFISH_GENOME_FA,
            chrom=zf_chr,
            start_bp=zf_st,
            end_bp=zf_en,
            window_bp=window_bp,
            stride_bp=stride_bp,
            batch_size=64,
            device=device
        )
        zf_locus_info = (zf_chr, zf_st, zf_en, zf_positions, zf_pred_abs)
        # Compute lineage-gain summary comparing candidate peaks to ZF ortholog maximum
        zf_max_abs = float(np.max(zf_pred_abs))
        df_enh["zf_ortholog_max_abs"] = zf_max_abs
        df_enh["lineage_gain_delta"] = df_enh["pred_abs"] - zf_max_abs
        df_enh.to_csv(tsv_path, sep="\t", index=False)
        print(f"[+] Computed lineage-gain delta vs Zebrafish ortholog max ({zf_max_abs:.4f}). Updated {tsv_path.name}")
except Exception as e:
    print(f"[-] Notice scanning Zebrafish orthologous locus: {e}")

# Cross-Species Scanning: Medaka Orthologous Locus (when ENABLE_MEDAKA)
med_locus_info = None
if ENABLE_MEDAKA and Path(MEDAKA_GTF).exists() and Path(MEDAKA_GENOME_FA).exists():
    try:
        med_genes_dict = {}
        with (gzip.open(MEDAKA_GTF, "rt") if str(MEDAKA_GTF).endswith(".gz") else open(MEDAKA_GTF, "rt")) as f:
            for line in f:
                if line.startswith("#"):
                    continue
                p = line.strip().split("\t")
                if len(p) >= 9 and p[2] == "gene":
                    for tg in TARGET_GENES:
                        tg_l = tg.lower()
                        # Match exact first, then fallback to stripping a single trailing a/b
                        if f'gene_name "{tg_l}"' in p[8].lower() or f'"{tg_l}"' in p[8].lower():
                            med_genes_dict[tg_l] = (p[0], int(p[3]), int(p[4]))
                        elif len(tg_l) > 3 and tg_l[-1] in "ab":
                            tg_stem = tg_l[:-1]
                            if f'gene_name "{tg_stem}"' in p[8].lower() or f'"{tg_stem}"' in p[8].lower():
                                med_genes_dict[tg_stem] = (p[0], int(p[3]), int(p[4]))
        if med_genes_dict:
            primary_tg = TARGET_GENES[0].lower()
            primary_stem = primary_tg[:-1] if (len(primary_tg) > 3 and primary_tg[-1] in "ab") else primary_tg
            if primary_tg in med_genes_dict:
                m_chr = med_genes_dict[primary_tg][0]
            elif primary_stem in med_genes_dict:
                m_chr = med_genes_dict[primary_stem][0]
            else:
                m_chr = list(med_genes_dict.values())[0][0]
            chr_med_genes = {k: v for k, v in med_genes_dict.items() if v[0] == m_chr}
            m_st = max(0, min(v[1] for v in chr_med_genes.values()) - LOCUS_FLANK_BP)
            m_en = max(v[2] for v in chr_med_genes.values()) + LOCUS_FLANK_BP
            print(f"[+] Scanning orthologous Medaka locus: {m_chr}:{m_st:,}-{m_en:,}")
            # CRITICAL LEAKAGE FIX: Score Medaka reference locus strictly with held-out fold_0 model
            eval_med_model = grafted_medaka_models[0] if (isinstance(grafted_medaka_models, list) and len(grafted_medaka_models) > 0) else (grafted_medaka_model if grafted_medaka_model is not None else grafted_zf_model)
            print(f"[*] Scoring Medaka reference locus strictly with held-out fold_0 model (zero data leakage)...")
            m_positions, m_pred_abs, m_pred_rel = scan_locus_sliding_window(
                grafted_model=eval_med_model,
                genome_fa_path=MEDAKA_GENOME_FA,
                chrom=m_chr,
                start_bp=m_st,
                end_bp=m_en,
                window_bp=window_bp,
                stride_bp=stride_bp,
                batch_size=64,
                device=device
            )
            med_locus_info = (m_chr, m_st, m_en, m_positions, m_pred_abs)
    except Exception as e:
        print(f"[-] Notice scanning Medaka orthologous locus: {e}")

# Synchronized Cross-Species Visualization Panel
panel_count = 3
if zf_locus_info is not None:
    panel_count += 1
if med_locus_info is not None:
    panel_count += 1
fig, axes = plt.subplots(panel_count, 1, figsize=(14, 3 * panel_count), sharex=False)

# Panel 1: Target Species Pre-MZT Accessibility
axes[0].plot(positions / 1e6, target_pred_abs, color="#d7191c", lw=1.8, label=f"{TARGET_SPECIES_NAME} Pre-MZT Accessibility (y_abs)")
axes[0].plot(positions / 1e6, target_pred_late, color="#fd8d3c", lw=1.2, linestyle="--", alpha=0.8, label=f"{TARGET_SPECIES_NAME} Late Accessibility (pred_late)")
axes[0].axhline(enhancer_threshold, color="black", linestyle="--", label="95th %ile Threshold")
axes[0].set_ylabel("Predicted y_abs", fontsize=10)
axes[0].set_title(f"Target Species Locus ({TARGET_SPECIES_NAME}: {locus_chr}:{locus_start:,}-{locus_end:,})", fontweight="bold")
axes[0].legend(loc="upper right", fontsize=9)

# Panel 2: Target Species Timing Ratio
axes[1].plot(positions / 1e6, target_pred_rel, color="#2b83ba", lw=1.8, label="Timing Ratio (y_rel)")
axes[1].axhline(0, color="gray", linestyle=":")
axes[1].set_ylabel("Predicted y_rel", fontsize=10)
axes[1].legend(loc="upper right", fontsize=9)

# Panel 3: Target Gene Models
ax_genes = axes[2]
ax_genes.set_ylim(-0.5, 1.5)
for gname, chrom, st, en in tg_coords:
    ax_genes.plot([st / 1e6, en / 1e6], [0.5, 0.5], lw=6, color="#252525")
    ax_genes.annotate(gname, ((st + en)/2e6, 0.8), ha="center", fontsize=11, fontweight="bold")
ax_genes.set_yticks([])
ax_genes.set_ylabel("Gene Models", fontsize=10)
ax_genes.set_xlabel(f"Genomic Coordinate on Chr {locus_chr} (Mb)", fontsize=11)

# Cross-Species Orthologous Loci & Observed BigWig Overlay Panels (with chromosome end clamping)
curr_ax_idx = 3
if zf_locus_info is not None:
    z_chr, z_st, z_en, z_pos, z_abs = zf_locus_info
    ax_zf = axes[curr_ax_idx]
    curr_ax_idx += 1
    ax_zf.plot(z_pos / 1e6, z_abs, color="#756bb1", lw=1.8, label="Zebrafish Ortholog Predicted y_abs")
    zf_early_bw = Path(WORK_DIR) / "atac" / "Danio_rerio" / "Danio_rerio_early_cpm.bw"
    if zf_early_bw.exists():
        try:
            with pyBigWig.open(str(zf_early_bw)) as bw:
                chrom_lens = bw.chroms()
                z_max = chrom_lens.get(z_chr, z_en)
                z_en_clamped = min(z_en, z_max)
                z_st_clamped = min(z_st, max(0, z_en_clamped - 1))
                vals = np.nan_to_num(bw.values(z_chr, z_st_clamped, z_en_clamped))
            bw_x = np.linspace(z_st_clamped / 1e6, z_en_clamped / 1e6, len(vals))
            ax_zf.plot(bw_x, np.log2(vals + 1.0), color="#2ca25f", alpha=0.6, lw=1.0, label="Observed 256-cell ATAC CPM")
        except Exception as e:
            print(f"[-] Notice reading Zebrafish BigWig: {e}")
    ax_zf.set_ylabel("y_abs / ATAC", fontsize=10)
    ax_zf.set_xlabel(f"Zebrafish Chr {z_chr} (Mb)", fontsize=11)
    ax_zf.set_title(f"Orthologous Zebrafish Locus ({z_chr}:{z_st:,}-{z_en:,})", fontweight="bold")
    ax_zf.legend(loc="upper right", fontsize=9)

if med_locus_info is not None:
    m_chr, m_st, m_en, m_pos, m_abs = med_locus_info
    ax_med = axes[curr_ax_idx]
    curr_ax_idx += 1
    ax_med.plot(m_pos / 1e6, m_abs, color="#31a354", lw=1.8, label="Medaka Ortholog Predicted y_abs")
    med_early_bw = Path(WORK_DIR) / "atac" / "Oryzias_latipes" / "Oryzias_latipes_early_cpm.bw"
    if med_early_bw.exists():
        try:
            with pyBigWig.open(str(med_early_bw)) as bw:
                chrom_lens_m = bw.chroms()
                m_max = chrom_lens_m.get(m_chr, m_en)
                m_en_clamped = min(m_en, m_max)
                m_st_clamped = min(m_st, max(0, m_en_clamped - 1))
                vals_m = np.nan_to_num(bw.values(m_chr, m_st_clamped, m_en_clamped))
            bw_m_x = np.linspace(m_st_clamped / 1e6, m_en_clamped / 1e6, len(vals_m))
            ax_med.plot(bw_m_x, np.log2(vals_m + 1.0), color="#fd8d3c", alpha=0.6, lw=1.0, label="Observed st9 ATAC CPM")
        except Exception as e:
            print(f"[-] Notice reading Medaka BigWig: {e}")
    ax_med.set_ylabel("y_abs / ATAC", fontsize=10)
    ax_med.set_xlabel(f"Medaka Chr {m_chr} (Mb)", fontsize=11)
    ax_med.set_title(f"Orthologous Medaka Locus ({m_chr}:{m_st:,}-{m_en:,})", fontweight="bold")
    ax_med.legend(loc="upper right", fontsize=9)

plt.tight_layout()
fig3_path = res_dir / "fig3_locus_scan_profile.png"
plt.savefig(fig3_path, dpi=300)
plt.savefig(res_dir / "fig3_locus_scan_profile.svg")
plt.show()
print(f"[+] Figure 3 exported to {fig3_path}")


---
## Block 6: Base-Resolution Attribution, Teleost-Normalized Pre-MZT TF Filtering & Evolutionary Swap

We dissect the causal sequence grammar of top candidate enhancers driving early activation:
1. **Gradient Saliency & In Silico Mutagenesis (ISM):** Compute base-resolution contribution and sensitivity logos.
2. **Pre-MZT TF Matching via TOMTOM:** Query causal seqlets against both:
   - Filtered `premzt_tfs.meme` with teleost paralog normalization (`POU5F3` $\rightarrow$ `POU5F1`, `MXTX2` $\rightarrow$ `MIXL1`, `VOX` $\rightarrow$ `VENTX`).
   - Full JASPAR database for unbiased screening.
3. **Reciprocal In Silico Evolutionary Swap:** Exchange top causal motifs between target species and Zebrafish orthologs, evaluating $\Delta$ predicted pre-MZT accessibility.

In [ ]:
# Fresh-kernel resume guards
device = "cuda" if torch.cuda.is_available() else "cpu"
if "window_bp" not in locals():
    window_bp = 2114
if "df_enh" not in locals() or df_enh is None:
    enh_path = res_dir / f"{TARGET_LOCUS_NAME}_candidate_enhancers.tsv"
    if enh_path.exists():
        df_enh = pd.read_csv(enh_path, sep="\t")
        print(f"[+] Fresh kernel: loaded candidate enhancers from {enh_path}")
    else:
        raise FileNotFoundError(f"Candidate enhancers table missing: {enh_path}. Run Block 5 first.")
if "locus_chr" not in locals() or locus_chr is None:
    locus_chr = str(df_enh["chrom"].iloc[0])
if "df_counts" not in locals() or df_counts is None:
    gene_sets_tsv = res_dir / f"{TARGET_LOCUS_NAME}_gene_sets.tsv"
    if not gene_sets_tsv.exists():
        gene_sets_tsv = Path(WORK_DIR) / "rnaseq" / "target_species_gene_sets.tsv"
    if gene_sets_tsv.exists():
        df_counts = pd.read_csv(gene_sets_tsv, sep="\t")
if "filtered_meme_path" not in locals() or filtered_meme_path is None or not Path(filtered_meme_path).exists():
    cand_meme = Path(WORK_DIR) / "downloads" / "premzt_tfs.meme"
    if not cand_meme.exists():
        cand_meme = Path(WORK_DIR) / "downloads" / "JASPAR2024_CORE_vertebrates_non-redundant.meme"
    if not cand_meme.exists() and "JASPAR_MEME_PATH" in locals():
        cand_meme = Path(JASPAR_MEME_PATH)
    filtered_meme_path = cand_meme
if "grafted_zf_model" not in locals() or grafted_zf_model is None:
    from scripts.infer_and_attribute import load_grafted_model
    zf_ckpt = Path(WORK_DIR) / "models" / "chrombpnet" / "Danio_rerio" / "fold_0" / "grafted_chrombpnet.pt"
    if zf_ckpt.exists():
        grafted_zf_model = load_grafted_model(zf_ckpt, device=device)
if "grafted_medaka_model" not in locals():
    grafted_medaka_model = None
if grafted_medaka_model is None:
    from scripts.infer_and_attribute import load_grafted_model
    med_ckpt = Path(WORK_DIR) / "models" / "chrombpnet" / "Oryzias_latipes" / "fold_0" / "grafted_chrombpnet.pt"
    if med_ckpt.exists():
        grafted_medaka_model = load_grafted_model(med_ckpt, device=device)

# Base-Resolution Attribution, In Silico Mutagenesis (ISM) & Evolutionary Swap
from scripts.infer_and_attribute import (
    one_hot_encode_sequence,
    compute_input_gradient_attribution,
    compute_integrated_gradients,
    compute_in_silico_mutagenesis,
    compute_wt_projected_ism_matrix,
    seqlet_to_meme,
    run_tomtom,
    annotate_motif_hits,
    filter_jaspar_meme,
    perform_evolutionary_swap
)
import logomaker
from pyfaidx import Fasta
import matplotlib.pyplot as plt
import gzip

# 1. Select Top K Non-Redundant Candidate Enhancers (NMS peaks strictly filtered)
cands_pool = df_enh[df_enh.get("is_candidate", df_enh.get("is_enhancer", True))]
if cands_pool.empty:
    cands_pool = df_enh
top_k = min(5, len(cands_pool))
candidates_top = cands_pool.sort_values(by="pred_abs", ascending=False).head(top_k)
print(f"[+] Evaluating Top {top_k} Non-Redundant Candidate Enhancers via Full ISM + TOMTOM + Reciprocal Swap:")

# Extract Zebrafish Orthologous Locus Sequence Dynamically First
zf_genome = Fasta(ZEBRAFISH_GENOME_FA, as_raw=True)
zf_locus_seq = ""
zf_chr = None
try:
    with (gzip.open(ZEBRAFISH_GTF, "rt") if str(ZEBRAFISH_GTF).endswith(".gz") else open(ZEBRAFISH_GTF, "rt")) as f_gtf_zf:
        zf_t_coords = []
        for line in f_gtf_zf:
            if line.startswith("#"):
                continue
            p = line.strip().split("\t")
            if len(p) >= 9 and p[2] == "gene":
                for tg in TARGET_GENES:
                    tg_l = tg.lower()
                    if f'gene_name "{tg_l}"' in p[8].lower() or f'"{tg_l}"' in p[8].lower():
                        zf_t_coords.append((p[0], int(p[3]), int(p[4])))
    if zf_t_coords:
        zf_chr = zf_t_coords[0][0]
        zf_chr_coords = [c for c in zf_t_coords if c[0] == zf_chr]
        zf_min = max(0, min(c[1] for c in zf_chr_coords) - LOCUS_FLANK_BP)
        zf_max = max(c[2] for c in zf_chr_coords) + LOCUS_FLANK_BP
        zf_locus_seq = str(zf_genome[zf_chr][zf_min:zf_max]).upper()
        print(f"[+] Dynamically extracted Zebrafish orthologous locus: {zf_chr}:{zf_min:,}-{zf_max:,} ({len(zf_locus_seq):,} bp)")
except Exception as e:
    print(f"[-] Notice dynamically extracting Zebrafish locus: {e}")

# Dynamically extract Medaka orthologous locus sequence when available
medaka_locus_seq = None
if ENABLE_MEDAKA and MEDAKA_GENOME_FA and Path(MEDAKA_GENOME_FA).exists() and MEDAKA_GTF and Path(MEDAKA_GTF).exists():
    try:
        med_genome = Fasta(MEDAKA_GENOME_FA, as_raw=True)
        med_genes_dict = {}
        with (gzip.open(MEDAKA_GTF, "rt") if str(MEDAKA_GTF).endswith(".gz") else open(MEDAKA_GTF, "rt")) as f_gtf_med:
            for line in f_gtf_med:
                if line.startswith("#"):
                    continue
                p = line.strip().split("\t")
                if len(p) >= 9 and p[2] == "gene":
                    for tg in TARGET_GENES:
                        tg_l = tg.lower()
                        if f'gene_name "{tg_l}"' in p[8].lower() or f'"{tg_l}"' in p[8].lower():
                            med_genes_dict[tg_l] = (p[0], int(p[3]), int(p[4]))
                        elif len(tg_l) > 3 and tg_l[-1] in "ab":
                            tg_stem = tg_l[:-1]
                            if f'gene_name "{tg_stem}"' in p[8].lower() or f'"{tg_stem}"' in p[8].lower():
                                med_genes_dict[tg_stem] = (p[0], int(p[3]), int(p[4]))
        if med_genes_dict:
            primary_tg = TARGET_GENES[0].lower()
            primary_stem = primary_tg[:-1] if (len(primary_tg) > 3 and primary_tg[-1] in "ab") else primary_tg
            if primary_tg in med_genes_dict:
                med_chr = med_genes_dict[primary_tg][0]
            elif primary_stem in med_genes_dict:
                med_chr = med_genes_dict[primary_stem][0]
            else:
                med_chr = list(med_genes_dict.values())[0][0]
            chr_med_genes = {k: v for k, v in med_genes_dict.items() if v[0] == med_chr}
            med_min = max(0, min(v[1] for v in chr_med_genes.values()) - LOCUS_FLANK_BP)
            med_max = max(v[2] for v in chr_med_genes.values()) + LOCUS_FLANK_BP
            medaka_locus_seq = str(med_genome[med_chr][med_min:med_max]).upper()
            print(f"[+] Dynamically extracted Medaka orthologous locus: {med_chr}:{med_min:,}-{med_max:,} ({len(medaka_locus_seq):,} bp)")
    except Exception as e_med_loc:
        print(f"[-] Notice dynamically extracting Medaka locus: {e_med_loc}")

# Build maternal RNA-seq CPM dictionary from Block 1 df_counts
rnaseq_cpm_dict = df_counts.dropna(subset=["gene_name"]).groupby(df_counts["gene_name"].str.lower())["cpm_early"].max().to_dict()
target_genome = Fasta(TARGET_GENOME_FA, as_raw=True)

center_offset = window_bp // 2
search_flank = min(400, center_offset)
all_swaps = []
all_cands_logo = {}
all_cands_med_logo = {}

all_cands_ism = {}
for cand_idx, (_, cand_row) in enumerate(candidates_top.iterrows()):
    c_rank = cand_idx + 1
    c_pos = int(cand_row["center"])
    w_st = max(0, c_pos - window_bp // 2)
    w_en = w_st + window_bp
    c_seq = str(target_genome[locus_chr][w_st:w_en]).upper()
    if len(c_seq) < window_bp:
        c_seq += "N" * (window_bp - len(c_seq))
    c_onehot = one_hot_encode_sequence(c_seq)

    # Integrated Gradients attribution to find causal center
    c_attr = compute_integrated_gradients(grafted_zf_model, c_onehot, target="y_abs", steps=20, baseline="dinuc", device=device)
    c_scores = np.sum(np.abs(c_attr), axis=0)
    c_grad_peak = (center_offset - search_flank) + int(np.argmax(c_scores[center_offset - search_flank : center_offset + search_flank]))

    # Centered 60 bp In Silico Mutagenesis (ISM)
    c_ism_st = max(0, min(window_bp - 60, c_grad_peak - 30))
    c_ism_mat, c_wt_importance = compute_in_silico_mutagenesis(
        grafted_model=grafted_zf_model,
        wt_one_hot=c_onehot,
        region_start=c_ism_st,
        region_len=60,
        target="y_abs",
        batch_size=64,
        device=device
    )

    all_cands_ism[c_rank] = c_ism_mat
    # WT-projected logo matrix
    wt_sub_onehot = c_onehot[:, c_ism_st : c_ism_st + 60]
    c_wt_logo = compute_wt_projected_ism_matrix(c_ism_mat, wt_sub_onehot, c_wt_importance)
    c_df_logo = pd.DataFrame(c_wt_logo.T, columns=["A", "C", "G", "T"])
    all_cands_logo[c_rank] = c_df_logo

    # Extract core causal seqlet
    from scripts.infer_and_attribute import extract_causal_seqlet
    c_off, c_end, _ = extract_causal_seqlet(c_wt_importance, min_len=8, max_len=18)
    c_motif_len = c_end - c_off
    c_causal_sub_ism = c_ism_mat[:, c_off:c_end]
    c_causal_seq = c_seq[c_ism_st + c_off : c_ism_st + c_end]
    c_motif_g_st = w_st + c_ism_st + c_off
    c_motif_g_en = c_motif_g_st + c_motif_len

    # TOMTOM motif matching against pre-MZT TFs
    c_query_meme = res_dir / f"candidate_{c_rank}_causal_motif.meme"
    seqlet_to_meme(ppm_or_pwm=c_causal_sub_ism, motif_id=f"CAUSAL_PREMZT_{c_rank}", out_meme_path=c_query_meme)
    c_df_tomtom = run_tomtom(query_meme_path=c_query_meme, target_meme_path=filtered_meme_path, thresh=0.5)
    c_df_tomtom = annotate_motif_hits(
        c_df_tomtom,
        tf_list_path=PREMZT_TF_LIST_PATH,
        rnaseq_cpm_dict=rnaseq_cpm_dict,
        target_meme_path=filtered_meme_path
    )
    best_tf = c_df_tomtom.iloc[0].get("TF_name", c_df_tomtom.iloc[0]["Target_ID"]) if not c_df_tomtom.empty else "None"
    best_qval = float(c_df_tomtom.iloc[0]["q-value"]) if not c_df_tomtom.empty else 1.0
    best_cpm = float(c_df_tomtom.iloc[0].get("maternal_cpm", 0.0)) if not c_df_tomtom.empty else 0.0

    # Evolutionary alignment & reciprocal swap
    if zf_locus_seq:
        c_swap = perform_evolutionary_swap(
            model=grafted_zf_model,
            target_species_seq=c_seq,
            ortholog_species_seq=zf_locus_seq,
            motif_start=c_ism_st + c_off,
            motif_len=c_motif_len,
            device=device
        )
    else:
        c_swap = {
            "status": "skipped_no_ortholog",
            "classification": "skipped_no_ortholog",
            "is_lineage_gain": False,
            "target_ablation_delta": 0.0,
            "target_wt": 0.0,
            "target_ablation_score": 0.0,
            "target_swap_score": None,
            "target_swap_delta": None,
            "ortholog_aligned_motif": None,
            "aligned_target_motif_window": None,
            "aligned_match_bar": None,
            "aligned_ortholog_motif_window": None,
            "target_flank": "",
            "ortholog_flank": ""
        }

    c_record = {
        "enhancer_rank": c_rank,
        "chrom": cand_row["chrom"],
        "start": int(cand_row["start"]),
        "end": int(cand_row["end"]),
        "center": c_pos,
        "pred_abs": float(cand_row["pred_abs"]),
        "pred_rel": float(cand_row["pred_rel"]),
        "motif_genomic_start": int(c_motif_g_st),
        "motif_genomic_end": int(c_motif_g_en),
        "causal_seq": str(c_causal_seq),
        "motif_len": int(c_motif_len),
        "ism_region_start": int(c_ism_st),
        "best_offset": int(c_off),
        "best_tf_hit": str(best_tf),
        "best_tf_qval": round(best_qval, 4),
        "best_tf_maternal_cpm": round(best_cpm, 2),
        **c_swap
    }
    all_swaps.append(c_record)
    print(f"    Rank {c_rank}: center={c_pos} | motif='{c_causal_seq}' (TF: {best_tf}, maternal CPM={best_cpm:.1f}) | class={c_swap.get('classification')} | swap_Δ={c_swap.get('target_swap_delta')}")

df_all_swaps = pd.DataFrame(all_swaps)

# Select the top-ranked Astatotilapia-gained causal motif
# Prioritize is_lineage_gain == True (status == "aligned_motif_turnover" with target_swap_delta < -0.05, OR status == "insertion_in_target_novel_element" with target_ablation_delta < -0.05)
mask_lineage = df_all_swaps["is_lineage_gain"].fillna(False).astype(bool) if "is_lineage_gain" in df_all_swaps.columns else pd.Series(False, index=df_all_swaps.index)
mask_turnover = ((df_all_swaps["status"] == "aligned_motif_turnover") & (df_all_swaps["target_swap_delta"] < -0.05)) if ("status" in df_all_swaps.columns and "target_swap_delta" in df_all_swaps.columns) else pd.Series(False, index=df_all_swaps.index)
mask_novel = ((df_all_swaps["status"] == "insertion_in_target_novel_element") & (df_all_swaps["target_ablation_delta"] < -0.05)) if ("status" in df_all_swaps.columns and "target_ablation_delta" in df_all_swaps.columns) else pd.Series(False, index=df_all_swaps.index)
gain_pool = df_all_swaps[mask_lineage | mask_turnover | mask_novel]
if not gain_pool.empty:
    selected_row = gain_pool.sort_values(by="target_ablation_delta", ascending=True).iloc[0]
else:
    selected_row = df_all_swaps.sort_values(by="target_ablation_delta", ascending=True).iloc[0]

best_rank = int(selected_row["enhancer_rank"])
df_all_swaps["is_selected"] = (df_all_swaps["enhancer_rank"] == best_rank)

swap_summary_path = res_dir / "evolutionary_swap_summary.tsv"
df_all_swaps.to_csv(swap_summary_path, sep="	", index=False)
print(f"[+] Saved comprehensive multi-candidate evolutionary swap summary to {swap_summary_path}")

swap_results = selected_row.to_dict()
best_pos = int(selected_row["center"])
sel_w_st = max(0, best_pos - window_bp // 2)
sel_w_en = sel_w_st + window_bp
sel_seq = str(target_genome[locus_chr][sel_w_st:sel_w_en]).upper()
if len(sel_seq) < window_bp:
    sel_seq += "N" * (window_bp - len(sel_seq))
sel_onehot = one_hot_encode_sequence(sel_seq)
motif_genomic_start = int(selected_row["motif_genomic_start"])
motif_genomic_end = int(selected_row["motif_genomic_end"])
causal_seq = str(selected_row["causal_seq"])
motif_len = int(selected_row["motif_len"])
ism_region_start = int(selected_row["ism_region_start"])
best_offset = int(selected_row["best_offset"])
df_logo = all_cands_logo[best_rank]

# Run TOMTOM against both filtered pre-MZT TFs and full JASPAR database for selected top enhancer
top_query_meme = res_dir / f"candidate_{best_rank}_causal_motif.meme"
if not top_query_meme.exists():
    c_causal_sub_ism = all_cands_ism[best_rank][:, best_offset : best_offset + motif_len] if "all_cands_ism" in locals() and best_rank in all_cands_ism else one_hot_encode_sequence(causal_seq)
    seqlet_to_meme(ppm_or_pwm=c_causal_sub_ism, motif_id=f"CAUSAL_PREMZT_{best_rank}", out_meme_path=top_query_meme)

# 1. TOMTOM against pre-MZT TFs
df_tomtom_premzt = run_tomtom(query_meme_path=top_query_meme, target_meme_path=filtered_meme_path, thresh=0.5)
df_tomtom_premzt = annotate_motif_hits(
    df_tomtom_premzt,
    tf_list_path=PREMZT_TF_LIST_PATH,
    rnaseq_cpm_dict=rnaseq_cpm_dict,
    target_meme_path=filtered_meme_path
)
tomtom_premzt_tsv = res_dir / "tomtom_premzt_tfs.tsv"
df_tomtom_premzt.to_csv(tomtom_premzt_tsv, sep="	", index=False)

# 2. TOMTOM against full JASPAR database
full_jaspar = Path(JASPAR_MEME_PATH) if ("JASPAR_MEME_PATH" in locals() and JASPAR_MEME_PATH and Path(JASPAR_MEME_PATH).exists()) else (Path(WORK_DIR) / "downloads" / "JASPAR2024_CORE_vertebrates_non-redundant.meme")
if full_jaspar.exists():
    df_tomtom_jaspar = run_tomtom(query_meme_path=top_query_meme, target_meme_path=full_jaspar, thresh=0.5)
    df_tomtom_jaspar = annotate_motif_hits(
        df_tomtom_jaspar,
        tf_list_path=PREMZT_TF_LIST_PATH,
        rnaseq_cpm_dict=rnaseq_cpm_dict,
        target_meme_path=full_jaspar
    )
else:
    df_tomtom_jaspar = df_tomtom_premzt.copy()
tomtom_jaspar_tsv = res_dir / "tomtom_jaspar.tsv"
df_tomtom_jaspar.to_csv(tomtom_jaspar_tsv, sep="	", index=False)

print("[*] Top TOMTOM Matches for Selected Enhancer (Pre-MZT TF Filtered):")
if not df_tomtom_premzt.empty:
    cols_p = [c for c in ["TF_name", "Target_ID", "p-value", "q-value", "maternal_cpm"] if c in df_tomtom_premzt.columns]
    print(df_tomtom_premzt[cols_p].head(5).to_string(index=False))

print("[*] Top TOMTOM Matches for Selected Enhancer (Full JASPAR Vertebrates):")
if not df_tomtom_jaspar.empty:
    cols_j = [c for c in ["TF_name", "Target_ID", "p-value", "q-value", "maternal_cpm"] if c in df_tomtom_jaspar.columns]
    print(df_tomtom_jaspar[cols_j].head(5).to_string(index=False))

# Compute timing-specific (y_rel) ISM scores for the selected candidate to compare timing vs openness
df_logo_rel = None
try:
    c_ism_mat_rel, c_wt_imp_rel = compute_in_silico_mutagenesis(
        grafted_model=grafted_zf_model,
        wt_one_hot=sel_onehot,
        region_start=ism_region_start,
        region_len=60,
        target="y_rel",
        batch_size=64,
        device=device
    )
    wt_sub_rel = sel_onehot[:, ism_region_start : ism_region_start + 60]
    c_wt_logo_rel = compute_wt_projected_ism_matrix(c_ism_mat_rel, wt_sub_rel, c_wt_imp_rel)
    df_logo_rel = pd.DataFrame(c_wt_logo_rel.T, columns=["A", "C", "G", "T"])
    rel_logo_path = res_dir / f"candidate_{best_rank}_y_rel_wt_logo.tsv"
    df_logo_rel.to_csv(rel_logo_path, sep="\t", index=False)
    print(f"[+] Computed timing-specific (y_rel) WT-projected ISM matrix and saved to {rel_logo_path}.")

    # Log Pearson and Spearman correlation between y_abs and y_rel WT importance
    from scipy.stats import pearsonr, spearmanr
    c_wt_imp_abs = df_logo.sum(axis=1).values
    pr_corr, pr_pval = pearsonr(c_wt_imp_abs, c_wt_imp_rel)
    sp_corr, sp_pval = spearmanr(c_wt_imp_abs, c_wt_imp_rel)
    print(f"[*] WT Importance Correlation (y_abs vs y_rel): Pearson r = {pr_corr:.3f} (p={pr_pval:.2e}), Spearman r = {sp_corr:.3f} (p={sp_pval:.2e})")
except Exception as e_rel_ism:
    print(f"[-] Notice computing y_rel ISM matrix: {e_rel_ism}")

# Perform evolutionary swap against Medaka when available
medaka_swap_results = None
if ENABLE_MEDAKA and medaka_locus_seq and grafted_medaka_model is not None:
    try:
        medaka_swap_results = perform_evolutionary_swap(
            model=grafted_medaka_model,
            target_species_seq=sel_seq,
            ortholog_species_seq=medaka_locus_seq,
            motif_start=ism_region_start + best_offset,
            motif_len=motif_len,
            device=device
        )
        print(f"[+] Medaka Evolutionary Swap: classification='{medaka_swap_results.get('classification')}', swap_Δ={medaka_swap_results.get('target_swap_delta')}")
        pd.DataFrame([medaka_swap_results]).to_csv(res_dir / "medaka_evolutionary_swap.tsv", sep="\t", index=False)
    except Exception as e_m_sw:
        print(f"[-] Notice running Medaka evolutionary swap: {e_m_sw}")

# Compute paired Medaka logo for the selected element when available
df_logo_med = None
if grafted_medaka_model is not None:
    try:
        ism_mat_med, wt_imp_med = compute_in_silico_mutagenesis(
            grafted_model=grafted_medaka_model,
            wt_one_hot=sel_onehot,
            region_start=ism_region_start,
            region_len=60,
            target="y_abs",
            batch_size=64,
            device=device
        )
        wt_sub_med = sel_onehot[:, ism_region_start : ism_region_start + 60]
        wt_logo_med = compute_wt_projected_ism_matrix(ism_mat_med, wt_sub_med, wt_imp_med)
        df_logo_med = pd.DataFrame(wt_logo_med.T, columns=["A", "C", "G", "T"])
        print("[+] Computed paired Medaka model WT-projected ISM matrix.")
    except Exception as e:
        print(f"[-] Notice computing paired Medaka attribution: {e}")

print("=" * 80)
print(f"[*] Selected Top Causal Motif (Enhancer Rank {best_rank}):")
print(f"    Coordinates:               {locus_chr}:{motif_genomic_start:,}-{motif_genomic_end:,} ({motif_len} bp)")
print(f"    Target Motif:              '{causal_seq}' (TF: {selected_row['best_tf_hit']}, maternal CPM: {selected_row['best_tf_maternal_cpm']})")
print(f"    Evolutionary Class:        {selected_row['classification']}")
if pd.notna(selected_row.get('target_ablation_delta')):
    print(f"    In Silico Ablation Δ:      Δ_abs = {float(selected_row['target_ablation_delta']):+.3f}")
if pd.notna(selected_row.get('target_swap_delta')):
    print(f"    Target -> ZF Swap Δ:       Δ_abs = {float(selected_row['target_swap_delta']):+.3f}")
if pd.notna(selected_row.get('ortholog_swap_delta')):
    print(f"    ZF -> Target Swap Δ:       Δ_abs = {float(selected_row['ortholog_swap_delta']):+.3f}")
print("--- Side-by-Side Nucleotide Alignment around Causal Motif (±25 bp) ---")
if pd.notna(selected_row.get('aligned_target_motif_window')):
    print(f"    Target:    {selected_row.get('aligned_target_motif_window')}")
    print(f"               {selected_row.get('aligned_match_bar', '')}")
    print(f"    Ortholog:  {selected_row.get('aligned_ortholog_motif_window')}")
else:
    print(f"    Astatotilapia: {selected_row.get('target_flank', 'N/A')}")
    print(f"    Zebrafish:     {selected_row.get('ortholog_flank', 'N/A')}")
print("=" * 80)

# Plot Figure 4 (with paired Medaka model logo comparison when available)
if df_logo_med is not None:
    fig = plt.figure(figsize=(16, 8))
    gs = fig.add_gridspec(3, 2, width_ratios=[2.5, 1])

    ax_logo1 = fig.add_subplot(gs[0, 0])
    sub_logo = df_logo.iloc[max(0, best_offset - 4) : min(len(df_logo), best_offset + motif_len + 4)].reset_index(drop=True)
    logomaker.Logo(sub_logo, ax=ax_logo1)
    ax_logo1.set_title(f"Zebrafish-Model Causal Motif: '{causal_seq}' (WT-Projected ISM)", fontweight="bold")
    ax_logo1.set_ylabel("WT Importance")

    ax_logo_med = fig.add_subplot(gs[1, 0])
    sub_logo_med = df_logo_med.iloc[max(0, best_offset - 4) : min(len(df_logo_med), best_offset + motif_len + 4)].reset_index(drop=True)
    logomaker.Logo(sub_logo_med, ax=ax_logo_med)
    ax_logo_med.set_title(f"Medaka-Model Paired Causal Motif (WT-Projected ISM)", fontweight="bold")
    ax_logo_med.set_ylabel("WT Importance")

    ax_logo2 = fig.add_subplot(gs[2, 0])
    logomaker.Logo(df_logo, ax=ax_logo2)
    ax_logo2.set_title("Central 60 bp Enhancer Grammar (WT-Projected Activating Profile)", fontweight="bold")
    ax_logo2.set_ylabel("WT Importance")
    ax_logo2.set_xlabel("Base Position (bp)")

    ax_swap = fig.add_subplot(gs[:, 1])
else:
    fig = plt.figure(figsize=(15, 6))
    gs = fig.add_gridspec(2, 2, width_ratios=[2.5, 1])

    ax_logo1 = fig.add_subplot(gs[0, 0])
    sub_logo = df_logo.iloc[max(0, best_offset - 4) : min(len(df_logo), best_offset + motif_len + 4)].reset_index(drop=True)
    logomaker.Logo(sub_logo, ax=ax_logo1)
    ax_logo1.set_title(f"Causal Motif: '{causal_seq}' (WT-Projected ISM - Activating Bases Positive)", fontweight="bold")
    ax_logo1.set_ylabel("WT Importance")

    ax_logo2 = fig.add_subplot(gs[1, 0])
    logomaker.Logo(df_logo, ax=ax_logo2)
    ax_logo2.set_title("Central 60 bp Enhancer Grammar (WT-Projected Activating Profile)", fontweight="bold")
    ax_logo2.set_ylabel("WT Importance")
    ax_logo2.set_xlabel("Base Position (bp)")

    ax_swap = fig.add_subplot(gs[:, 1])

cats = ["In Silico Ablation"]
deltas = [float(selected_row["target_ablation_delta"]) if pd.notna(selected_row.get("target_ablation_delta")) else 0.0]
if pd.notna(selected_row.get('target_swap_delta')):
    cats.append("Ortholog Swap")
    deltas.append(float(selected_row["target_swap_delta"]))
colors = ["#d7191c" if d < 0 else "#2b83ba" for d in deltas]
ax_swap.bar(cats, deltas, color=colors, width=0.4, edgecolor='black')
ax_swap.axhline(0, color="gray", linestyle="--")
ax_swap.set_ylabel("Predicted Δ Pre-MZT Accessibility (y_abs)")
ax_swap.set_title("Causal Element Perturbation Impact", fontweight="bold")
ax_swap.grid(True, linestyle=":", alpha=0.5)
plt.tight_layout()
fig4_path = res_dir / "fig4_motif_attribution_logo.png"
plt.savefig(fig4_path, dpi=300)
plt.show()
print(f"[+] Figure 4 exported to {fig4_path}")


---
## Block 7: Multi-Task `NTv3` Fine-Tuning & Publication Baselines Benchmark

We integrate the multi-task genomic foundation model (**Nucleotide Transformer v3**):
1. **Reuse Block 1's Processed SLAM-seq BigWigs:** Directly connects to Zebrafish nascent RNA SLAM-seq BigWigs ($0.75\text{ hpf}$, $2.5\text{ hpf}$, $3.0–3.5\text{ hpf}$, $4.5\text{ hpf}$) alongside ATAC BigWigs ($256\text{-cell}$, $\text{dome}$).
2. **Publication Baselines & Frozen-Head Benchmark (`NTV3_RUN_BASELINES = True`):**
   - Evaluates (1) Raw pretrained `NTv3_tracks`, (2) Frozen trunk + linear head (`frozen_head`), and (3) LoRA / full fine-tuning.
   - Reports held-out Zebrafish ATAC-seq Spearman $r$, early-vs-late timing AUROC (excluding `miR-430`), zero-shot Medaka `st9` ATAC AUROC, and zero-shot target-species `strictly_zygotic_minor_wave` & `slam_confirmed_minor_wave` AUROC.
3. **16 GB VRAM Activation Checkpointing:** Manual non-reentrant activation checkpointing wraps heavy Transformer blocks, maintaining peak VRAM under 7.5 GB during 65,536 bp LoRA fine-tuning on RTX 3080 Laptop GPUs.


In [ ]:
# Launch Multi-Task NTv3 Epigenomics/SLAM-seq Fine-Tuning Task
zf_atac_dir = Path(WORK_DIR) / "atac" / "Danio_rerio"
chrom_sizes_zf = zf_atac_dir / "chrom.sizes"
chr_fold_json_zf = (zf_atac_dir / "chr_folds.json") if len(FOLDS_TO_TRAIN) > 1 else (zf_atac_dir / "chr_fold.json")

for f_idx in FOLDS_TO_TRAIN:
    fold_arg = ["--fold", str(f_idx)] if len(FOLDS_TO_TRAIN) > 1 else []
    task_name = f"train_ntv3_zf_fold_{f_idx}" if len(FOLDS_TO_TRAIN) > 1 else "train_ntv3_zf"

    cmd_base = [
        "torchrun",
        "--nproc_per_node", str(NUM_GPUS)
    ] if (COMPUTE_PROFILE == "hpc_1mb" or NUM_GPUS > 1) else ["python", "-u"]

    ntv3_cmd = cmd_base + [
        "scripts/train_ntv3.py",
        "--work-dir", WORK_DIR,
        "--genome-fa", str(ZEBRAFISH_GENOME_FA),
        "--chrom-sizes", str(chrom_sizes_zf),
        "--chr-fold-json", str(chr_fold_json_zf),
        "--model-id", NTV3_MODEL_ID,
        "--compute-profile", COMPUTE_PROFILE,
        "--context-len", str(NTV3_CONTEXT_LEN),
        "--train-mode", NTV3_TRAIN_MODE,
        "--batch-size", str(NTV3_BATCH_SIZE),
        "--grad-accum", str(NTV3_GRAD_ACCUM),
        "--task-name", task_name,
        "--epochs", "3"
    ]
    if NTV3_GRAD_CHECKPOINTING:
        ntv3_cmd.append("--grad-checkpointing")
    else:
        ntv3_cmd.append("--no-grad-checkpointing")

    if NTV3_RUN_BASELINES:
        ntv3_cmd.append("--run-baselines")
    if ENABLE_OPTION2_TARGET_RNA_ABLATION:
        ntv3_cmd.append("--enable-option2-target-rna")
    ntv3_cmd.extend(fold_arg)

    env_vars = {}
    if HF_TOKEN:
        env_vars["HF_TOKEN"] = HF_TOKEN

    tm.submit(task_name, ntv3_cmd, env_vars=env_vars, depends_on=["prep_atac_danio_rerio", "prep_slamseq_zf"], force_rerun=FORCE_RERUN, requires_gpu=True)

display(tm.status())


In [ ]:
# ==============================================================================
# Monitor Multi-Task NTv3 Training & Inspect Publication Baselines Benchmark
# ==============================================================================
display(tm.status())

bench_tsv = Path(WORK_DIR) / "models" / "ntv3" / "ntv3_baselines_benchmark.tsv"
if not bench_tsv.exists():
    bench_tsv_f0 = Path(WORK_DIR) / "models" / "ntv3" / "ntv3_baselines_benchmark_fold_0.tsv"
    if bench_tsv_f0.exists():
        bench_tsv = bench_tsv_f0

if bench_tsv.exists():
    df_baselines = pd.read_csv(bench_tsv, sep="\t")
    print("=" * 80)
    print("[*] Publication Cross-Architecture & Baselines Benchmark:")
    print(df_baselines.to_string())
    print("=" * 80)
    display(df_baselines)
else:
    print("[*] NTv3 benchmark table not yet generated (requires completed model training).")
    print("    Task 'train_ntv3_zf' is currently queued or in progress.")


---
## Block 8: NTv3 Inference & Head-to-Head Comparison with ChromBPNet

We execute downstream inference with fine-tuned `NTv3` and perform head-to-head cross-architecture comparisons with `ChromBPNet`:
1. **Matched Zero-Shot Gene Benchmark:** Evaluates `NTv3` (max `atac_256cell` and strand-matched `slam_2.5hpf` 3'-end signal) on the identical target-species gene set from Block 4.
2. **Locus-Wide Tiled NTv3 Scan:** Tiles `TARGET_GENES +/- LOCUS_FLANK_BP` with continuous `ntv3_atac_256cell`, `ntv3_atac_dome`, and `ntv3_slam_2.5hpf` tracks.
3. **Distal QuantSeq 3'-End Knockout, Zebrafish Motif Swap & Saliency:** Perturbs the top Block 6 causal motif in `NTv3` and computes input-embedding gradient saliency.
4. **Publication Visualizations (Figures 5, 6, 7):**
   - **Figure 5:** Synchronized cross-architecture locus overlay on `TARGET_LOCUS_NAME` (ChromBPNet vs NTv3 vs exon structures).
   - **Figure 6:** Per-gene distal QuantSeq 3'-end knockout / Zebrafish-swap bar plots & motif-level logo vs saliency.
   - **Figure 7:** Unified cross-architecture benchmark comparing ChromBPNet, NTv3 Raw, Frozen Head, and LoRA fine-tuning.


In [ ]:
# ==============================================================================
# Block 8a: Launch Standalone NTv3 Inference Worker via TaskManager
# ==============================================================================
block8_cmd = [
    "python", "-u", "scripts/ntv3_infer.py",
    "--work-dir", WORK_DIR,
    "--config", f"{WORK_DIR}/config.json",
    "--context-len", str(NTV3_CONTEXT_LEN),
    "--flank-bp", str(LOCUS_FLANK_BP)
]

tm.submit(
    "block8_ntv3_inference",
    block8_cmd,
    depends_on=["train_ntv3_zf"],
    force_rerun=FORCE_RERUN,
    requires_gpu=True
)

display(tm.status())


In [ ]:
# ==============================================================================
# Block 8b: Cross-Architecture Publication Comparisons (Figures 5, 6, 7 & Summary)
# ==============================================================================
from pathlib import Path
import json
import gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logomaker
from scipy import stats
from sklearn.metrics import roc_curve, precision_recall_curve
from scripts.infer_and_attribute import benchmark_transfer_metrics

# Load summary metrics and results
ntv3_scored_tsv = res_dir / "ntv3_target_genes_scored.tsv"
chrombpnet_scored_tsv = res_dir / "target_genes_scored_metrics.tsv"
locus_scan_tsv = res_dir / "ntv3_locus_scan.tsv"
ko_summary_tsv = res_dir / "ntv3_knockout_saliency_summary.tsv"
saliency_npz_path = res_dir / "ntv3_saliency_tracks.npz"
bench_tsv = Path(WORK_DIR) / "models" / "ntv3" / "ntv3_baselines_benchmark.tsv"
if not bench_tsv.exists():
    bench_tsv_f0 = Path(WORK_DIR) / "models" / "ntv3" / "ntv3_baselines_benchmark_fold_0.tsv"
    if bench_tsv_f0.exists():
        bench_tsv = bench_tsv_f0

# Parse gene exons for TARGET_GENES from TARGET_GTF
target_gene_exons = {}
target_gene_spans = {}
try:
    with (gzip.open(TARGET_GTF, "rt") if str(TARGET_GTF).endswith(".gz") else open(TARGET_GTF, "rt")) as f_gtf:
        for g_line in f_gtf:
            if g_line.startswith("#"):
                continue
            g_parts = g_line.strip().split("\t")
            if len(g_parts) >= 9:
                for tg in TARGET_GENES:
                    if f'"{tg.lower()}"' in g_parts[8].lower():
                        if g_parts[2] == "gene":
                            target_gene_spans[tg] = (g_parts[0], int(g_parts[3]), int(g_parts[4]), g_parts[6])
                        elif g_parts[2] == "exon":
                            if tg not in target_gene_exons:
                                target_gene_exons[tg] = []
                            target_gene_exons[tg].append((int(g_parts[3]) - 1, int(g_parts[4])))
except Exception as e_ex:
    print(f"[-] Notice parsing gene spans: {e_ex}")

# ------------------------------------------------------------------------------
# FIGURE 5: Synchronized Cross-Architecture Locus Overlay on TARGET_LOCUS_NAME
# ------------------------------------------------------------------------------
print("[*] Generating Figure 5: Cross-Architecture Locus Overlay...")
enh_tsv = res_dir / f"{TARGET_LOCUS_NAME}_candidate_enhancers.tsv"
df_enh = pd.read_csv(enh_tsv, sep="\t") if enh_tsv.exists() else pd.DataFrame()
df_scan = pd.read_csv(locus_scan_tsv, sep="\t") if locus_scan_tsv.exists() else pd.DataFrame()

fig5, axes5 = plt.subplots(4, 1, figsize=(14, 11), sharex=False, gridspec_kw={'height_ratios': [1.8, 1.8, 1.2, 0.8]})

# Panel 1: ChromBPNet y_abs and y_rel
ax_cbp = axes5[0]
if not df_enh.empty:
    x_mb = df_enh["center"].values / 1e6
    ax_cbp.plot(x_mb, df_enh["pred_abs"], color="#d7191c", lw=2.0, label="ChromBPNet Pre-MZT Accessibility (y_abs)")
    if "pred_rel" in df_enh.columns:
        ax_cbp.plot(x_mb, df_enh["pred_rel"], color="#2b83ba", lw=1.6, linestyle="--", label="Timing Specificity (y_rel)")
    cands_pts = df_enh[df_enh.get("is_candidate", df_enh.get("is_enhancer", False))]
    if not cands_pts.empty:
        ax_cbp.scatter(cands_pts["center"] / 1e6, cands_pts["pred_abs"], color="#d7191c", s=45, zorder=5, label="Candidate Enhancers (NMS)")
ax_cbp.set_ylabel("ChromBPNet Score", fontsize=10.5)
ax_cbp.set_title(f"A. Stage 2 Grafted ChromBPNet ({TARGET_LOCUS_NAME})", fontweight="bold", fontsize=12)
ax_cbp.legend(loc="upper right", frameon=True, fontsize=9)
ax_cbp.grid(True, linestyle=":", alpha=0.5)

# Panel 2: NTv3 Predicted Tracks (ATAC 256-cell, ATAC dome, SLAM 2.5 hpf)
ax_nt = axes5[1]
if not df_scan.empty:
    x_scan_mb = df_scan["center"].values / 1e6
    ax_nt.plot(x_scan_mb, df_scan["atac_256cell"], color="#e66101", lw=2.0, label="NTv3 ATAC (256-cell)")
    ax_nt.plot(x_scan_mb, df_scan["atac_dome"], color="#fdb863", lw=1.5, linestyle=":", label="NTv3 ATAC (dome)")
    slam_comb = np.maximum(df_scan["slam_2.5hpf_fwd"], df_scan["slam_2.5hpf_rev"])
    ax_nt.plot(x_scan_mb, slam_comb, color="#5e3c99", lw=1.8, label="NTv3 Nascent SLAM-seq (2.5 hpf)")
ax_nt.set_ylabel("NTv3 Signal (Log1p)", fontsize=10.5)
ax_nt.set_title("B. Multi-Task Nucleotide Transformer v3 (Tiled Scan)", fontweight="bold", fontsize=12)
ax_nt.legend(loc="upper right", frameon=True, fontsize=9)
ax_nt.grid(True, linestyle=":", alpha=0.5)

# Panel 3: NTv3 Saliency Profile around Causal Enhancer
ax_sal = axes5[2]
if saliency_npz_path.exists():
    sal_data = np.load(saliency_npz_path)
    tg_first = TARGET_GENES[0]
    if f"{tg_first}_sal_atac" in sal_data:
        sal_arr = sal_data[f"{tg_first}_sal_atac"]
        w_st_a = sal_data[f"{tg_first}_w_st_atac"][0]
        x_sal_mb = np.linspace(w_st_a, w_st_a + len(sal_arr), len(sal_arr)) / 1e6
        ax_sal.plot(x_sal_mb, sal_arr, color="#ca0020", lw=1.6, label="NTv3 Enhancer Saliency (|grad|)")
ax_sal.set_ylabel("Saliency", fontsize=10.5)
ax_sal.set_title("C. Base-Resolution Saliency Attribution (|grad|)", fontweight="bold", fontsize=12)
ax_sal.legend(loc="upper right", frameon=True, fontsize=9)
ax_sal.grid(True, linestyle=":", alpha=0.5)

# Panel 4: Exon and Gene Models
ax_tx = axes5[3]
ax_tx.set_yticks([])
locus_c = str(df_enh["chrom"].iloc[0]) if not df_enh.empty else "chrom"
for g_i, (tg, span) in enumerate(target_gene_spans.items()):
    c, s, e, strand = span
    y_g = 0.5 + (0.3 if g_i % 2 == 0 else -0.3)
    ax_tx.plot([s / 1e6, e / 1e6], [y_g, y_g], color="#333333", lw=2.5)
    arrow = ">" if strand == "+" else "<"
    ax_tx.text((s + e) / 2e6, y_g + 0.12, f"{tg} ({arrow})", ha="center", va="bottom", fontsize=10, fontweight="bold")
    if tg in target_gene_exons:
        for ex_s, ex_e in target_gene_exons[tg]:
            ax_tx.fill_between([ex_s / 1e6, ex_e / 1e6], y_g - 0.08, y_g + 0.08, color="#2c7bb6", alpha=0.85)

ax_tx.set_ylim(0, 1)
ax_tx.set_xlabel(f"Genomic Coordinate on {locus_c} (Mb)", fontsize=11)
ax_tx.set_title("D. Target Gene Annotation", fontweight="bold", fontsize=12)

plt.tight_layout()
fig5_path = res_dir / "fig5_cross_architecture_locus_overlay.png"
fig5_alt = res_dir / "fig5_cross_architecture_benchmark.png"
plt.savefig(fig5_path, dpi=300)
plt.savefig(fig5_alt, dpi=300)
plt.show()
print(f"[+] Figure 5 exported to {fig5_path}")

# ------------------------------------------------------------------------------
# FIGURE 6: Distal QuantSeq Knockout & Motif-Level Logo vs Saliency
# ------------------------------------------------------------------------------
print("[*] Generating Figure 6: Knockout and Motif Attribution...")
fig6, axes6 = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={'width_ratios': [1.2, 1.8]})

# Left: Bar chart of WT vs Ablated vs Swap nascent 3' expression
ax_bar = axes6[0]
if ko_summary_tsv.exists():
    df_ko = pd.read_csv(ko_summary_tsv, sep="\t")
    genes = df_ko["gene"].tolist()
    x_pos = np.arange(len(genes))
    w = 0.25
    ax_bar.bar(x_pos - w, df_ko["wt_nascent_3p_log1p"], width=w, label="Wild-Type", color="#2b83ba", edgecolor="black")
    ax_bar.bar(x_pos, df_ko["abl_nascent_3p_log1p"], width=w, label="Distal Causal Motif Ablated", color="#d7191c", edgecolor="black")
    if "swap_nascent_3p_log1p" in df_ko.columns and df_ko["swap_nascent_3p_log1p"].notna().any():
        ax_bar.bar(x_pos + w, df_ko["swap_nascent_3p_log1p"], width=w, label="Zebrafish Motif Swap", color="#fdae61", edgecolor="black")
    ax_bar.set_xticks(x_pos)
    ax_bar.set_xticklabels(genes, fontsize=10.5, fontweight="bold")
    ax_bar.set_ylabel("Predicted Nascent 3' UTR Signal (Log1p)", fontsize=10.5)
    ax_bar.set_title("A. Distal QuantSeq 3'-End Knockout Impact", fontweight="bold", fontsize=11.5)
    ax_bar.legend(loc="upper right", frameon=True, fontsize=9)
    ax_bar.grid(True, linestyle=":", alpha=0.5)

# Right: ChromBPNet ISM Logo vs NTv3 Base-Resolution Saliency
ax_logo = axes6[1]
try:
    rel_logo_tsv = list(res_dir.glob("candidate_*_y_rel_wt_logo.tsv"))
    if rel_logo_tsv:
        df_l = pd.read_csv(rel_logo_tsv[0], sep="\t")
        logomaker.Logo(df_l, ax=ax_logo)
        ax_logo.set_title("B. Causal Motif Base-Resolution Grammar (WT-Projected ISM)", fontweight="bold", fontsize=11.5)
        ax_logo.set_xlabel("Base Position relative to Summit (bp)", fontsize=10.5)
        ax_logo.set_ylabel("WT Projected Importance", fontsize=10.5)
    else:
        ax_logo.text(0.5, 0.5, "Causal Motif ISM Profile (Run Block 6)", ha="center", va="center", fontsize=11)
except Exception as e_lg:
    print(f"[-] Notice plotting motif logo: {e_lg}")

plt.tight_layout()
fig6_path = res_dir / "fig6_knockout_and_motif_saliency.png"
plt.savefig(fig6_path, dpi=300)
plt.show()
print(f"[+] Figure 6 exported to {fig6_path}")

# ------------------------------------------------------------------------------
# FIGURE 7: Unified Cross-Architecture Benchmark (Overlaid ROC/PR & Bar Comparison)
# ------------------------------------------------------------------------------
print("[*] Generating Figure 7: Unified Cross-Architecture Benchmark...")
fig7, axes7 = plt.subplots(1, 3, figsize=(16, 5))

# Load matched gene tables for ChromBPNet and NTv3
df_cbp_sc = pd.read_csv(chrombpnet_scored_tsv, sep="\t") if chrombpnet_scored_tsv.exists() else pd.DataFrame()
df_nt_sc = pd.read_csv(ntv3_scored_tsv, sep="\t") if ntv3_scored_tsv.exists() else pd.DataFrame()

# Panel 1: ROC Curves
ax_roc = axes7[0]
curves_to_plot = []
if not df_cbp_sc.empty:
    m_cbp = benchmark_transfer_metrics(df_cbp_sc, score_col="pred_max_y_abs", pos_category="strictly_zygotic_minor_wave", ctrl_category="control_non_activated")
    curves_to_plot.append(("ChromBPNet (Grafted Stage 2)", df_cbp_sc["pred_max_y_abs"], "#d7191c", "-"))
if not df_nt_sc.empty and "ntv3_pred_atac_256cell_max" in df_nt_sc.columns:
    curves_to_plot.append(("NTv3 (ATAC 256-cell)", df_nt_sc["ntv3_pred_atac_256cell_max"], "#e66101", "-"))
if not df_nt_sc.empty and "ntv3_pred_slam_2.5hpf_3p" in df_nt_sc.columns:
    curves_to_plot.append(("NTv3 (SLAM 2.5 hpf 3'-End)", df_nt_sc["ntv3_pred_slam_2.5hpf_3p"], "#5e3c99", "-"))

for label_m, score_s, color_m, ls_m in curves_to_plot:
    df_eval = df_cbp_sc if len(score_s) == len(df_cbp_sc) else df_nt_sc
    pos_mask = (df_eval["category"] == "strictly_zygotic_minor_wave") | (df_eval.get("is_strictly_zygotic_minor_wave", False) == True)
    ctrl_mask = df_eval["category"] == "control_non_activated"
    p_vals = score_s[pos_mask].dropna().values
    c_vals = score_s[ctrl_mask].dropna().values
    if len(p_vals) > 0 and len(c_vals) > 0:
        y_true = np.concatenate([np.ones_like(p_vals), np.zeros_like(c_vals)])
        y_score = np.concatenate([p_vals, c_vals])
        fpr, tpr, _ = roc_curve(y_true, y_score)
        m_auc = benchmark_transfer_metrics(df_eval.assign(_s=score_s), score_col="_s", pos_category="strictly_zygotic_minor_wave", ctrl_category="control_non_activated").get("roc_auc", 0.0)
        ax_roc.plot(fpr, tpr, label=f"{label_m} (AUC={m_auc:.2f})", color=color_m, lw=2.0, linestyle=ls_m)

ax_roc.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax_roc.set_xlabel("False Positive Rate", fontsize=10.5)
ax_roc.set_ylabel("True Positive Rate", fontsize=10.5)
ax_roc.set_title("A. Zero-Shot ROC: Minor Wave", fontweight="bold", fontsize=11.5)
ax_roc.legend(loc="lower right", frameon=True, fontsize=8.5)
ax_roc.grid(True, linestyle=":", alpha=0.5)

# Panel 2: PR Curves
ax_pr = axes7[1]
for label_m, score_s, color_m, ls_m in curves_to_plot:
    df_eval = df_cbp_sc if len(score_s) == len(df_cbp_sc) else df_nt_sc
    pos_mask = (df_eval["category"] == "strictly_zygotic_minor_wave") | (df_eval.get("is_strictly_zygotic_minor_wave", False) == True)
    ctrl_mask = df_eval["category"] == "control_non_activated"
    p_vals = score_s[pos_mask].dropna().values
    c_vals = score_s[ctrl_mask].dropna().values
    if len(p_vals) > 0 and len(c_vals) > 0:
        y_true = np.concatenate([np.ones_like(p_vals), np.zeros_like(c_vals)])
        y_score = np.concatenate([p_vals, c_vals])
        prec, rec, _ = precision_recall_curve(y_true, y_score)
        m_pr = benchmark_transfer_metrics(df_eval.assign(_s=score_s), score_col="_s", pos_category="strictly_zygotic_minor_wave", ctrl_category="control_non_activated").get("pr_auc", 0.0)
        ax_pr.plot(rec, prec, label=f"{label_m} (PR={m_pr:.2f})", color=color_m, lw=2.0, linestyle=ls_m)

ax_pr.set_xlabel("Recall", fontsize=10.5)
ax_pr.set_ylabel("Precision", fontsize=10.5)
ax_pr.set_title("B. Precision-Recall: Minor Wave", fontweight="bold", fontsize=11.5)
ax_pr.legend(loc="lower left", frameon=True, fontsize=8.5)
ax_pr.grid(True, linestyle=":", alpha=0.5)

# Panel 3: Held-out Zebrafish Benchmark Comparison Bar Chart
ax_bench = axes7[2]
if bench_tsv.exists():
    try:
        df_b = pd.read_csv(bench_tsv, sep="\t")
        df_b_eval = df_b[df_b.get("status", "") == "evaluated"].copy()
        if not df_b_eval.empty:
            models = df_b_eval["model"].tolist()
            x = np.arange(len(models))
            vals_atac = pd.to_numeric(df_b_eval.get("atac_256cell_spearman_r", 0), errors="coerce").fillna(0).values
            vals_auc = pd.to_numeric(df_b_eval.get("early_vs_late_auroc", 0), errors="coerce").fillna(0.5).values
            w = 0.35
            ax_bench.bar(x - w/2, vals_atac, width=w, label="ATAC 256-cell r", color="#3182bd")
            ax_bench.bar(x + w/2, vals_auc, width=w, label="Early-vs-Late AUC", color="#de2d26")
            ax_bench.set_xticks(x)
            ax_bench.set_xticklabels(models, rotation=25, ha="right", fontsize=9.5)
            ax_bench.set_title("C. Held-Out Zebrafish Benchmarks", fontweight="bold", fontsize=11.5)
            ax_bench.legend(loc="upper left", frameon=True, fontsize=8.5)
            ax_bench.grid(True, linestyle=":", alpha=0.5)
    except Exception as e_b:
        print(f"[-] Notice plotting benchmark bar chart: {e_b}")

plt.tight_layout()
fig7_path = res_dir / "fig7_cross_architecture_benchmark.png"
plt.savefig(fig7_path, dpi=300)
plt.show()
print(f"[+] Figure 7 exported to {fig7_path}")

# ------------------------------------------------------------------------------
# Export Cross-Architecture Summary Table
# ------------------------------------------------------------------------------
summary_rows = []
if bench_tsv.exists():
    df_b = pd.read_csv(bench_tsv, sep="\t")
    for _, row in df_b.iterrows():
        summary_rows.append(row.to_dict())

# Add target-species zero-shot minor wave metrics
if not df_nt_sc.empty:
    m_nt_st = benchmark_transfer_metrics(df_nt_sc, score_col="ntv3_pred_atac_256cell_max", pos_category="strictly_zygotic_minor_wave", ctrl_category="control_non_activated")
    m_nt_sl = benchmark_transfer_metrics(df_nt_sc, score_col="ntv3_pred_slam_2.5hpf_3p", pos_category="strictly_zygotic_minor_wave", ctrl_category="control_non_activated")
    summary_rows.append({
        "model": "NTv3_target_minor_wave_eval",
        "architecture": "NTv3_tracks",
        "strictly_zygotic_auroc": m_nt_st.get("roc_auc"),
        "slam_confirmed_auroc": m_nt_sl.get("roc_auc"),
        "status": "evaluated"
    })

df_cross_summary = pd.DataFrame(summary_rows)
cross_summary_path = res_dir / f"{TARGET_LOCUS_NAME}_cross_architecture_summary.tsv"
df_cross_summary.to_csv(cross_summary_path, sep="\t", index=False)
print(f"[+] Saved cross-architecture summary table to {cross_summary_path}")
display(df_cross_summary)
